## 1. Initial Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
from qblox_instruments import Cluster, ClusterType
import contextlib
import pickle
import qutip
from scipy.linalg import sqrtm

from quantify_core.data.handling import (
    default_datadir,
    set_datadir,
    get_datadir,
    load_dataset,
    load_snapshot,
)
from quantify_core.data.experiment import (QuantifyExperiment)

from quantify_core.measurement import MeasurementControl
from quantify_scheduler.instrument_coordinator import InstrumentCoordinator
from quantify_scheduler.instrument_coordinator.components.qblox import ClusterComponent
from quantify_core.visualization.pyqt_plotmon import PlotMonitor_pyqt as PlotMonitor
from quantify_core.visualization.instrument_monitor import InstrumentMonitor

from quantify_scheduler.device_under_test.quantum_device import QuantumDevice
from quantify_scheduler.device_under_test.transmon_element import BasicTransmonElement

from quantify_scheduler import Schedule
from quantify_scheduler.gettables import ScheduleGettable
from quantify_scheduler.schedules import (
    heterodyne_spec_sched_nco, 
    two_tone_spec_sched_nco,
    two_tone_spec_sched,
    rabi_sched, 
    t1_sched,
    ramsey_sched,
    echo_sched,
    readout_calibration_sched,
    allxy_sched
)

from qcodes import ManualParameter
from qcodes.instrument import find_or_create_instrument

from quantify_scheduler.backends.qblox.operations.rf_switch_toggle import RFSwitchToggle

from quantify_core.analysis.cosine_analysis import CosineAnalysis
from quantify_core.analysis.single_qubit_timedomain import (
    RabiAnalysis, 
    RamseyAnalysis, 
    T1Analysis,
    EchoAnalysis,
    AllXYAnalysis,
)
from quantify_core.analysis.spectroscopy_analysis import (
    ResonatorSpectroscopyAnalysis,
    QubitSpectroscopyAnalysis,
)
from quantify_core.analysis.readout_calibration_analysis import (
    ReadoutCalibrationAnalysis,
)
from quantify_core.analysis.base_analysis import Basic2DAnalysis
from quantify_core.analysis.fitting_models import ResonatorModel, LorentzianModel, ExpDecayModel

resonator_model = ResonatorModel()
lorentzian_model = LorentzianModel()
exp_decay_model = ExpDecayModel()

from scipy.interpolate import CubicSpline, UnivariateSpline
import time
from tqdm.auto import tqdm
import sys

from snl_qblox.helpers import *
from snl_qblox.tomography_tools import *
from snl_qblox.HP83732B import *
from snl_qblox.TWPA_schedule import *
from snl_qblox.calibration_nodes import *

In [ ]:
import warnings

# warnings.filterwarnings(
#     "ignore",
#     category=FutureWarning,
#     module="quantify_scheduler.operations.acquisition_library"
# )

warnings.filterwarnings(
    "ignore",
    message=".*conflicting frequency definitions.*",
    category=RuntimeWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*zero setpoint range passed to pyqtgraph.*",
    category=UserWarning,
)

In [ ]:
from matplotlib import style
style.use('dark_background')
# style.use('default')

### 1.1 Cluster

In [ ]:
!qblox-pnp list

In [ ]:
cluster_ip = "172.16.0.3"
cluster_name = "cluster0"

In [ ]:
# Close the chosen QCodes instrument to prevent name clash
with contextlib.suppress(KeyError):
    Cluster.find_instrument(cluster_name).close()

cluster = Cluster(
    name=cluster_name,
    identifier=cluster_ip,
)

In [ ]:
for mod in cluster.modules:
    if mod.present() and mod.is_qcm_type and mod.is_rf_type:
        print('QCM-RF:')
        print(mod)
    if mod.present() and mod.is_qrm_type and mod.is_rf_type:
        print('QRM-RF:')
        print(mod)

In [ ]:
readout_module = cluster.modules[7-1]
control_module1 = cluster.modules[2-1]
control_module2 = cluster.modules[4-1]

In [ ]:
cluster.reset()
print(cluster.get_system_status())

In [ ]:
control_module1.out0_lo_en(True)
control_module1.out1_lo_en(False)
control_module2.out0_lo_en(True)
control_module2.out1_lo_en(False)

In [ ]:
for i in range(6):
    getattr(readout_module, f"sequencer{i}").nco_prop_delay_comp_en(True)

### 1.2 Quantify

In [ ]:
import os
DATA_DIR = os.environ.get("QBLOX_DATADIR", "CD1")  # quantify data dir; override with QBLOX_DATADIR
set_datadir(DATA_DIR)  # change me!
get_datadir() # confirm dir

In [ ]:
# plotmon.main_QtPlot # reactivate plot window
# plotmon.tuids_max_num(3)
# plotmon.tuids(['20260513-132738'])

In [ ]:
# insmon.create_widget()

In [ ]:
# cluster.reboot()

In [ ]:
# # Rebuild the cluster connection and the quantify stack (keeps quantum_device and its calibrations).
# # Use after a dropped connection (e.g. ConnectionAbortedError) or a cluster reboot.
# import contextlib, gc
# from qcodes.instrument.base import Instrument
# from quantify_scheduler.instrument_coordinator.components.base import InstrumentCoordinatorComponentBase


# def rebuild_cluster_stack(cluster_name="cluster0", cluster_ip="172.16.0.3", reset=None, bp0_max_before=31.55):
#     """reset=None: reset only if the cluster rebooted (its max-since-boot backplane temperature fell below
#     bp0_max_before - 0.3 C); True/False forces it. Returns the new objects; also binds them as notebook globals."""
#     g = globals()
#     # 1) close and unregister the cluster, its coordinator components (incl. per-module ones), ic and meas_ctrl
#     for name in [n for n in list(Instrument._all_instruments)
#                  if n in ("meas_ctrl", "ic", "ic_generic", cluster_name) or n.startswith(f"ic_{cluster_name}")]:
#         with contextlib.suppress(Exception):
#             Instrument._all_instruments[name].close()
#         Instrument._all_instruments.pop(name, None)
#     for key in [k for k in list(InstrumentCoordinatorComponentBase._no_gc_instances)
#                 if k in (cluster_name, "ic_generic") or k.startswith(f"ic_{cluster_name}")]:
#         InstrumentCoordinatorComponentBase._no_gc_instances.pop(key, None)
#     for name in ("cluster", "readout_module", "control_module1", "control_module2", "ic", "meas_ctrl", "nodes"):
#         g[name] = None
#     gc.collect()   # a leftover component collected later would close the NEW socket (ConnectionAbortedError 10053)

#     # 2) new connection; reset only after a reboot
#     cluster = Cluster(name=cluster_name, identifier=cluster_ip)
#     rebooted = cluster.get_maximum_bp_temperature_0() < bp0_max_before - 0.3
#     if reset or (reset is None and rebooted):
#         cluster.reset()
#     readout_module, control_module1, control_module2 = cluster.modules[7 - 1], cluster.modules[2 - 1], cluster.modules[4 - 1]
#     control_module1.out0_lo_en(True); control_module1.out1_lo_en(False)
#     control_module2.out0_lo_en(True); control_module2.out1_lo_en(False)
#     for i in range(6):
#         getattr(readout_module, f"sequencer{i}").nco_prop_delay_comp_en(True)

#     # 3) quantify stack (plot monitor reused if it exists)
#     meas_ctrl = MeasurementControl("meas_ctrl")
#     ic = InstrumentCoordinator("ic")
#     ic.timeout(120)
#     ic.add_component(ClusterComponent(cluster))
#     plotmon = g.get("plotmon") or PlotMonitor("plotmon")
#     meas_ctrl.instr_plotmon(plotmon.name)
#     quantum_device.instr_instrument_coordinator(ic.name)
#     nodes = CalibrationNodes(quantum_device, meas_ctrl, ACQ_DELAY, instrument_coordinator=ic)
#     gc.collect()

#     g.update(cluster=cluster, readout_module=readout_module, control_module1=control_module1,
#              control_module2=control_module2, meas_ctrl=meas_ctrl, ic=ic, plotmon=plotmon, nodes=nodes)
#     print("rebuilt: %s | %s | reset: %s | ic components %s | error queue %s" % (
#         cluster_name, cluster.get_system_status(), bool(reset or (reset is None and rebooted)),
#         list(ic.components()), cluster.get_system_error()))
#     print("note: re-null the iSWAP Stark phases before gate use (re-enabling the LOs shifts the difference phase)")
#     return cluster, ic, meas_ctrl, nodes


# cluster, ic, meas_ctrl, nodes = rebuild_cluster_stack()


In [ ]:
meas_ctrl = MeasurementControl("meas_ctrl")

ic = InstrumentCoordinator("ic")
ic.timeout(120)
ic.add_component(ClusterComponent(cluster))

# Create the live plotting instrument which handles the graphical interface
# Two windows will be created, the main will feature 1D plots and any 2D plots will go to the secondary
plotmon = PlotMonitor("plotmon")
# Connect the live plotting monitor to the measurement control
meas_ctrl.instr_plotmon(plotmon.name)
plotmon.tuids_max_num(3)

# The instrument monitor will give an overview of all parameters of all instruments
insmon = InstrumentMonitor("InstrumentMonitor")

In [ ]:
# We create an InstrumentCoordinator to control the cluster and add it to the
# QuantumDevice.
quantum_device = find_or_create_instrument(QuantumDevice, recreate=True, name="quantum_device")
quantum_device.instr_instrument_coordinator(ic.name)

qubit1 = find_or_create_instrument(BasicTransmonElement, recreate=True, name="qubit1")
quantum_device.add_element(qubit1)
qubit1.measure.acq_channel(0)

qubit2 = find_or_create_instrument(BasicTransmonElement, recreate=True, name="qubit2")
quantum_device.add_element(qubit2)
qubit2.measure.acq_channel(0)

snail = find_or_create_instrument(BasicTransmonElement, recreate=True, name="snail")
quantum_device.add_element(snail)
snail.measure.acq_channel(0)

In [ ]:
# To plot the pulse sequences
from quantify_scheduler.backends.graph_compilation import SerialCompiler
device_compiler = SerialCompiler("Device compiler", quantum_device)

In [ ]:
# Device parameters
QUBIT1_READOUT_FREQ = 7.145292e9
QUBIT2_READOUT_FREQ = 7.266316e9
SNAIL_READOUT_FREQ = 9.676e9
# time-of-flight calibrated, change me!
ACQ_DELAY = 216e-9 # 4ns time grid!

# LO_FREQ_READOUT1 = QUBIT1_READOUT_FREQ - 50e6
# LO_FREQ_READOUT2 = QUBIT2_READOUT_FREQ - 50e6
LO_FREQ_READOUT = 7.1e9

LO_FREQ_READOUTS = 9.62e9
LO_FREQ_QUBIT1 = 4.43e9
LO_FREQ_QUBIT2 = 4.69e9 #LO_FREQ_QUBIT1 + pump_freq
LO_FREQ_SNAIL = 7.4e9

In [ ]:
# "output_att": {
#             "qubit:mw-qubit.01": 0,
#         },

In [ ]:
hardware_cfg = {
    "config_type": "quantify_scheduler.backends.qblox_backend.QbloxHardwareCompilationConfig",
    "hardware_description": {
        f"{cluster.name}": {
            "instrument_type": "Cluster",
            "ref": "external",
            "modules": {
                # "2": {"instrument_type": "QCM_RF", "rf_output_on":False},
                "2": {"instrument_type": "QCM_RF"},
                "4": {"instrument_type": "QCM_RF"},
                "7": {"instrument_type": "QRM_RF"},
            },
        },
    },
    "hardware_options": {
        "modulation_frequencies": {
            "qubit1:res-qubit1.ro": {"lo_freq": LO_FREQ_READOUT},
            "qubit1:mw-qubit1.01": {"lo_freq": LO_FREQ_QUBIT1},
            "qubit1:fs-qubit1.12": {"lo_freq": LO_FREQ_QUBIT1},
            "qubit1:mw-qubit1.stark": {"lo_freq": LO_FREQ_QUBIT1},

            "qubit2:res-qubit2.ro": {"lo_freq": LO_FREQ_READOUT},
            "qubit2:mw-qubit2.01": {"lo_freq": LO_FREQ_QUBIT2},
            "qubit2:fs-qubit2.12": {"lo_freq": LO_FREQ_QUBIT2},
            "qubit2:mw-qubit2.stark": {"lo_freq": LO_FREQ_QUBIT2},
            
            "snail:res-snail.ro": {"lo_freq": LO_FREQ_READOUTS},
            "snail:mw-snail.01": {"lo_freq": LO_FREQ_SNAIL},
        },
        "mixer_corrections": {
            "qubit1:res-qubit1.ro": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "qubit1:mw-qubit1.01": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "qubit1:fs-qubit1.12": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "qubit1:mw-qubit1.stark": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "qubit2:res-qubit2.ro": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "qubit2:mw-qubit2.01": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "qubit2:fs-qubit2.12": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "qubit2:mw-qubit2.stark": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "snail:res-snail.ro": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
            "snail:mw-snail.01": {"auto_lo_cal": "on_lo_interm_freq_change", "auto_sideband_cal": "on_interm_freq_change"},
        }
        
        
    },
    "connectivity": {
        "graph": [
            ("cluster0.module2.complex_output_0", ["qubit1:mw"]),
            ("cluster0.module2.complex_output_0", ["qubit1:fs"]),
            ("cluster0.module7.complex_output_0", ["qubit1:res"]),
            ("cluster0.module7.digital_output_0", ["qubit1:switch"]),

            ("cluster0.module4.complex_output_0", ["qubit2:mw"]),
            ("cluster0.module4.complex_output_0", ["qubit2:fs"]),
            ("cluster0.module7.complex_output_0", ["qubit2:res"]),
            ("cluster0.module7.digital_output_0", ["qubit2:switch"]),

            ("cluster0.module4.complex_output_1", ["snail:mw"]),
            ("cluster0.module7.complex_output_0", ["snail:res"]),
            ("cluster0.module4.digital_output_1", ["snail:switch"]),
        ]
    },
}


In [ ]:
# This hardware configuration should also be added to the quantum device.
quantum_device.hardware_config(hardware_cfg)

In [ ]:
nodes = CalibrationNodes(quantum_device, meas_ctrl, ACQ_DELAY, instrument_coordinator=ic)

In [ ]:
# from quantify_scheduler.backends.types.common import Connectivity

# Connectivity.model_validate(hardware_cfg['connectivity']).draw()

In [ ]:
# change me if there is already a setup
# from quantify_core.utilities.experiment_helpers import load_settings_onto_instrument
# load_settings_onto_instrument(instrument=cluster, tuid="20250625-102836-807-a38d1b")

In [ ]:
def set_from_snapshot(tuid):
    snapshot = load_snapshot(tuid=tuid)
    # control_module1.out0_att(snapshot['instruments']['cluster0']['submodules']['module2']['parameters']['out0_att']['value'])
    # control_module2.out0_att(snapshot['instruments']['cluster0']['submodules']['module4']['parameters']['out0_att']['value'])
    # readout_module.out0_att(snapshot['instruments']['cluster0']['submodules']['module7']['parameters']['out0_att']['value'])

    control_module1.out0_att(20)
    control_module2.out0_att(20)
    readout_module.out0_att(40)

    qubit1_sub = snapshot['instruments']['qubit1']['submodules']
    qubit2_sub = snapshot['instruments']['qubit2']['submodules']

    for sub in qubit1_sub:
        for par in qubit1_sub[sub]['parameters']:
            val = qubit1_sub[sub]['parameters'][par]['value']
            if type(val) == float:
                getattr(getattr(qubit1, sub), par)(qubit1_sub[sub]['parameters'][par]['value'])

    for sub in qubit2_sub:
        for par in qubit2_sub[sub]['parameters']:
            val = qubit2_sub[sub]['parameters'][par]['value']
            if type(val) == float:
                getattr(getattr(qubit2, sub), par)(qubit2_sub[sub]['parameters'][par]['value']) 

In [ ]:
# '20260922-222430' 0 mA
# '20261001-160433' 0.715 mA
set_from_snapshot('20261001-160433')

### 1.3 Add Instrument

#### 1.3.1 HP83732B

In [ ]:
hp_ssg = find_or_create_instrument(HP83732B, recreate=True, name='HP83732B_1', address='TCPIP0::172.16.0.12::1234::SOCKET')
hp_ssg.timeout(1)

In [ ]:
hp_ssg.frequency(8.407e9)
hp_ssg.power(14.2)
hp_ssg.status('ON')
hp_ssg.pulsemod_state('ON')
hp_ssg.pulsemod_source('EXT')

In [ ]:
# hp_ssg.pulsemod_state('OFF')

#### 1.3.2 SGS100A

In [ ]:
from qcodes.instrument_drivers.rohde_schwarz import RohdeSchwarzSGS100A

SGS100A_1 = find_or_create_instrument(RohdeSchwarzSGS100A, recreate=True, name='SGS100A_1', address='TCPIP::172.16.0.11::INSTR')
SGS100A_1.timeout(1)

In [ ]:
# SGS100A_1.frequency(8.431e9)
# SGS100A_1.power(15.7)
SGS100A_1.frequency(100e6)
SGS100A_1.power(-100)
SGS100A_1.status('ON')
SGS100A_1.pulsemod_state('ON')
SGS100A_1.pulsemod_source('EXT')
SGS100A_1.pulsemod_trig_mode('EXT')
SGS100A_1.ref_osc_source('EXT')
SGS100A_1.ref_osc_external_freq('10MHz')

In [ ]:
SGS100A_1.on()

#### 1.3.3 GS210

In [ ]:
from qcodes.instrument_drivers.yokogawa import YokogawaGS200

In [ ]:
# Open http://172.16.0.28 in a browser (same machine/network as the instrument, no login needed) 
# → Status tab → note the "Command Socket Port" value (this GS210 unit: 7655, not the generic SCPI default 5025).
GS210_1 = find_or_create_instrument(YokogawaGS200, recreate=True, name='GS210_1', address='TCPIP0::172.16.0.28::7655::SOCKET')
# GS210_1 = find_or_create_instrument(YokogawaGS200, recreate=True, name='GS210_1', address='TCPIP0::172.16.0.28::7655::SOCKET', terminator='\n')
GS210_1.timeout(1)

In [ ]:
GS210_1.off()
GS210_1.source_mode('CURR')
GS210_1.auto_range(True)
GS210_1.current_limit(2e-3)
GS210_1.current(0.00e-3)

In [ ]:
GS210_1.current(0.715e-3)
# GS210_1.current(0.84e-3)
# GS210_1.current(0.8766e-3)
GS210_1.on()

In [ ]:
curr_down = np.linspace(1,0,401)
curr_down[::2] = -curr_down[::2]

for _ in range(1):
    for c in tqdm(curr_down):
        GS210_1.current(c*0.715e-3)
        time.sleep(0.05)

## 2.1 Resonator1

### 2.1.1 Basic Spectroscopy

In [ ]:
readout_module.out0_att(20)

In [ ]:
# comp_sched = device_compiler.compile(heterodyne_spec_sched_nco_TWPA(pulse_amp=1, pulse_duration=10e-6, frequencies=np.array([QUBIT1_READOUT_FREQ]), acquisition_delay=ACQ_DELAY, integration_time=10e-6,
#                                                                     port=qubit1.ports.readout(), clock=qubit1.name + ".ro",))
# comp_sched.plot_pulse_diagram(plot_backend="plotly")

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 800

spec_sched_kwargs = dict(
    pulse_amp=0.1,
    pulse_duration=300e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=300e-6,
    init_duration=20e-6,
    port=qubit1.ports.readout(),
    clock=qubit1.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(50)

center = QUBIT1_READOUT_FREQ
frequency_setpoints = np.linspace(center - 2e6, center + 1e6, 800)
meas_ctrl.settables(freq)
meas_ctrl.setpoints(frequency_setpoints)

rs_ds = meas_ctrl.run("resonator spectroscopy")
rs_ds.tuid

In [ ]:
rs_analysis = ResonatorSpectroscopyAnalysis(dataset=rs_ds)
rs_analysis.run().display_figs_mpl()

In [ ]:
print(f'f_res = {rs_analysis.quantities_of_interest["fr"].nominal_value/1e9:.6f} GHz')
print(f'QL = {rs_analysis.quantities_of_interest["Ql"].nominal_value/1e3:.0f} k')
print(f'Qc = {rs_analysis.quantities_of_interest["Qc"].nominal_value/1e3:.0f} k')
print(f'Qi = {rs_analysis.quantities_of_interest["Qi"].nominal_value/1e3:.0f} k')

# QUBIT1_READOUT_FREQ = rs_analysis.quantities_of_interest["fr"].nominal_value # uncomment it after find bare frequency

In [ ]:
qubit1.clock_freqs.readout(rs_analysis.quantities_of_interest["fr"].nominal_value) # dressed g
# qubit1.clock_freqs.readout(7158915451.501994)
# qubit1.clock_freqs.readout(np.NaN)

In [ ]:
# with open('resonator1_current_interp.pkl', 'rb') as file:
#     resonator1_current_interp = pickle.load(file)

In [ ]:
# resonator1_current_interp(0.0)

#### 2.1.2.1 Phase Noise

In [ ]:
hp_ssg.status('OFF')
hp_ssg.pulsemod_state('ON')

In [ ]:
readout_module.out0_att(20)

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 1000

ro_pulse_amp = ManualParameter(name="ro_pulse_amp", unit="", label="Readout pulse amplitude")
ro_pulse_amp.batched = False

spec_sched_kwargs = dict(
    pulse_amp=ro_pulse_amp,
    pulse_duration=10e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=10e-6,
    init_duration=10e-6,
    port=qubit1.ports.readout(),
    clock=qubit1.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
qubit1.clock_freqs.readout(np.nan)

In [ ]:
quantum_device.cfg_sched_repetitions(1)

center = QUBIT1_READOUT_FREQ
frequency_setpoints = np.linspace(center - 0.5e6, center + 0.5e6, 200)

meas_ctrl.settables([freq, ro_pulse_amp])
# meas_ctrl.setpoints_grid((frequency_setpoints, np.tile([0.1, 1], 100)*0.2))
meas_ctrl.setpoints_grid(([center], np.ones(10000)*0.2))


# meas_ctrl.verbose(False)

for n in tqdm(range(1)): 
    plotmon.tuids([])
    phase_noise_ds = meas_ctrl.run("phase noise")
    time.sleep(0)

# meas_ctrl.verbose(True)

In [ ]:
phase_noise_ds.tuid

#### 2.1.2.2 Current Dependent

In [ ]:
hp_ssg.status('ON')
hp_ssg.pulsemod_state('ON')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 400

spec_sched_kwargs = dict(
    pulse_amp=0.2,
    pulse_duration=50e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=50e-6,
    init_duration=10e-6,
    port=qubit1.ports.readout(),
    clock=qubit1.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)

center = QUBIT1_READOUT_FREQ
frequency_setpoints = np.linspace(center - 1e6, center + 1e6, 400)

meas_ctrl.settables([freq, GS210_1.current])
meas_ctrl.setpoints_grid((frequency_setpoints, np.linspace(-1, 1, 201)/1e3))

GS210_1.on()

current_dep_ds = meas_ctrl.run("current dependent", soft_avg=1)

GS210_1.off()
current_dep_ds.tuid

### 2.1.2 Cavity Map

In [ ]:
hp_ssg.status('OFF')
hp_ssg.pulsemod_state('ON')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 200

ro_pulse_amp = ManualParameter(name="ro_pulse_amp", unit="", label="Readout pulse amplitude")
ro_pulse_amp.batched = False

spec_sched_kwargs = dict(
    pulse_amp=ro_pulse_amp,
    pulse_duration=100e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=100e-6,
    init_duration=10e-6,
    port=qubit1.ports.readout(),
    clock=qubit1.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)
frequency_setpoints = np.linspace(7.145294e9-0.75e6, 7.145294e9+0.25e6, 200)

meas_ctrl.settables([freq, ro_pulse_amp])
meas_ctrl.setpoints_grid((frequency_setpoints, np.geomspace(0.1e-3, 0.3, 30)))

punchout_ds = meas_ctrl.run("resonator punchout")
punchout_ds

In [ ]:
cavity_map(punchout_ds)

### 2.1.3 Dressed E Frequency

In [ ]:
hp_ssg.status('ON')
hp_ssg.pulsemod_state('ON')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 400

spec_sched_kwargs = dict(
    pulse_amp=0.2,
    pulse_duration=4e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=4e-6,
    init_duration=10e-6,
    qubit=qubit1.name,
    port=qubit1.ports.readout(),
    clock=qubit1.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=dressed_e_cavity_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)
frequency_setpoints = np.linspace(7.145295e9 - 0.5e6, 7.145295e9 + 0.25e6, 400)

meas_ctrl.settables(freq)
meas_ctrl.setpoints(frequency_setpoints)

freq_e_ds = meas_ctrl.run("dressed e frequency", soft_avg=1)
freq_e_ds.tuid

In [ ]:
freq_e_ana = ResonatorSpectroscopyAnalysis(freq_e_ds)
freq_e_ana.run().display_figs_mpl()

freq_e = freq_e_ana.quantities_of_interest["fr"].nominal_value

In [ ]:
print(f'f_drs_e = {freq_e/1e9:.6f} GHz')

### 2.1.4 Dressed E Cavity Map

In [ ]:
hp_ssg.status('OFF')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 400

ro_pulse_amp = ManualParameter(name="ro_pulse_amp", unit="", label="Readout pulse amplitude")
ro_pulse_amp.batched = False

spec_sched_kwargs = dict(
    pulse_amp=ro_pulse_amp,
    pulse_duration=20e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=20e-6,
    init_duration=10e-6,
    qubit=qubit1.name,
    port=qubit1.ports.readout(),
    clock=qubit1.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=dressed_e_cavity_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)
frequency_setpoints = np.linspace(QUBIT1_READOUT_FREQ - 0.25e6, QUBIT1_READOUT_FREQ + 0.25e6, 400)

meas_ctrl.settables([freq, ro_pulse_amp])
meas_ctrl.setpoints_grid((frequency_setpoints, np.geomspace(0.1e-3, 1, 30)))

dressed_e_ds = meas_ctrl.run("dressed e punchout")
dressed_e_ds

In [ ]:
cavity_map(dressed_e_ds)

## 2.2 Resonator2

### 2.2.1 Basic Spectroscopy

In [ ]:
# GS210_1.current(-0.98968e-3)
# GS210_1.current(-0.885e-3)

In [ ]:
readout_module.out0_att(0)

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 800

spec_sched_kwargs = dict(
    pulse_amp=0.1,
    pulse_duration=200e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=200e-6,
    init_duration=10e-6,
    port=qubit2.ports.readout(),
    clock=qubit2.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
# SGS100A_1.status(1)

In [ ]:
quantum_device.cfg_sched_repetitions(100)

center = QUBIT2_READOUT_FREQ
frequency_setpoints = np.linspace(center - 2e6, center + 2e6, 800)
meas_ctrl.settables(freq)
meas_ctrl.setpoints(frequency_setpoints)

rs_ds = meas_ctrl.run("resonator spectroscopy", soft_avg=1)
rs_ds.tuid

In [ ]:
rs_analysis = ResonatorSpectroscopyAnalysis(tuid=rs_ds.attrs["tuid"], dataset=rs_ds)
rs_analysis.run().display_figs_mpl()

In [ ]:
print(f'f_res = {rs_analysis.quantities_of_interest["fr"].nominal_value/1e9:.6f} GHz')
print(f'QL = {rs_analysis.quantities_of_interest["Ql"].nominal_value/1e3:.0f} k')
print(f'Qc = {rs_analysis.quantities_of_interest["Qc"].nominal_value/1e3:.0f} k')
print(f'Qi = {rs_analysis.quantities_of_interest["Qi"].nominal_value/1e3:.0f} k')

# QUBIT2_READOUT_FREQ = rs_analysis.quantities_of_interest["fr"].nominal_value # uncomment it after find bare frequency

In [ ]:
qubit2.clock_freqs.readout(rs_analysis.quantities_of_interest["fr"].nominal_value) # dressed g
# qubit2.clock_freqs.readout(np.NaN)

#### 2.2.2.1 Phase Noise

In [ ]:
hp_ssg.status('ON')
hp_ssg.pulsemod_state('ON')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 1000

ro_pulse_amp = ManualParameter(name="ro_pulse_amp", unit="", label="Readout pulse amplitude")
ro_pulse_amp.batched = False

spec_sched_kwargs = dict(
    pulse_amp=ro_pulse_amp,
    pulse_duration=10e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=10e-6,
    init_duration=10e-6,
    port=qubit2.ports.readout(),
    clock=qubit2.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
qubit2.clock_freqs.readout(np.nan)

In [ ]:
quantum_device.cfg_sched_repetitions(1)

center = QUBIT2_READOUT_FREQ
frequency_setpoints = np.linspace(center - 0.75e6, center + 0.25e6, 200)

meas_ctrl.settables([freq, ro_pulse_amp])
# meas_ctrl.setpoints_grid((frequency_setpoints, np.tile([0.1, 1], 50)*0.2))
meas_ctrl.setpoints_grid(([center], np.ones(10000)*0.1))

# meas_ctrl.verbose(False)

for n in tqdm(range(1)): 
    phase_noise_ds = meas_ctrl.run("phase noise")
    # time.sleep(30)

# meas_ctrl.verbose(True)

In [ ]:
phase_noise_ds.tuid

In [ ]:
plt.plot(phase_noise_ds.y0)
plt.plot(phase_noise_ds.y1)

#### 2.2.2.2 Current Dependent

In [ ]:
hp_ssg.status('ON')
hp_ssg.pulsemod_state('ON')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 400

spec_sched_kwargs = dict(
    pulse_amp=0.006,
    pulse_duration=100e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=100e-6,
    init_duration=10e-6,
    port=qubit2.ports.readout(),
    clock=qubit2.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

center = QUBIT2_READOUT_FREQ
frequency_setpoints = np.linspace(center - 1e6, center + 1e6, 400)

meas_ctrl.settables([freq, GS210_1.current])
meas_ctrl.setpoints_grid((frequency_setpoints, np.linspace(0.54, 0.64, 51)/1e3))

GS210_1.on()

current_dep_ds = meas_ctrl.run("current dependent", soft_avg=1)

GS210_1.off()
current_dep_ds.tuid

### 2.2.2 Cavity Map

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 200

ro_pulse_amp = ManualParameter(name="ro_pulse_amp", unit="", label="Readout pulse amplitude")
ro_pulse_amp.batched = False

spec_sched_kwargs = dict(
    pulse_amp=ro_pulse_amp,
    pulse_duration=100e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=100e-6,
    init_duration=10e-6,
    port=qubit2.ports.readout(),
    clock=qubit2.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=heterodyne_spec_sched_nco_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)
frequency_setpoints = np.linspace(7.266314e9-0.75e6, 7.266314e9+0.25e6, 200)

meas_ctrl.settables([freq, ro_pulse_amp])
meas_ctrl.setpoints_grid((frequency_setpoints, np.geomspace(0.1e-3, 0.3, 30)))

punchout_ds = meas_ctrl.run("resonator punchout")
punchout_ds.tuid

In [ ]:
cavity_map(punchout_ds)

### 2.2.3 Dressed E Frequency

In [ ]:
hp_ssg.status('ON')
hp_ssg.pulsemod_state('ON')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 400

spec_sched_kwargs = dict(
    pulse_amp=0.2,
    pulse_duration=4e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=4e-6,
    init_duration=10e-6,
    qubit=qubit2.name,
    port=qubit2.ports.readout(),
    clock=qubit2.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=dressed_e_cavity_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)
frequency_setpoints = np.linspace(QUBIT2_READOUT_FREQ - 1e6, QUBIT2_READOUT_FREQ + 1e6, 400)

meas_ctrl.settables(freq)
meas_ctrl.setpoints(frequency_setpoints)

freq_e_ds = meas_ctrl.run("dressed e frequency", soft_avg=1)
freq_e_ds.tuid

In [ ]:
freq_e_ana = ResonatorSpectroscopyAnalysis(freq_e_ds)
freq_e_ana.run().display_figs_mpl()

freq_e = freq_e_ana.quantities_of_interest["fr"].nominal_value

In [ ]:
print(f'f_drs_e = {freq_e/1e9:.6f} GHz')

### 2.1.4 Dressed E Cavity Map

In [ ]:
hp_ssg.rf_output('ON')

In [ ]:
freq = ManualParameter(name="freq", unit="Hz", label="Frequency")
freq.batched = True
freq.batch_size = 100

ro_pulse_amp = ManualParameter(name="ro_pulse_amp", unit="", label="Readout pulse amplitude")
ro_pulse_amp.batched = False

spec_sched_kwargs = dict(
    pulse_amp=ro_pulse_amp,
    pulse_duration=10e-6,
    frequencies=freq,
    acquisition_delay=ACQ_DELAY,
    integration_time=10e-6,
    init_duration=10e-6,
    qubit=qubit.name,
    port=qubit.ports.readout(),
    clock=qubit.name + ".ro",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=TWPA_schedule.dressed_e_cavity_TWPA,
    schedule_kwargs=spec_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(400)
frequency_setpoints = np.linspace(QUBIT_READOUT_FREQ - 1e6, QUBIT_READOUT_FREQ + 4e6, 400)

meas_ctrl.settables([freq, ro_pulse_amp])
meas_ctrl.setpoints_grid((frequency_setpoints, np.geomspace(0.0001, 1, 30)))

dressed_e_ds = meas_ctrl.run("dressed e punchout")
dressed_e_ds

In [ ]:
cavity_map(dressed_e_ds)

## 3. Qubit1 (low power)

### 3.1 Qubit Spectroscopy

In [ ]:
nodes.resonator_calibration(qubit1)

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 500

spec_amp = ManualParameter(name="spec_amp", unit="V", label="Spec_Amplitude")
spec_amp.batched = False

In [ ]:
RO_AMP = 0.2
RO_LEN = 10e-6

In [ ]:
schedule_kwargs = dict(
    spec_pulse_amp = spec_amp,
    spec_pulse_duration = 200e-6, # long enough to stay at the mixed state
    spec_pulse_port = "qubit1:mw",
    spec_pulse_clock = "qubit1.01",
    spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
    ro_pulse_amp = RO_AMP, # choose the power from the cavity map that at the bottom of trace f_res
    ro_pulse_duration = RO_LEN, # can be small compared to qubit decay time
    ro_pulse_delay = 200e-9,
    ro_pulse_port = "qubit1:res",
    ro_pulse_clock = "qubit1.ro",
    ro_pulse_frequency = qubit1.clock_freqs.readout(), # just use resonator frequency
    ro_acquisition_delay = ACQ_DELAY,
    ro_integration_time = RO_LEN, # same as the ro_pulse_duration
    init_duration = 200e-6, # dead time long enough to reset the qubit >> T1
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=two_tone_spec_sched_nco_TWPA,
    schedule_kwargs=schedule_kwargs,
    real_imag=True,
    batched=True,
)

qubit1.measure.acq_delay(schedule_kwargs["ro_acquisition_delay"])
meas_ctrl.gettables(spec_gettable)

In [ ]:
# # # LO_FREQ_QUBIT1_arr = [2.575e9, 2.675e9]
LO_FREQ_QUBIT1_arr = np.arange(8.8, 8.9, 0.2)*1e9

for LO_FREQ_QUBIT1 in LO_FREQ_QUBIT1_arr:
    hardware_cfg['hardware_options']['modulation_frequencies']['qubit1:mw-qubit1.01']['lo_freq'] = LO_FREQ_QUBIT1
    quantum_device.hardware_config(hardware_cfg)

    quantum_device.cfg_sched_repetitions(200)
    # qubit1.clock_freqs.f01(np.NaN)
    
    spec_freq_setpoints = np.arange(LO_FREQ_QUBIT1+50e6, LO_FREQ_QUBIT1+250e6, 200e3)
    meas_ctrl.settables([spec_freq, spec_amp])
    meas_ctrl.setpoints_grid((spec_freq_setpoints, [1]))
    
    qs_ds = meas_ctrl.run("Two-tone", soft_avg=1)
    
    qs_analysis = QubitSpectroscopyAnalysis(dataset=qs_ds)
    qs_analysis.run().display_figs_mpl()

In [ ]:
# LO_FREQ_QUBIT1 = 3.76e9
# LO_FREQ_QUBIT1 = 4.05e9

# hardware_cfg['hardware_options']['modulation_frequencies']['qubit1:mw-qubit1.01']['lo_freq'] = LO_FREQ_QUBIT1
# quantum_device.hardware_config(hardware_cfg)

In [ ]:
quantum_device.cfg_sched_repetitions(100)
# qubit1.clock_freqs.f01(np.NaN)

spec_freq_setpoints = np.arange(4.507882e9-10e6, 4.507882e9+10e6, 50e3)
# spec_freq_setpoints = np.arange(3.90e9, 3.98e9, 200e3)
# spec_freq_setpoints = np.arange(7.57165e9/2-0.5e6+1.15e6, 7.57165e9/2+0.5e6+1.15e6, 5e3)

# spec_freq_setpoints = np.arange(np.loadtxt('Q1_curr_freq_fine_0818.csv')[64]-0.25e6, np.loadtxt('Q1_curr_freq_fine_0818.csv')[64]+0.25e6, 2.5e3)
meas_ctrl.settables([spec_freq, spec_amp])
meas_ctrl.setpoints_grid([spec_freq_setpoints, np.ones(1)*0.3e-2])
# meas_ctrl.setpoints_grid([spec_freq_setpoints, np.geomspace(1e-2, 1, 30)])

qs_ds = meas_ctrl.run("Two-tone", soft_avg=1)
qs_ds.tuid

In [ ]:
qs_analysis = QubitSpectroscopyAnalysis(dataset=qs_ds.where(qs_ds.x0 > 3.945e9, drop=True))
qs_analysis.run().display_figs_mpl()
# 3.9328 to 3.9270

In [ ]:
qubit1.clock_freqs.f01(qs_analysis.quantities_of_interest["frequency_01"].nominal_value)

In [ ]:
# "20260513-132738" Possible TLS? Weird doube peaks around f01
# "20260513-134219" Clean f02/2 peak
# left_freq = 3.930485; right freq = 3.94272,

In [ ]:
# qs_analysis_left = QubitSpectroscopyAnalysis(dataset=qs_ds.where(qs_ds.x0<3.20755e9, drop=True))
# qs_analysis_left.run().display_figs_mpl()
# qs_analysis_right = QubitSpectroscopyAnalysis(dataset=qs_ds.where(qs_ds.x0>3.20755e9, drop=True))
# qs_analysis_right.run().display_figs_mpl()

# qubit1.clock_freqs.f01(qs_analysis_left.quantities_of_interest["frequency_01"].nominal_value/2 + qs_analysis_right.quantities_of_interest["frequency_01"].nominal_value/2)

#### 3.1.1 Currend Dependent

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 800

In [ ]:
RO_AMP = 0.15
RO_LEN = 10e-6

In [ ]:
# Flux_R1_fit = np.load('Flux_R1_0622_fit.npz')
# Q1_curr_freq = np.loadtxt('Q1_curr_freq.csv')

R1_curr = np.loadtxt('R1_curr075_0623.csv', delimiter=',', skiprows=1)[:, 0]
R1_freq = np.loadtxt('R1_curr075_0623.csv', delimiter=',', skiprows=1)[:, 1]

In [ ]:
ic.timeout(180)

In [ ]:
qubit1.clock_freqs.readout(np.nan)
qubit1.clock_freqs.f01(np.nan)

In [ ]:
for i, c in enumerate(tqdm(np.linspace(0.79, 0.81, 11))):
    schedule_kwargs = dict(
        spec_pulse_amp = 0.05,
        spec_pulse_duration = 200e-6, # long enough to stay at the mixed state
        spec_pulse_port = "qubit1:mw",
        spec_pulse_clock = "qubit1.01",
        spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
        ro_pulse_amp = RO_AMP, # choose the power from the cavity map that at the bottom of trace f_res
        ro_pulse_duration = RO_LEN, # can be small compared to qubit decay time
        ro_pulse_delay = 200e-9,
        ro_pulse_port = "qubit1:res",
        ro_pulse_clock = "qubit1.ro",
        ro_pulse_frequency = qubit1.clock_freqs.readout(), # just use resonator frequency
        ro_acquisition_delay = ACQ_DELAY,
        ro_integration_time = RO_LEN, # same as the ro_pulse_duration
        init_duration = 200e-6, # dead time long enough to reset the qubit >> T1
    )
    spec_gettable = ScheduleGettable(
        quantum_device=quantum_device,
        schedule_function=two_tone_spec_sched_nco_TWPA,
        schedule_kwargs=schedule_kwargs,
        real_imag=False,
        batched=True,
    )
    
    meas_ctrl.gettables(spec_gettable)

    quantum_device.cfg_sched_repetitions(200)
    
    # spec_freq_setpoints = np.arange(4.509586e9-20e6, 4.509586e9+0e6, 50e3)
    spec_freq_setpoints = np.arange(4.5126e9-70e6, 4.5126e9+70e6, 400e3)
    # spec_freq_setpoints = np.arange(Q1_curr_freq[i]-2e6+1e6, Q1_curr_freq[i]+2e6+1e6, 20e3)
    meas_ctrl.settables(spec_freq)
    meas_ctrl.setpoints(spec_freq_setpoints)
    meas_ctrl.verbose(False)
    
    GS210_1.on()
    GS210_1.current(c/1e3)
    qs_ds = meas_ctrl.run("spectroscopy_current", soft_avg=1)
    GS210_1.off()

In [ ]:
# from quantify_scheduler.operations import NumericalPulse

# def rxy_custom_waveforms(th):
#         theta = round(_modulo_360_with_mapping(theta), 8)

#         phi = round(phi % 360, 8)

#         theta_r = np.deg2rad(theta)
#         phi_r = np.deg2rad(phi)

#         waveform_i =  np.cos(theta_r / 2),
#                     -1j * np.exp(-1j * phi_r) * np.sin(theta_r / 2),
#         wavefrom_q =      -1j * np.exp(1j * phi_r) * np.sin(theta_r / 2),

#     return waveform_i+waveform_q

# def custom_gate(qubit, complex_waveform)`a
#     sched = Schedule('custom',repititions)
#     if 'square':
#         squarepulse
#     if soft
#     sched.add(NumericalPulse(waveforms, port=qubit.ports.microwave()), clock=qubit.clock.f01)
#     retur sched

### 3.2 Rabi

In [ ]:
# comp_sched = device_compiler.compile(TWPA_schedule.rabi_sched_TWPA(qubit=qubit, frequency=qubit.clock_freqs.f01(), pulse_duration=100e-9, pulse_amp=10e-3))
# comp_sched.plot_pulse_diagram(plot_backend="plotly")

In [ ]:
# qubit1.measure.acq_delay(schedule_kwargs["ro_acquisition_delay"])

# qubit1.measure.pulse_amp(0.1)
# qubit1.measure.pulse_duration(10e-6)
# qubit1.measure.integration_time(10e-6)
# qubit1.reset.duration(300e-6)

In [ ]:
control_module1.out0_att(20)
qubit1.rxy.duration(100e-9)

In [ ]:
nodes.rabi_calibration(qubit1)

In [ ]:
rabi_pulse_amp = ManualParameter(name="rabi_pulse_amplitude", unit="V", label="Rabi Pulse Amplitude")
rabi_pulse_amp.batched = True
rabi_pulse_amp.batch_size = 200

rabi_sched_kwargs = {
    "pulse_amp": rabi_pulse_amp,
    "pulse_duration": qubit1.rxy.duration(),
    "frequency": qubit1.clock_freqs.f01(),
    "qubit": qubit1,
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=rabi_sched_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)

rabi_amplitude_setpoints = np.linspace(-1, 1, 101)

meas_ctrl.settables(rabi_pulse_amp)
meas_ctrl.setpoints(rabi_amplitude_setpoints)

rabi_ds = meas_ctrl.run("Rabi", soft_avg=1)
rabi_ds.tuid

In [ ]:
rabi_analysis = RabiAnalysis(dataset=rabi_ds)
rabi_analysis.run().display_figs_mpl()

In [ ]:
qubit1.rxy.amp180(rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value)
rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value

#### 3.3.1 Rabi Error Amplification

In [ ]:
nodes.rabi_amplification_calibration(qubit1, plot=True)            # with diagnostic plots

In [ ]:
meas_ctrl.verbose(True)

In [ ]:
rabi_pulse_amp = ManualParameter(name="rabi_pulse_amplitude", unit="V", label="Rabi Pulse Amplitude")
rabi_pulse_amp.batched = True
rabi_pulse_amp.batch_size = 100

pi_number = ManualParameter(name="number_of_pi", label="Number of Pi")

rabi_sched_kwargs = {
    "pulse_amp": rabi_pulse_amp,
    "pulse_duration": qubit1.rxy.duration(),
    "frequency": qubit1.clock_freqs.f01(),
    "qubit": qubit1,
    "pi_number": pi_number
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=rabi_amplification_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(50)

rabi_amplitude_setpoints = np.linspace(qubit1.rxy.amp180()-0.025, qubit1.rxy.amp180()+0.025, 21)
pi_number_setpoints = np.arange(0, 80, 2)

meas_ctrl.settables([rabi_pulse_amp, pi_number])
meas_ctrl.setpoints_grid([rabi_amplitude_setpoints, pi_number_setpoints])

rabi_amp_ds = meas_ctrl.run("Rabi Amplification", soft_avg=1)
rabi_amp_ds.tuid

In [ ]:
rabi_amplification_fit(rabi_amp_ds, plot=True)

In [ ]:
xlen = rabi_amp_ds.xlen
ylen = rabi_amp_ds.ylen

rabi_amp = np.unique(rabi_amp_ds.x0)
nb_rabi = np.unique(rabi_amp_ds.x1)
# y0data = rabi_amp_ds.y0.values.reshape(rabi_amp_ds.ylen,rabi_amp_ds.xlen)
y0data = np.abs(rotate_real_imag(rabi_amp_ds.y0.values, rabi_amp_ds.y1.values)[0].reshape(ylen, xlen))

plt.pcolor(rabi_amp, nb_rabi, y0data, rasterized=True)

plt.xlabel('Rabi Pulse Amplitude [V]')
plt.ylabel(f'Number of $\\pi$ Pulses')
plt.show()

In [ ]:
sum_pulses = y0data.sum(axis=0)
plt.plot(rabi_amp, sum_pulses)
plt.xlabel('Rabi Pulse Amplitude [V]')
plt.ylabel('Sum Along Pulses')

eliminator = np.where(sum_pulses>np.percentile(sum_pulses, 60))[0]

left_indice = eliminator[eliminator < np.argmin(sum_pulses)].max()
rihgt_indice = eliminator[eliminator > np.argmin(sum_pulses)].min()

rabi_amp_fit = rabi_amp[left_indice:rihgt_indice+1]
sum_pulses_fit = sum_pulses[left_indice:rihgt_indice+1]

US_fit = UnivariateSpline(rabi_amp_fit, sum_pulses_fit, k=4, s=1e-3)
US_x = np.linspace(rabi_amp_fit.min(), rabi_amp_fit.max(), 200)

plt.plot(rabi_amp_fit, sum_pulses_fit)
plt.plot(US_x, US_fit(US_x))

RABI_AMP = US_x[np.argmin(US_fit(US_x))]

In [ ]:
print(f'Rabi amp: {RABI_AMP}')
qubit1.rxy.amp180(RABI_AMP)

In [ ]:
from snl_qblox.helpers import rabi_amplification_fit

#### 3.2.2 Test Time Rabi

In [ ]:
rabi_pulse_duration = ManualParameter(name="rabi_pulse_duration", unit="s", label="Rabi Pulse Duration")
rabi_pulse_duration.batched = True
rabi_pulse_duration.batch_size = 10

rabi_sched_kwargs = {
    "pulse_amp": 0.3,
    "pulse_duration": rabi_pulse_duration,
    "frequency": qubit1.clock_freqs.f01(),
    "qubit": qubit1,
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=rabi_sched_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

rabi_duration_setpoints = np.arange(4e-9, 1000e-9, 12e-9)

meas_ctrl.settables(rabi_pulse_duration)
meas_ctrl.setpoints(rabi_duration_setpoints)

rabi_ds = meas_ctrl.run("Time Rabi", soft_avg=1)

rabi_ds.tuid

In [ ]:
rabi_analysis = RabiAnalysis(dataset=rabi_ds)
rabi_analysis.run().display_figs_mpl()

In [ ]:
qubit1.rxy.amp180(rabi_sched_kwargs["pulse_amp"])
qubit1.rxy.duration((rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value * 1e9) // 4 * 4e-9)

### 3.3 Ramsey

In [ ]:
# comp_sched = device_compiler.compile(ramsey_sched(qubit=qubit.name, times=qubit.rxy.duration(), artificial_detuning=0))
# comp_sched.plot_pulse_diagram(plot_backend="plotly")

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True

tau_setpoints = np.arange(qubit2.rxy.duration(), 20e-6, 0.1e-6)
tau.batch_size = tau_setpoints.size

ramsey_sched_kwargs = {
    "qubit": qubit1,
    "times": tau,
    "artificial_detuning": 200e3,
    "acq_protocol": "SSBIntegrationComplex",
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
    max_batch_size=tau_setpoints.size,
)
meas_ctrl.gettables(gettable)

In [ ]:
# meas_ctrl.settables(tau)
# meas_ctrl.setpoints(tau_setpoints)
meas_ctrl.settables([tau, SGS100A_1.phase])
meas_ctrl.setpoints_grid([tau_setpoints, np.zeros(100)])

quantum_device.cfg_sched_repetitions(200)
ramsey_ds = meas_ctrl.run("ramsey", soft_avg=1)
ramsey_ds.tuid
# 20260929-160042

In [ ]:
ramsey_analysis = RamseyAnalysis(dataset=ramsey_ds)
ramsey_analysis.run(
    artificial_detuning=ramsey_sched_kwargs["artificial_detuning"]
).display_figs_mpl()

In [ ]:
guess = decay_osci_model.guess(ramsey_ds.y0.values, t=ramsey_ds.x0.values*1e6)
fit_result = decay_osci_model.fit(ramsey_ds.y0.values, params=guess, t=ramsey_ds.x0.values*1e6)
fit_result.plot_fit()
plt.ylim(0, 1)
plt.grid()

plt.xlabel(r'$\tau$ [μs]')
plt.ylabel('Excited State Population')
plt.title('T2 Ramsey Decay Fit')
print(f"T2 = {round(fit_result.params['tau'].value, 1)} [μs]")

#### 3.3.0 Current Dependent

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 800

In [ ]:
RO_AMP = 0.15
RO_LEN = 10e-6

In [ ]:
Flux_R1_fit = np.load('Flux_R2_0622_fit.npz')
# Q1_curr_freq = np.loadtxt('Q1_curr_freq.csv')

In [ ]:
ic.timeout(60)

In [ ]:
qubit1.clock_freqs.readout(np.nan)
# qubit1.clock_freqs.f01(np.nan)

In [ ]:
for i, c in enumerate(tqdm(Flux_R1_fit['current'])):
    qubit1.clock_freqs.readout(Flux_R1_fit['fr_arr'][i]*1e9)
    tau = ManualParameter(name="tau", unit="s", label="Time")
    tau.batched = True
    tau_setpoints = np.arange(0, 10e-6, 0.05e-6)

    ramsey_sched_kwargs = {
        "qubit": qubit1,
        "times": tau,
        "artificial_detuning": 500e3,
    }

    gettable = ScheduleGettable(
        quantum_device,
        schedule_function=ramsey_sched_TWPA,
        schedule_kwargs=ramsey_sched_kwargs,
        real_imag=False,
        batched=True,
        max_batch_size=tau_setpoints.size,
    )
    meas_ctrl.gettables(gettable)

    quantum_device.cfg_sched_repetitions(100)
    
    meas_ctrl.settables(tau)
    meas_ctrl.setpoints(tau_setpoints)
    
    GS210_1.on()
    GS210_1.current(c/1e3)
    ramsey_ds = meas_ctrl.run("ramsey_current", soft_avg=1)
    GS210_1.off()

#### 3.3.1 Ramsey Chevron

In [ ]:
nodes.ramsey_chevron_calibration(qubit1, plot_data=True)

In [ ]:
qubit1.reset.duration(250e-6)

In [ ]:
qubit1.rxy.motzoi(0)

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True
tau_setpoints = np.arange(0, 10e-6, 0.02e-6)

detuning = ManualParameter(name="detuning", unit="Hz", label="Detuning")
detuning.batched = False
detuning_setpoints = np.linspace(-1000e3, 1000e3, 11)

ramsey_sched_kwargs = {
    "qubit": qubit1,
    "times": tau,
    "artificial_detuning": detuning,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
    max_batch_size=tau_setpoints.size,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables([tau, detuning])
meas_ctrl.setpoints_grid([tau_setpoints, detuning_setpoints])

quantum_device.cfg_sched_repetitions(100)
ram_che_ds = meas_ctrl.run("Q1 Ramsey Chevron", soft_avg=1)
ram_che_ds.tuid

In [ ]:
qubit1.clock_freqs.f01(ramsey_chevron_fit(ram_che_ds.tuid, 'qubit1', plot_fit=True, plot_data=True, real_imag=True))
# qubit1.clock_freqs.f01(3905385704.179070)

#### 3.3.2 Stark Ramsey

In [ ]:
from snl_qblox.TWPA_schedule import stark_ramsey_sched_TWPA

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True

stark_amp = ManualParameter(name="stark_amp", unit="V", label="Stark Pulse Amplitude")
stark_amp.batched = False

stark_freq = ManualParameter(name="stark_freq", unit="Hz", label="Stark Pulse Frequency")
stark_freq.batched = False

tau_setpoints = np.arange(qubit2.rxy.duration(), 40e-6, 1.0e-6)
tau.batch_size = tau_setpoints.size

stark_freq_setpoints = np.linspace(qubit1.clock_freqs.f01()-50e6, qubit1.clock_freqs.f01()+50e6, 41)

stark_ramsey_sched_kwargs = {
    "qubit": qubit1,
    "times": tau,
    "stark_amp": 0.6,
    "stark_freq": stark_freq,
    "artificial_detuning": 000e3,
    "acq_protocol": "SSBIntegrationComplex",
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=stark_ramsey_sched_TWPA,
    schedule_kwargs=stark_ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
    max_batch_size=tau_setpoints.size,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables([tau, stark_freq])
meas_ctrl.setpoints_grid([tau_setpoints, stark_freq_setpoints])

quantum_device.cfg_sched_repetitions(300)
stark_ramsey_ds = meas_ctrl.run("Stark Ramsey", soft_avg=1)
stark_ramsey_ds.tuid
# 20260819-114801

In [ ]:
x0_data = np.unique(stark_ramsey_ds.x0)
x1_data = np.unique(stark_ramsey_ds.x1)
y0data = rotate_real_imag(stark_ramsey_ds.y0.values, stark_ramsey_ds.y1.values)[0].reshape(x1_data.size, x0_data.size)
# y0data -= y0data.min()

T2_ram_arr = np.zeros(x1_data.size)
fit_f_arr = np.zeros(x1_data.size)

for i in range(x1_data.size):
    temp_dataset = stark_ramsey_ds.where(stark_ramsey_ds.x1==x1_data[i], drop=1)
    guess = decay_osci_model.guess(y0data[i], t=temp_dataset.x0.values)
    fit_result = decay_osci_model.fit(y0data[i], params=guess, t=temp_dataset.x0.values)

    if i == 40:
        fit_result.plot_fit()
        

    if fit_result.rsquared < 0.8:
        print(f"Fit for stark_freq={x1_data[i]} Hz has low R-squared: {fit_result.rsquared:.3f}. Skipping.")
        T2_ram_arr[i] = np.nan
        fit_f_arr[i] = np.nan
        continue
    T2_ram_arr[i] = fit_result.params['tau'].value
    fit_f_arr[i] = fit_result.params['frequency'].value

fit_f_arr -= stark_ramsey_sched_kwargs["artificial_detuning"]

In [ ]:
plt.pcolormesh(x0_data * 1e6, (x1_data-qubit1.clock_freqs.f01()) / 1e6, y0data)

In [ ]:
plt.plot((x1_data-qubit1.clock_freqs.f01()) / 1e6, fit_f_arr/1e6)
plt.grid()

### 3.4 T1

In [ ]:
# comp_sched = device_compiler.compile(t1_sched(qubit=qubit1.name, times=qubit1.rxy.duration()))
# comp_sched.plot_pulse_diagram(plot_backend="plotly")

In [ ]:
tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True

tau_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 200, 2))) * 1e-6
tau.batch_size = tau_setpoints.size # 180

t1_sched_kwargs = {"qubit": qubit1, "times": tau, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=t1_sched_TWPA,
    schedule_kwargs=t1_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables(tau)
meas_ctrl.setpoints(tau_setpoints)

quantum_device.cfg_sched_repetitions(100)

t1_ds = meas_ctrl.run("T1 experiment", soft_avg=1)
t1_ds.tuid

In [ ]:
# t1_analysis = T1Analysis(dataset=t1_ds)
# t1_analysis.run().display_figs_mpl()

In [ ]:
guess = exp_decay_model.guess(t1_ds.y0.values, delay=t1_ds.x0.values*1e6)
fig_result = exp_decay_model.fit(t1_ds.y0, t=t1_ds.x0*1e6, params=guess)
fig_result.plot_fit()
# plt.ylim(0, 1)
plt.grid()

plt.xlabel(r'$\tau$ [μs]')
plt.ylabel('Excited State Population')
plt.title('T1 Decay Fit')
print(f"T1 = {round(fig_result.params['tau'].value, 1)} [μs]")

### 3.5 T2 Echo

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True

tau_setpoints = np.hstack((np.arange(0, 80, 0.8), np.arange(80, 400, 4))) * 1e-6
tau.batch_size = tau_setpoints.size # 180

echo_sched_kwargs = {"qubit": qubit1, "times": tau, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=echo_sched_TWPA,
    schedule_kwargs=echo_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables(tau)
meas_ctrl.setpoints(tau_setpoints)

quantum_device.cfg_sched_repetitions(200)
echo_ds = meas_ctrl.run("T2 echo", soft_avg=1)
echo_ds.tuid

In [ ]:
guess = exp_decay_model.guess(echo_ds.y0.values, delay=echo_ds.x0.values*1e6)
fig_result = exp_decay_model.fit(echo_ds.y0, t=echo_ds.x0*1e6, params=guess)
fig_result.plot_fit()
plt.ylim(0, 1)
plt.grid()

plt.xlabel(r'$\tau$ [μs]')
plt.ylabel('Excited State Population')
plt.title('T2 Echo Decay Fit')
print(f"T2 = {round(fig_result.params['tau'].value, 1)} [μs]")

In [ ]:
# echo_analysis = EchoAnalysis(dataset=echo_ds)
# echo_analysis.run().display_figs_mpl()

In [ ]:
# Γ_φ = (113*1e-6)**(-1) - (2*106.8*1e-6)**(-1)
# Γ_φ

### 3.6 T1&T2 Alternating

In [ ]:
tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True
tau.batch_size = 200

case = ManualParameter(name="case", label="Case")

t1_and_t2_sched_kwargs = {"qubit": qubit1, "times": tau, "case": case, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=t1_and_t2_TWPA,
    schedule_kwargs=t1_and_t2_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
tau_setpoints = np.hstack((np.arange(0, 40, 2), np.arange(40, 200, 4))) * 1e-6
case_setpoints = np.tile([1,2], 10)

meas_ctrl.settables([tau, case])
meas_ctrl.setpoints_grid([tau_setpoints, case_setpoints])

quantum_device.cfg_sched_repetitions(200)
t1_and_t2_ds = meas_ctrl.run("T1 and T2", soft_avg=1)
t1_and_t2_ds.tuid

### 3.9 ALLXY

In [ ]:
elements = ManualParameter(name="elements")
elements.batched = True
elements.batch_size = 21

allxy_sched_kwargs = {"qubit": qubit1, "element_select_idx": elements}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=allxy_sched_TWPA,
    schedule_kwargs=allxy_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables(elements)
meas_ctrl.setpoints(np.arange(21))

quantum_device.cfg_sched_repetitions(1000)
allxy_ds = meas_ctrl.run("ALLXY", soft_avg=1)

allxy_ds.tuid

In [ ]:
allxy_analysis = AllXYAnalysis(dataset=allxy_ds)
allxy_analysis.run().display_figs_mpl()

#### 3.10.1 DRAG Pulse Calibration

In [ ]:
nodes.drag_calibration(qubit1)

In [ ]:
motzoi_list = np.linspace(-0.1, 0.1, 41)
drag_ds1 = DRAG_calibration_sched(qubit=qubit1, motzoi_list=motzoi_list, instrument_coordinator=ic, quantum_device=quantum_device, repetitions=1000)

In [ ]:
qubit1.rxy.motzoi(DRAG_fit(drag_ds1, motzoi_list))

### 3.10 Threshold/Readout calibration

In [ ]:
nodes.readout_calibration(qubit1, plot=True)

In [ ]:
# readout_module.out0_att(40)
# qubit1.measure.pulse_amp(0.1)
# qubit2.measure.pulse_duration(10e-6)
# qubit2.measure.integration_time(10e-6)

qubit1.measure.pulse_amp(0.21)
qubit1.measure.pulse_duration(3.9e-6)
qubit1.measure.integration_time(3.9e-6)

In [ ]:
states = ManualParameter(name="states")
states.batched = True
states.batch_size = 600

RO_cali_sched_kwargs = {"qubit": qubit1, "prepared_states": states, "acq_protocol": "NumericalSeparatedWeightedIntegration"} 
# SSBIntegrationComplex or NumericalSeparatedWeightedIntegration

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=readout_calibration_sched_TWPA,
    schedule_kwargs=RO_cali_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
states_setpoints = np.hstack((np.zeros(3000), np.ones(3000)))

meas_ctrl.settables(states)
meas_ctrl.setpoints(states_setpoints)

quantum_device.cfg_sched_repetitions(1)
ro_ds = meas_ctrl.run("Readout Calibration")
ro_ds.tuid

In [ ]:
ro_analysis = ReadoutCalibrationAnalysis(dataset=ro_ds.where(ro_ds.x0!=2, drop=True))
# ro_analysis = ReadoutCalibrationAnalysis(dataset=ro_ds)
ro_analysis.run().display_figs_mpl()

threshold = ro_analysis.quantities_of_interest['acq_threshold'].nominal_value
rotation = (np.rad2deg(ro_analysis.quantities_of_interest['acq_rotation_rad'].nominal_value)) % 360
qubit1.measure.acq_threshold(threshold)
qubit1.measure.acq_rotation(rotation)

In [ ]:
# ro_ds_copy = ro_ds.copy()
# ro_ds_copy.x0.values = np.where(ro_ds_copy.x0.values==2, 1, ro_ds_copy.x0.values)
# ro_analysis = ReadoutCalibrationAnalysis(dataset=ro_ds_copy.where(ro_ds_copy.x0!=2, drop=True))
# ro_analysis.run().display_figs_mpl()

In [ ]:
threshold = ro_analysis.quantities_of_interest['acq_threshold'].nominal_value
rotation = (np.rad2deg(ro_analysis.quantities_of_interest['acq_rotation_rad'].nominal_value)) % 360
qubit1.measure.acq_threshold(threshold)
qubit1.measure.acq_rotation(rotation)

# qubit1.measure.acq_threshold(np.NaN)
# qubit1.measure.acq_rotation(np.NaN)

In [ ]:
rotate_IQ_blob(ro_ds)

### 3.11 Readout Optimization

In [ ]:
qubit1.measure.pulse_amp(0.18)
qubit1.measure.pulse_duration(3.9e-6)
qubit1.measure.integration_time(3.9e-6)

#### 3.11.1 RO FREQ

In [ ]:
nodes.ro_freq_optimization(qubit1, plot=True)

In [ ]:
# frequency_list = np.linspace(qubit1.clock_freqs.readout()- 0.2e6, qubit1.clock_freqs.readout() + 0.2e6, 41)
# test_ds = readout_freq_optimization_TWPA(qubit=qubit1, frequencies=frequency_list, 
#     repetitions=1000, instrument_coordinator=ic, quantum_device=quantum_device)

#### 3.11.2 RO AMP

In [ ]:
nodes.ro_amp_optimization(qubit1, amp_setpoints=np.linspace(0.2, 0.5, 41), plot=True)

In [ ]:
# amplitude_list = np.linspace(qubit1.measure.pulse_amp() * 0.8, qubit1.measure.pulse_amp() * 1.2, 41)
# test_ds = readout_amp_optimization_TWPA(qubit=qubit1, amps=amplitude_list, 
#     repetitions=500, instrument_coordinator=ic, quantum_device=quantum_device)

#### 3.11.3 RO LEN

In [ ]:
nodes.ro_len_optimization(qubit1, len_setpoints=np.arange(2.0, 6.1, 0.1)/1e6, plot=True)

In [ ]:
# test_ds = readout_len_optimization_TWPA(qubit=qubit1, lens=[4e-6], 
#     repetitions=100, instrument_coordinator=ic, quantum_device=quantum_device)

#### 3.11.5 RO Weight

In [ ]:
nodes.ro_weight_calibration(qubit1, plot=True)

In [ ]:
test_ds = readout_weight_optimization_TWPA(qubit=qubit1,
    repetitions=5000, instrument_coordinator=ic, quantum_device=quantum_device)

In [ ]:
def normalize_complex_array(arr):
    # Calculate the simple norm of the complex array
    norm = np.sqrt(np.sum(np.abs(arr) ** 2))

    # Normalize the complex array by dividing it by the norm
    normalized_arr = arr / norm

    # Rescale the normalized array so that the maximum value is 1
    max_val = np.max(np.abs(normalized_arr))
    rescaled_arr = normalized_arr / max_val

    return rescaled_arr

g_trace = test_ds[0][0].values.reshape(-1)
e_trace = test_ds[1][0].values.reshape(-1)

time_trace = np.arange(0, qubit1.measure.pulse_duration()-0.5e-9, 1e-9)
NCO_freq = qubit1.clock_freqs.readout() - quantum_device.generate_hardware_config()['hardware_options']['modulation_frequencies']['qubit1:res-qubit1.ro']['lo_freq']

demodulated_g_trace = g_trace * np.exp(-2j*np.pi * NCO_freq * (time_trace)) 
demodulated_e_trace = e_trace * np.exp(-2j*np.pi * NCO_freq * (time_trace))

divition_length = 10
demodulated_g_div = np.mean(demodulated_g_trace.reshape(-1, divition_length), axis=1)
demodulated_e_div = np.mean(demodulated_e_trace.reshape(-1, divition_length), axis=1)

subtracted_trace = demodulated_e_div - demodulated_g_div
norm_subtracted_trace = normalize_complex_array(subtracted_trace)

In [ ]:
plt.plot(time_trace.reshape(-1, divition_length)[:, 0]*1e6, norm_subtracted_trace.real, label='real')
plt.plot(time_trace.reshape(-1, divition_length)[:, 0]*1e6, norm_subtracted_trace.imag, label='imag')

plt.title("SNR")
plt.xlabel("Readout time [ns]")
plt.ylabel("subtracted traces [a.u.]")
plt.legend()

In [ ]:
weights_a = np.repeat(norm_subtracted_trace.real, divition_length)
# weights_a = weights_a / np.abs(weights_a).sum()

weights_b = np.repeat(norm_subtracted_trace.imag, divition_length)
# weights_b = weights_b / np.abs(weights_b).sum()

In [ ]:
weights_b.sum()

In [ ]:
qubit1.measure.acq_weights_a(np.abs(weights_a))
qubit1.measure.acq_weights_b(np.abs(weights_b))

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# dt = time_trace[1] - time_trace[0]          # sample spacing (1e-9 s)
# N = len(demodulated_g_trace)

# fft_vals = np.fft.fftshift(np.fft.fft(demodulated_g_trace))
# baseband_freqs = np.fft.fftshift(np.fft.fftfreq(N, d=dt))

# abs_freqs = baseband_freqs + NCO_freq        # shift back to absolute RF frequency

# magnitude = np.abs(fft_vals) / N             # normalize by number of samples
# magnitude_dB = 20 * np.log10(magnitude / np.max(magnitude))  # dB relative to peak

# plt.figure()
# plt.plot(abs_freqs / 1e6, magnitude_dB)
# plt.xlabel("Absolute Frequency (MHz)")
# plt.ylabel("Magnitude (dB)")
# plt.title("FFT of demodulated_g_trace (absolute frequency)")
# plt.ylim(-70, 10)
# plt.xlim(-100, 100)
# # plt.grid(True)
# plt.show()

### 3.12 Randomized Benchmarking

In [ ]:
nodes.randomized_benchmarking(qubit1)

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import RBAnalysis, randomized_benchmarking_schedule

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of Clifford gates")
length.batched = True
length_setpoints = np.arange(0, 341, 20)

seed = ManualParameter(name="seed", unit="", label="Seeds")
seed.batched = True
seed_setpoints = np.random.randint(0, 2**31 - 1, size=20, dtype=np.int32)

rb_sched_kwargs = {
    "qubit_specifier": qubit1,
    "lengths": length,
    "seeds": seed,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=randomized_benchmarking_schedule,
    schedule_kwargs=rb_sched_kwargs,
    real_imag=False,
    batched=True,
    num_channels=1,
    max_batch_size=length_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

meas_ctrl.settables([length, seed])
meas_ctrl.setpoints_grid([length_setpoints, seed_setpoints])

quantum_device.cfg_sched_repetitions(300)

In [ ]:
rb_coherent_error_ds = meas_ctrl.run("Randomized benchmarking on Q1")
display(rb_coherent_error_ds.tuid)

In [ ]:
# rb_coherent_error_ds = load_dataset('20250829-1635')

rb_coherent_error_result = RBAnalysis(
    dataset=rb_coherent_error_ds,
    label="Randomized benchmarking",
    settings_overwrite={"mpl_transparent_background": False},
    repetitions=gettable.compiled_schedule.repetitions,
).run()
print(f"Depolarizing parameter: \u03b1={round(rb_coherent_error_result.alpha, 4)}")
print(f"Error per Clifford: r_c={np.format_float_scientific(rb_coherent_error_result.r, precision=3)}")
print(f"Clifford Fidelity: F_c={round((1 - rb_coherent_error_result.r) * 100, 3)}%")
print(f"Error per Gate: r_g={np.format_float_scientific(rb_coherent_error_result.r / 1.875, precision=3)}")
print(f"Gate Fidelity: F_g={round((1 - rb_coherent_error_result.r / 1.875) * 100, 3)}%")
print(
    f"Sample size explains {100 * rb_coherent_error_result.sample_size_variance:.1f} (+{100 * rb_coherent_error_result.szv_hi_err:.1f}, -{100 * rb_coherent_error_result.szv_lo_err:.1f})% of the variance (2\u03c3 confidence). High variance may indicate coherent errors."
)
display(rb_coherent_error_ds.tuid)
rb_coherent_error_result.display_figs_mpl()

In [ ]:
if_plot = True
ro_roadmap = [
    ('RO Frequency Optimization', 'ro_freq_optimization', {'plot': if_plot}),
    ('RO Amplitude Optimization', 'ro_amp_optimization',   {'amp_setpoints': np.linspace(0.15, 0.25, 41), 'plot': if_plot}),
    ('RO Duration Optimization',  'ro_len_optimization',   {'len_setpoints': np.arange(2.0, 6.1, 0.1)/1e6, 'plot': if_plot}),
]

n_loops = 2
all_logs = []
for i in range(n_loops):
    print(f'\n########## Loop {i+1}/{n_loops} ##########')
    log = nodes.run_roadmap(ro_roadmap, [qubit2])
    all_logs.append(log)

In [ ]:
rb_roadmap = [
    ('Rabi',              'rabi_calibration',                {}),
    ('Rabi Amplification','rabi_amplification_calibration',  {}),
    ('DRAG',              'drag_calibration',                {}),
    ('ALLXY',             'allxy_calibration',               {}),
    ('RB',                'randomized_benchmarking',         {}),
]

for d in range(80, 101, 4):
    qubit1.rxy.duration(d*1e-9)
    print(f'Running RB with duration {d*1e-9:.2e} s')
    log = nodes.run_roadmap(rb_roadmap, [qubit1], stop_on_fail=True)

### 3.13 E-F Transition

In [ ]:
freq_g = 7150163814.785255
freq_e = 7150102761.57382

In [ ]:
qubit1.clock_freqs.readout(freq_e)

#### 3.13.1 Spectroscopy

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 400

# spec_amp = ManualParameter(name="spec_amp", unit="V", label="Spec_Amplitude")
# spec_amp.batched = False

spec_dur = ManualParameter(name="spec_dur", unit="S", label="Spec_Duration")
spec_dur.batched = False

In [ ]:
schedule_kwargs = dict(
    qubit = qubit1,
    spec_pulse_amp = 2e-2,
    spec_pulse_duration = spec_dur,
    spec_pulse_port = "qubit1:fs",
    spec_pulse_clock = "qubit1.12",
    spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=f_state_spec_sched_nco_TWPA,
    schedule_kwargs=schedule_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(spec_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

spec_freq_setpoints = np.arange(3.699489e9-5e6, 3.699489e9+5e6, 40e3)
meas_ctrl.settables([spec_freq, spec_dur])
meas_ctrl.setpoints_grid((spec_freq_setpoints, [30e-6]))

fs_ds = meas_ctrl.run("fs spec", soft_avg=1)
fs_ds.tuid

In [ ]:
fs_analysis = QubitSpectroscopyAnalysis(dataset=fs_ds)
fs_analysis.run().display_figs_mpl()

In [ ]:
qubit1.clock_freqs.f12(fs_analysis.quantities_of_interest["frequency_01"].nominal_value)
qubit1.clock_freqs.f12()

#### 3.13.2 Rabi

In [ ]:
rabi_pulse_amp = ManualParameter(name="rabi_pulse_amplitude", unit="V", label="Rabi Pulse Amplitude")
rabi_pulse_amp.batched = True
rabi_pulse_amp.batch_size = 200

rabi_sched_kwargs = {
    "pulse_amp": rabi_pulse_amp,
    "pulse_duration": 400e-9,
    "frequency": qubit1.clock_freqs.f12(),
    "qubit": qubit1,
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=f_state_rabi_sched_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

rabi_amplitude_setpoints = np.linspace(-1, 1, 200)

meas_ctrl.settables(rabi_pulse_amp)
meas_ctrl.setpoints(rabi_amplitude_setpoints)

rabi_fs = meas_ctrl.run("fs Rabi", soft_avg=1)
rabi_fs.tuid

In [ ]:
rabi_analysis = RabiAnalysis(dataset=rabi_fs)
rabi_analysis.run().display_figs_mpl()

In [ ]:
rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value

#### 3.13.3 F-state T1

In [ ]:
tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True
tau.batch_size = 400

case = ManualParameter(name="case", label="Case")

fs_t1_sched_kwargs = {"qubit": qubit1, "times": tau, "case": case, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=f_state_t1_TWPA,
    schedule_kwargs=fs_t1_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
delay_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 200, 4))) * 1e-6
case_setpoints = np.tile([1,2], 50)

meas_ctrl.settables([tau, case])
meas_ctrl.setpoints_grid([delay_setpoints, case_setpoints])

quantum_device.cfg_sched_repetitions(100)
fs_t1_ds = meas_ctrl.run("F State T1", soft_avg=1)
fs_t1_ds.tuid

In [ ]:
# 20260520-153757 post ge and ef, ge pulse; 50 data points

fs_t1_ds = load_dataset('20260520-153757')

xlen = fs_t1_ds.xlen
ylen = fs_t1_ds.ylen

tau = fs_t1_ds.x0.values.reshape((ylen, xlen))[0] * 1e6
cases = fs_t1_ds.x1.values.reshape((ylen, xlen)).T[0]
y0data = fs_t1_ds.y0.values.reshape((ylen, xlen))

In [ ]:
exp_decay_model.set_param_hint("tau", value=30, min=0)
exp_decay_model.set_param_hint("offset", value=y0data[0][-1])
exp_decay_model.set_param_hint("amplitude", value=y0data[0][0] - y0data[0][-1])
exp_decay_model.set_param_hint("n_factor", expr="1", vary=False)

t1_arr = []

for i in range(0, ylen, 2):
    fit_result = exp_decay_model.fit(y0data[i], t=tau)
    t1_arr.append(fit_result.params['tau'].value)

t1_arr = np.array(t1_arr)

exp_decay_model.set_param_hint("tau", value=20, min=0)
exp_decay_model.set_param_hint("offset", value=y0data[1][-1])
exp_decay_model.set_param_hint("amplitude", value=y0data[1][0] - y0data[1][-1])
exp_decay_model.set_param_hint("n_factor", expr="1", vary=False)

fs_t1_arr = []

for i in range(1, ylen, 2):
    fit_result = exp_decay_model.fit(y0data[i], t=tau)
    fs_t1_arr.append(fit_result.params['tau'].value)

fs_t1_arr = np.array(fs_t1_arr)

In [ ]:
plt.plot(tau, 1-y0data[0:ylen:2][40])
plt.plot(tau, 1-y0data[1:ylen:2][40])

In [ ]:
t1_arr.mean()

In [ ]:
fs_t1_arr.mean()

In [ ]:
plt.plot(t1_arr / 2)
plt.plot(fs_t1_arr)

In [ ]:
plt.scatter(t1_arr / 2, fs_t1_arr)

## 4. Qubit2 (low power)

### 4.1 Qubit Spectroscopy

In [ ]:
# readout_module.out0_att(40)

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 500

spec_amp = ManualParameter(name="spec1_amp", unit="V", label="Spec_Amplitude")
spec_amp.batched = False

In [ ]:
RO_AMP = 0.2
RO_LEN = 10e-6

In [ ]:
schedule_kwargs = dict(
    spec_pulse_amp = spec_amp,
    spec_pulse_duration = 200e-6, # long enough to stay at the mixed state
    spec_pulse_port = "qubit2:mw",
    spec_pulse_clock = "qubit2.01",
    spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
    ro_pulse_amp = RO_AMP, # choose the power from the cavity map that at the bottom of trace f_res
    ro_pulse_duration = RO_LEN, # can be small compared to qubit decay time
    ro_pulse_delay = 200e-9,
    ro_pulse_port = "qubit2:res",
    ro_pulse_clock = "qubit2.ro",
    ro_pulse_frequency = qubit2.clock_freqs.readout(), # just use resonator frequency
    ro_acquisition_delay = ACQ_DELAY,
    ro_integration_time = RO_LEN, # same as the ro_pulse_duration
    init_duration = 200e-6, # dead time long enough to reset the qubit >> T1
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=two_tone_spec_sched_nco_TWPA,
    schedule_kwargs=schedule_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(spec_gettable)

In [ ]:
control_module2.out0_att(10)
control_module2.out0_att()

In [ ]:
# # # LO_FREQ_QUBIT2_arr = [2.575e9, 3.25e9]
LO_FREQ_QUBIT2_arr = np.arange(4.0, 4.5, 0.2)*1e9

for LO_FREQ_QUBIT2 in LO_FREQ_QUBIT2_arr:
    hardware_cfg['hardware_options']['modulation_frequencies']['qubit2:mw-qubit2.01']['lo_freq'] = LO_FREQ_QUBIT2
    quantum_device.hardware_config(hardware_cfg)

    quantum_device.cfg_sched_repetitions(100)
    
    spec_freq_setpoints = np.arange(LO_FREQ_QUBIT2+50e6, LO_FREQ_QUBIT2+250e6, 200e3)
    meas_ctrl.settables([spec_freq, spec_amp])
    meas_ctrl.setpoints_grid((spec_freq_setpoints, [1]))
    
    qs_ds = meas_ctrl.run("Two-tone", soft_avg=1)
    
    qs_analysis = QubitSpectroscopyAnalysis(dataset=qs_ds)
    qs_analysis.run().display_figs_mpl()

In [ ]:
# LO_FREQ_QUBIT2 = 4.29e9

# hardware_cfg['hardware_options']['modulation_frequencies']['qubit2:mw-qubit2.01']['lo_freq'] = LO_FREQ_QUBIT2
# quantum_device.hardware_config(hardware_cfg)

In [ ]:
quantum_device.cfg_sched_repetitions(100)
# qubit2.clock_freqs.f01(np.NaN)

spec_freq_setpoints = np.arange(4.769544e9-20e6, 4.769544e9+10e6, 100e3)
# spec_freq_setpoints = np.arange(4.2e9, 4.24e9, 200e3)
# spec_freq_setpoints = np.arange(np.loadtxt('Q2_curr_freq.csv')[64]-0.25e6, np.loadtxt('Q2_curr_freq.csv')[64]+0.25e6, 2.5e3)
meas_ctrl.settables([spec_freq, spec_amp])
meas_ctrl.setpoints_grid([spec_freq_setpoints, np.ones(1)*3e-2])

qs_ds = meas_ctrl.run("Two-tone", soft_avg=1)
qs_ds.tuid

In [ ]:
qs_analysis = QubitSpectroscopyAnalysis(dataset=qs_ds)
qs_analysis.run().display_figs_mpl()

In [ ]:
qubit2.clock_freqs.f01(qs_analysis.quantities_of_interest["frequency_01"].nominal_value)

In [ ]:
# qs_analysis_left = QubitSpectroscopyAnalysis(dataset=qs_ds.where(qs_ds.x0<3.3254e9, drop=True))
# qs_analysis_left.run().display_figs_mpl()
# qs_analysis_right = QubitSpectroscopyAnalysis(dataset=qs_ds.where(qs_ds.x0>3.3254e9, drop=True))
# qs_analysis_right.run().display_figs_mpl()

# qubit2.clock_freqs.f01(qs_analysis_left.quantities_of_interest["frequency_01"].nominal_value/2 + qs_analysis_right.quantities_of_interest["frequency_01"].nominal_value/2)

#### 4.1.1 Currend Dependent

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 800

In [ ]:
RO_AMP = 0.15
RO_LEN = 10e-6

In [ ]:
# Flux_R2_fit = np.load('Flux_R2_0622_fit.npz')
# Q2_curr_freq = np.loadtxt('Q2_curr_freq.csv')

R2_curr = np.loadtxt('R2_curr075_0623.csv', delimiter=',', skiprows=1)[:, 0]
R2_freq = np.loadtxt('R2_curr075_0623.csv', delimiter=',', skiprows=1)[:, 1]

In [ ]:
ic.timeout(180)

In [ ]:
qubit2.clock_freqs.readout(np.nan)
qubit2.clock_freqs.f01(np.nan)

In [ ]:
for i, c in enumerate(tqdm(np.linspace(0.765, 0.825, 21))):
    schedule_kwargs = dict(
        spec_pulse_amp = 0.01,
        spec_pulse_duration = 200e-6, # long enough to stay at the mixed state
        spec_pulse_port = "qubit2:mw",
        spec_pulse_clock = "qubit2.01",
        spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
        ro_pulse_amp = RO_AMP, # choose the power from the cavity map that at the bottom of trace f_res
        ro_pulse_duration = RO_LEN, # can be small compared to qubit decay time
        ro_pulse_delay = 200e-9,
        ro_pulse_port = "qubit2:res",
        ro_pulse_clock = "qubit2.ro",
        ro_pulse_frequency = qubit2.clock_freqs.readout(), # just use resonator frequency
        ro_acquisition_delay = ACQ_DELAY,
        ro_integration_time = RO_LEN, # same as the ro_pulse_duration
        init_duration = 200e-6, # dead time long enough to reset the qubit >> T1
    )
    spec_gettable = ScheduleGettable(
        quantum_device=quantum_device,
        schedule_function=two_tone_spec_sched_nco_TWPA,
        schedule_kwargs=schedule_kwargs,
        real_imag=True,
        batched=True,
    )
    
    meas_ctrl.gettables(spec_gettable)

    quantum_device.cfg_sched_repetitions(200)
        
    # spec_freq_setpoints = np.arange(4.769544e9-20e6, 4.769544e9+0e6, 50e3)
    spec_freq_setpoints = np.arange(4.772e9-70e6, 4.772e9+70e6, 400e3)
    # spec_freq_setpoints = np.arange(Q2_curr_freq[i]-1e6, Q2_curr_freq[i]+1e6, 10e3)
    meas_ctrl.settables(spec_freq)
    meas_ctrl.setpoints(spec_freq_setpoints)
    meas_ctrl.verbose(False)
    
    GS210_1.on()
    GS210_1.current(c/1e3)
    qs_ds = meas_ctrl.run("spectroscopy_current", soft_avg=1)
    GS210_1.off()

In [ ]:
temp_mag = np.abs(qs_ds.y0 + 1j*qs_ds.y1).values
temp_mag = qs_ds.y1.values
temp_freq = qs_ds.x0.values
fit_result = lorentzian_model.fit(temp_mag, params=lorentzian_model.guess(temp_mag, x=temp_freq), x=temp_freq)
display(fit_result.params['x0'].value)
fit_result.plot_fit()

### 4.2 Rabi

In [ ]:
# qubit2.measure.acq_delay(schedule_kwargs["ro_acquisition_delay"])

# qubit2.measure.pulse_amp(0.1)
# qubit2.measure.pulse_duration(10e-6)
# qubit2.measure.integration_time(10e-6)
# qubit2.reset.duration(300e-6)

In [ ]:
control_module2.out0_att(20)
qubit2.rxy.duration(100e-9)

In [ ]:
rabi_pulse_amp = ManualParameter(name="rabi_pulse_amplitude", unit="V", label="Rabi Pulse Amplitude")
rabi_pulse_amp.batched = True
rabi_pulse_amp.batch_size = 200

rabi_sched_kwargs = {
    "pulse_amp": rabi_pulse_amp,
    "pulse_duration": qubit2.rxy.duration(),
    "frequency": qubit2.clock_freqs.f01(),
    "qubit": qubit2,
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=rabi_sched_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)

rabi_amplitude_setpoints = np.linspace(-1, 1, 101)

meas_ctrl.settables(rabi_pulse_amp)
meas_ctrl.setpoints(rabi_amplitude_setpoints)

rabi_ds = meas_ctrl.run("Rabi", soft_avg=1)

rabi_ds.tuid

In [ ]:
rabi_analysis = RabiAnalysis(dataset=rabi_ds)
rabi_analysis.run().display_figs_mpl()

In [ ]:
qubit2.rxy.amp180(rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value)
rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value

#### 4.3.2 Rabi Error Amplification

In [ ]:
rabi_pulse_amp = ManualParameter(name="rabi_pulse_amplitude", unit="V", label="Rabi Pulse Amplitude")
rabi_pulse_amp.batched = True
rabi_pulse_amp.batch_size = 100

pi_number = ManualParameter(name="number_of_pi", label="Number of Pi")

rabi_sched_kwargs = {
    "pulse_amp": rabi_pulse_amp,
    "pulse_duration": qubit2.rxy.duration(),
    "frequency": qubit2.clock_freqs.f01(),
    "qubit": qubit2,
    "pi_number": pi_number
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=rabi_amplification_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(50)

rabi_amplitude_setpoints = np.linspace(qubit2.rxy.amp180()-0.025, qubit2.rxy.amp180()+0.025, 21)
pi_number_setpoints = np.arange(0, 80, 2)

meas_ctrl.settables([rabi_pulse_amp, pi_number])
meas_ctrl.setpoints_grid([rabi_amplitude_setpoints, pi_number_setpoints])

rabi_amp_ds = meas_ctrl.run("Rabi Amplification", soft_avg=1)
rabi_amp_ds.tuid

In [ ]:
xlen = rabi_amp_ds.xlen
ylen = rabi_amp_ds.ylen

rabi_amp = np.unique(rabi_amp_ds.x0)
nb_rabi = np.unique(rabi_amp_ds.x1)
# y0data = rabi_amp_ds.y0.values.reshape(rabi_amp_ds.ylen,rabi_amp_ds.xlen)
y0data = np.abs(rotate_real_imag(rabi_amp_ds.y0.values, rabi_amp_ds.y1.values)[0].reshape(ylen, xlen))

plt.pcolor(rabi_amp, nb_rabi, y0data, rasterized=True)

plt.xlabel('Rabi Pulse Amplitude [V]')
plt.ylabel(f'Number of $\\pi$ Pulses')
plt.show()

In [ ]:
sum_pulses = y0data.sum(axis=0)
plt.plot(rabi_amp, sum_pulses)
plt.xlabel('Rabi Pulse Amplitude [V]')
plt.ylabel('Sum Along Pulses')

eliminator = np.where(sum_pulses>np.percentile(sum_pulses, 60))[0]

left_indice = eliminator[eliminator < np.argmin(sum_pulses)].max()
rihgt_indice = eliminator[eliminator > np.argmin(sum_pulses)].min()

rabi_amp_fit = rabi_amp[left_indice:rihgt_indice+1]
sum_pulses_fit = sum_pulses[left_indice:rihgt_indice+1]

US_fit = UnivariateSpline(rabi_amp_fit, sum_pulses_fit, k=4, s=1e-3)
US_x = np.linspace(rabi_amp_fit.min(), rabi_amp_fit.max(), 200)

plt.plot(rabi_amp_fit, sum_pulses_fit)
plt.plot(US_x, US_fit(US_x))

RABI_AMP = US_x[np.argmin(US_fit(US_x))]

In [ ]:
print(f'Rabi amp: {RABI_AMP}')
qubit2.rxy.amp180(RABI_AMP)

#### 4.2.1 Test Time Rabi

In [ ]:
rabi_pulse_duration = ManualParameter(name="rabi_pulse_duration", unit="s", label="Rabi Pulse Duration")
rabi_pulse_duration.batched = True
rabi_pulse_duration.batch_size = 10

rabi_sched_kwargs = {
    "pulse_amp": qubit2.rxy.amp180(),
    "pulse_duration": rabi_pulse_duration,
    "frequency": qubit2.clock_freqs.f01(),
    "qubit": qubit2,
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=rabi_sched_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

rabi_duration_setpoints = np.arange(4e-9, 1600e-9, 20e-9)

meas_ctrl.settables(rabi_pulse_duration)
meas_ctrl.setpoints(rabi_duration_setpoints)

rabi_ds = meas_ctrl.run("Time Rabi", soft_avg=1)

rabi_ds.tuid

In [ ]:
rabi_analysis = RabiAnalysis(dataset=rabi_ds)
rabi_analysis.run().display_figs_mpl()

In [ ]:
qubit2.rxy.amp180(rabi_sched_kwargs["pulse_amp"])
qubit2.rxy.duration((rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value * 1e9) // 4 * 4e-9)

### 4.3 Ramsey

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True
tau.batch_size = 400

ramsey_sched_kwargs = {
    "qubit": qubit2,
    "times": tau,
    "artificial_detuning": 200e3,
    "acq_protocol": "SSBIntegrationComplex",
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
tau_setpoints = np.arange(qubit2.rxy.duration(), 100e-6, 0.8e-6)

meas_ctrl.settables(tau)
meas_ctrl.setpoints(tau_setpoints)

quantum_device.cfg_sched_repetitions(200)
ramsey_ds = meas_ctrl.run("ramsey", soft_avg=1)
ramsey_ds.tuid

In [ ]:
# ramsey_ds = ramsey_ds.rename({'y0':'y2'})
# ramsey_ds = ramsey_ds.rename({'y1':'y0'})

In [ ]:
ramsey_analysis = RamseyAnalysis(dataset=ramsey_ds)
ramsey_analysis.run(
    artificial_detuning=ramsey_sched_kwargs["artificial_detuning"]
).display_figs_mpl()

In [ ]:
guess = decay_osci_model.guess(ramsey_ds.y0.values, t=ramsey_ds.x0.values*1e6)
fit_result = decay_osci_model.fit(ramsey_ds.y0.values, params=guess, t=ramsey_ds.x0.values*1e6)
fit_result.plot_fit()
plt.ylim(0, 1)
plt.grid()

plt.xlabel(r'$\tau$ [μs]')
plt.ylabel('Excited State Population')
plt.title('T2 Ramsey Decay Fit')
print(f"T2 = {round(fit_result.params['tau'].value, 1)} [μs]")

#### 4.3.1 Ramsey Chevron

In [ ]:
nodes.ramsey_chevron_calibration(qubit2)

In [ ]:
qubit2.rxy.motzoi(0)

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True
tau_setpoints = np.arange(0, 20e-6, 0.2e-6)

detuning = ManualParameter(name="detuning", unit="Hz", label="Detuning")
detuning.batched = False
detuning_setpoints = np.linspace(-300e3, 300e3, 11)

ramsey_sched_kwargs = {
    "qubit": qubit2,
    "times": tau,
    "artificial_detuning": detuning,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables([tau, detuning])
meas_ctrl.setpoints_grid([tau_setpoints, detuning_setpoints])

quantum_device.cfg_sched_repetitions(100)
ram_che_ds = meas_ctrl.run("ramsey chevron", soft_avg=1)
ram_che_ds.tuid

In [ ]:
qubit2.clock_freqs.f01(ramsey_chevron_fit(ram_che_ds.tuid, 'qubit2', plot_data=True, plot_fit=True, real_imag=True))
# qubit2.clock_freqs.f01(4769071156.303957)
# 20260720-145453 with TLS

#### 4.3.2 Stark Ramsey

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True

stark_amp = ManualParameter(name="stark_amp", unit="V", label="Stark Pulse Amplitude")
stark_amp.batched = False

stark_freq = ManualParameter(name="stark_freq", unit="Hz", label="Stark Pulse Frequency")
stark_freq.batched = False

tau_setpoints = np.arange(qubit2.rxy.duration(), 40e-6, 1.e-6)
tau.batch_size = tau_setpoints.size

stark_freq_setpoints = np.linspace(qubit2.clock_freqs.f01()+50e6, qubit2.clock_freqs.f01()+250e6, 21)

ramsey_sched_kwargs = {
    "qubit": qubit2,
    "times": tau,
    "stark_amp": 1,
    "stark_freq": stark_freq,
    "artificial_detuning": 500e3,
    "acq_protocol": "SSBIntegrationComplex",
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=stark_ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
    max_batch_size=tau_setpoints.size,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables([tau, stark_freq])
meas_ctrl.setpoints_grid([tau_setpoints, stark_freq_setpoints])

quantum_device.cfg_sched_repetitions(100)
stark_ramsey_ds = meas_ctrl.run("Stark Ramsey", soft_avg=1)
stark_ramsey_ds.tuid
# 20260819-114801

In [ ]:
x0_data = np.unique(stark_ramsey_ds.x0)
x1_data = np.unique(stark_ramsey_ds.x1)
y0data = rotate_real_imag(stark_ramsey_ds.y0.values, stark_ramsey_ds.y1.values)[0].reshape(x1_data.size, x0_data.size)
# y0data -= y0data.min()

T2_ram_arr = np.zeros(x1_data.size)
fit_f_arr = np.zeros(x1_data.size)

for i in range(x1_data.size):
    temp_dataset = stark_ramsey_ds.where(stark_ramsey_ds.x1==x1_data[i], drop=1)
    guess = decay_osci_model.guess(y0data[i], t=temp_dataset.x0.values)
    fit_result = decay_osci_model.fit(y0data[i], params=guess, t=temp_dataset.x0.values)

    if i == 10:
        fit_result.plot_fit()

    if fit_result.rsquared < 0.8:
        print(f"Fit for stark_freq={x1_data[i]} Hz has low R-squared: {fit_result.rsquared:.3f}. Skipping.")
        T2_ram_arr[i] = np.nan
        fit_f_arr[i] = np.nan
        continue
    T2_ram_arr[i] = fit_result.params['tau'].value
    fit_f_arr[i] = fit_result.params['frequency'].value

fit_f_arr -= stark_ramsey_sched_kwargs["artificial_detuning"]

In [ ]:
plt.pcolormesh(x0_data * 1e6, (x1_data-qubit2.clock_freqs.f01()) / 1e6, y0data)

In [ ]:
plt.plot((x1_data-qubit2.clock_freqs.f01()) / 1e6, fit_f_arr/1e6)

### 4.4 T1

In [ ]:
nodes.T1_calibration(qubit2)

In [ ]:
tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True

delay_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 200, 4))) * 1e-6
tau.batch_size = delay_setpoints.size

t1_sched_kwargs = {"qubit": qubit2, "times": tau, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=t1_sched_TWPA,
    schedule_kwargs=t1_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables(tau)
meas_ctrl.setpoints(delay_setpoints)

quantum_device.cfg_sched_repetitions(200)
t1_ds = meas_ctrl.run("T1 experiment", soft_avg=1)
t1_ds.tuid # '20260210-151439-709-393af1'

In [ ]:
guess = exp_decay_model.guess(t1_ds.y0.values, delay=t1_ds.x0.values*1e6)
fig_result = exp_decay_model.fit(t1_ds.y0, t=t1_ds.x0*1e6, params=guess)
fig_result.plot_fit()
plt.ylim(0, 1)
plt.grid()

plt.xlabel(r'$\tau$ [μs]')
plt.ylabel('Excited State Population')
plt.title('T1 Decay Fit')
print(f"T1 = {round(fig_result.params['tau'].value, 1)} [μs]")

In [ ]:
# t1_analysis = T1Analysis(dataset=t1_ds)
# t1_analysis.run().display_figs_mpl()

### 4.5 T2 Echo

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True

tau_setpoints = np.hstack((np.arange(0, 80, 0.8), np.arange(80, 400, 4))) * 1e-6
tau.batch_size = tau_setpoints.size # 180

echo_sched_kwargs = {"qubit": qubit2, "times": tau, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=echo_sched_TWPA,
    schedule_kwargs=echo_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables(tau)
meas_ctrl.setpoints(tau_setpoints)

quantum_device.cfg_sched_repetitions(200)
echo_ds = meas_ctrl.run("T2 echo", soft_avg=1)
echo_ds.tuid

In [ ]:
guess = exp_decay_model.guess(echo_ds.y0.values, delay=echo_ds.x0.values*1e6)
fig_result = exp_decay_model.fit(echo_ds.y0, t=echo_ds.x0*1e6, params=guess)
fig_result.plot_fit()
plt.ylim(0, 1)
plt.grid()

plt.xlabel(r'$\tau$ [μs]')
plt.ylabel('Excited State Population')
plt.title('T2 Echo Decay Fit')
print(f"T2 = {round(fig_result.params['tau'].value, 1)} [μs]")

In [ ]:
# echo_analysis = EchoAnalysis(dataset=echo_ds)
# echo_analysis.run().display_figs_mpl()

### 4.6 T1&T2 Alternating

In [ ]:
tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True
tau.batch_size = 200

case = ManualParameter(name="case", label="Case")

t1_and_t2_sched_kwargs = {"qubit": qubit2, "times": tau, "case": case, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=t1_and_t2_TWPA,
    schedule_kwargs=t1_and_t2_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
tau_setpoints = np.hstack((np.arange(0, 40, 2), np.arange(40, 200, 4))) * 1e-6
case_setpoints = np.tile([1,2], 10)

meas_ctrl.settables([tau, case])
meas_ctrl.setpoints_grid([tau_setpoints, case_setpoints])

quantum_device.cfg_sched_repetitions(200)
t1_and_t2_ds = meas_ctrl.run("T1 and T2", soft_avg=1)
t1_and_t2_ds.tuid

### 4.9 ALLXY

In [ ]:
elements = ManualParameter(name="elements")
elements.batched = True
elements.batch_size = 21

allxy_sched_kwargs = {"qubit": qubit2, "element_select_idx": elements}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=allxy_sched_TWPA,
    schedule_kwargs=allxy_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
meas_ctrl.settables(elements)
meas_ctrl.setpoints(np.arange(21))

quantum_device.cfg_sched_repetitions(1000)
allxy_ds = meas_ctrl.run("ALLXY", soft_avg=1)

allxy_ds.tuid

In [ ]:
allxy_analysis = AllXYAnalysis(dataset=allxy_ds)
allxy_analysis.run().display_figs_mpl()

#### 4.9.1 DRAG Pulse Calibration

In [ ]:
motzoi_list = np.linspace(-0.1, 0.1, 41)
drag_ds2 = DRAG_calibration_sched(qubit=qubit2, motzoi_list=motzoi_list, instrument_coordinator=ic, quantum_device=quantum_device, repetitions=1000)

In [ ]:
qubit2.rxy.motzoi(DRAG_fit(drag_ds2, motzoi_list))

### 4.10 Threshold/Readout calibration

In [ ]:
qubit2.measure.pulse_amp(0.225)
# qubit2.measure.pulse_amp(0.21)
qubit2.measure.pulse_duration(4e-6)
qubit2.measure.integration_time(4e-6)

In [ ]:
meas_ctrl.verbose(True)

In [ ]:
states = ManualParameter(name="states")
states.batched = True
states.batch_size = 600

RO_cali_sched_kwargs = {"qubit": qubit2, "prepared_states": states, "acq_protocol": "SSBIntegrationComplex"} 
# SSBIntegrationComplex or NumericalSeparatedWeightedIntegration

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=readout_calibration_sched_TWPA,
    schedule_kwargs=RO_cali_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
states_setpoints = np.hstack((np.zeros(3000), np.ones(3000)))

meas_ctrl.settables(states)
meas_ctrl.setpoints(states_setpoints)

quantum_device.cfg_sched_repetitions(1)
ro_ds = meas_ctrl.run("Readout Calibration")
ro_ds.tuid # 20260211-123414

In [ ]:
ro_analysis = ReadoutCalibrationAnalysis(dataset=ro_ds.where(ro_ds.x0!=2, drop=True))
# ro_analysis = ReadoutCalibrationAnalysis(dataset=ro_ds)
ro_analysis.run().display_figs_mpl()

threshold = ro_analysis.quantities_of_interest['acq_threshold'].nominal_value
rotation = (np.rad2deg(ro_analysis.quantities_of_interest['acq_rotation_rad'].nominal_value)) % 360
qubit2.measure.acq_threshold(threshold)
qubit2.measure.acq_rotation(rotation)

In [ ]:
# ro_ds_copy = ro_ds.copy()
# ro_ds_copy.x0.values = np.where(ro_ds_copy.x0.values==2, 1, ro_ds_copy.x0.values)
# ro_analysis = ReadoutCalibrationAnalysis(dataset=ro_ds_copy.where(ro_ds_copy.x0!=2, drop=True))
# ro_analysis.run().display_figs_mpl()

In [ ]:
threshold = ro_analysis.quantities_of_interest['acq_threshold'].nominal_value
rotation = (np.rad2deg(ro_analysis.quantities_of_interest['acq_rotation_rad'].nominal_value)) % 360
qubit2.measure.acq_threshold(threshold)
qubit2.measure.acq_rotation(rotation)

# qubit2.measure.acq_threshold(np.NaN)
# qubit2.measure.acq_rotation(np.NaN)

In [ ]:
rotate_IQ_blob(ro_ds)

In [ ]:
g_blob = ro_ds.where(ro_ds.x0==0, drop=True)
e_blob = ro_ds.where(ro_ds.x0==1, drop=True)
f_blob = ro_ds.where(ro_ds.x0==2, drop=True)

In [ ]:
plt.scatter(g_blob.y0, g_blob.y1, s=2, alpha=0.5, color='C0')
plt.scatter(e_blob.y0, e_blob.y1, s=2, alpha=0.5, color='C1')
plt.scatter(f_blob.y0, f_blob.y1, s=2, alpha=0.5, color='C2')

plt.scatter(g_blob.y0.mean(), g_blob.y1.mean(), s=20, color='C0', edgecolors='black')
plt.scatter(e_blob.y0.mean(), e_blob.y1.mean(), s=20, color='C1', edgecolors='black')
plt.scatter(f_blob.y0.mean(), f_blob.y1.mean(), s=20, color='C2', edgecolors='black')

plt.gca().set_aspect('equal', 'box')
plt.xlabel('I [mV]')
plt.ylabel('Q [mV]')
plt.show()

### 4.11 Readout Optimization

#### 4.11.1 RO FREQ

In [ ]:
nodes.ro_freq_optimization(qubit2, plot=True, repetitions=1500)

#### 4.11.2 RO AMP

In [ ]:
nodes.ro_amp_optimization(qubit2, plot=True, repetitions=1500)

#### 4.11.3 RO LEN

### 4.12 Randomized Benchmarking

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of Clifford gates")
length.batched = True
length_setpoints = np.arange(0, 32, 1)

seed = ManualParameter(name="seed", unit="", label="Seeds")
seed.batched = True
seed_setpoints = np.random.randint(0, 2**31 - 1, size=20, dtype=np.int32)

rb_sched_kwargs = {
    "qubit_specifier": qubit2,
    "lengths": length,
    "seeds": seed,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=randomized_benchmarking_schedule,
    schedule_kwargs=rb_sched_kwargs,
    real_imag=False,
    batched=True,
    num_channels=1,
    max_batch_size=length_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

meas_ctrl.settables([length, seed])
meas_ctrl.setpoints_grid([length_setpoints, seed_setpoints])

quantum_device.cfg_sched_repetitions(300)

In [ ]:
rb_coherent_error_ds = meas_ctrl.run("Randomized benchmarking on Q2")

display(rb_coherent_error_ds.tuid)

In [ ]:
# rb_coherent_error_ds = load_dataset('20250911-1508')

rb_coherent_error_result = RBAnalysis(
    dataset=rb_coherent_error_ds,
    label="Randomized benchmarking",
    settings_overwrite={"mpl_transparent_background": False},
    repetitions=gettable.compiled_schedule.repetitions,
).run()
print(f"Depolarizing parameter: \u03b1={round(rb_coherent_error_result.alpha, 4)}")
print(f"Error per Clifford: r_c={np.format_float_scientific(rb_coherent_error_result.r, precision=3)}")
print(f"Clifford Fidelity: F_c={round((1 - rb_coherent_error_result.r) * 100, 3)}%")
print(f"Error per Gate: r_g={np.format_float_scientific(rb_coherent_error_result.r / 1.875, precision=3)}")
print(f"Gate Fidelity: F_g={round((1 - rb_coherent_error_result.r / 1.875) * 100, 3)}%")
print(
    f"Sample size explains {100 * rb_coherent_error_result.sample_size_variance:.1f} (+{100 * rb_coherent_error_result.szv_hi_err:.1f}, -{100 * rb_coherent_error_result.szv_lo_err:.1f})% of the variance (2\u03c3 confidence). High variance may indicate coherent errors."
)
rb_coherent_error_ds.tuid
rb_coherent_error_result.display_figs_mpl()

In [ ]:
# for i in range(10):
#     if i % 2 == 0:
#         qubit2.rxy.motzoi(0)
#     else:
#         qubit2.rxy.motzoi(qubit2_motzoi)

#     hp_ssg.pulsemod_state('OFF')

#     rb_coherent_error_ds = meas_ctrl.run(
#         "Randomized benchmarking on " + rb_sched_kwargs["qubit_specifier"]
#     )
    
#     hp_ssg.pulsemod_state('ON')
#     display(rb_coherent_error_ds.tuid)

### 4.13 E-F Transition

In [ ]:
freq_g = 7272249120.414608
freq_e = 7272140167.56538

In [ ]:
qubit2.clock_freqs.readout(freq_e)

#### 4.13.1 Spectroscopy

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 400

# spec_amp = ManualParameter(name="spec_amp", unit="V", label="Spec_Amplitude")
# spec_amp.batched = False

spec_dur = ManualParameter(name="spec_dur", unit="S", label="Spec_Duration")
spec_dur.batched = False

In [ ]:
schedule_kwargs = dict(
    qubit = qubit2,
    spec_pulse_amp = 2e-2,
    spec_pulse_duration = spec_dur,
    spec_pulse_port = "qubit2:fs",
    spec_pulse_clock = "qubit2.12",
    spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=f_state_spec_sched_nco_TWPA,
    schedule_kwargs=schedule_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(spec_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

spec_freq_setpoints = np.arange(3.9814e9-2e6, 3.9814e9+2e6, 20e3)
meas_ctrl.settables([spec_freq, spec_dur])
meas_ctrl.setpoints_grid((spec_freq_setpoints, [30e-6]))

fs_ds = meas_ctrl.run("fs spec", soft_avg=1)
fs_ds

In [ ]:
fs_analysis = QubitSpectroscopyAnalysis(dataset=fs_ds)
fs_analysis.run().display_figs_mpl()

In [ ]:
qubit2.clock_freqs.f12(fs_analysis.quantities_of_interest["frequency_01"].nominal_value)
qubit2.clock_freqs.f12(3981413451.9768195)

#### 4.13.2 Rabi

In [ ]:
rabi_pulse_amp = ManualParameter(name="rabi_pulse_amplitude", unit="V", label="Rabi Pulse Amplitude")
rabi_pulse_amp.batched = True
rabi_pulse_amp.batch_size = 200

rabi_sched_kwargs = {
    "pulse_amp": rabi_pulse_amp,
    "pulse_duration": 200e-9,
    "frequency": qubit2.clock_freqs.f12(),
    "qubit": qubit2,
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=f_state_rabi_sched_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

rabi_amplitude_setpoints = np.linspace(-1, 1, 200)

meas_ctrl.settables(rabi_pulse_amp)
meas_ctrl.setpoints(rabi_amplitude_setpoints)

rabi_fs = meas_ctrl.run("fs Rabi", soft_avg=1)
rabi_fs.tuid

In [ ]:
rabi_analysis = RabiAnalysis(dataset=rabi_fs)
rabi_analysis.run().display_figs_mpl()

In [ ]:
rabi_analysis.quantities_of_interest["Pi-pulse amplitude"].nominal_value

In [ ]:
dres_g_ge_rabi_tuid = '20260513-155821'
dres_g_ef_rabi_tuid = '20260513-155803'

dres_e_ge_rabi_tuid = '20260513-155433'
dres_e_ef_rabi_tuid = '20260513-155415'

dres_g_ge_rabi_ds = load_dataset(dres_g_ge_rabi_tuid)
dres_g_ef_rabi_ds = load_dataset(dres_g_ef_rabi_tuid)

dres_e_ge_rabi_ds = load_dataset(dres_e_ge_rabi_tuid)
dres_e_ef_rabi_ds = load_dataset(dres_e_ef_rabi_tuid)

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(dres_g_ge_rabi_ds.y0.values, dres_g_ge_rabi_ds.y1.values, c='C0', s=10, alpha=0.5)
# plt.scatter(dres_g_ge_rabi_ds.y0.values[100], dres_g_ge_rabi_ds.y1.values[100], c='C0', s=300, alpha=1)
plt.scatter(dres_g_ef_rabi_ds.y0.values, dres_g_ef_rabi_ds.y1.values, c='C0', s=10, alpha=0.5, marker='*')
# plt.scatter(dres_g_ef_rabi_ds.y0.values[100], dres_g_ef_rabi_ds.y1.values[100], c='C0', s=300, marker='*', alpha=1)

plt.scatter(dres_e_ge_rabi_ds.y0.values, dres_e_ge_rabi_ds.y1.values, c='C1', s=10, alpha=0.5)
# plt.scatter(dres_e_ge_rabi_ds.y0.values[100], dres_e_ge_rabi_ds.y1.values[100], c='C1', s=300, alpha=1)
plt.scatter(dres_e_ef_rabi_ds.y0.values, dres_e_ef_rabi_ds.y1.values, c='C1', s=10, alpha=0.5, marker='*')
# plt.scatter(dres_e_ef_rabi_ds.y0.values[100], dres_e_ef_rabi_ds.y1.values[100], c='C1', s=300, marker='*', alpha=1)

plt.scatter(0, 0, c='black')
plt.gca().set_aspect('equal', 'box')
plt.xlabel('I [mV]')
plt.ylabel('Q [mV]')
plt.show()

#### 4.13.3 F-state T1

In [ ]:
tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True
tau.batch_size = 400

case = ManualParameter(name="case", label="Case")

fs_t1_sched_kwargs = {"qubit": qubit2, "times": tau, "case": case, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=f_state_t1_TWPA,
    schedule_kwargs=fs_t1_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
delay_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 200, 4))) * 1e-6
case_setpoints = np.tile([1,2], 50)

meas_ctrl.settables([tau, case])
meas_ctrl.setpoints_grid([delay_setpoints, case_setpoints])

quantum_device.cfg_sched_repetitions(100)
fs_t1_ds = meas_ctrl.run("F State T1", soft_avg=1)
fs_t1_ds.tuid

In [ ]:
# 20260518-170908 
# 20260518-171621 post ef, ge pulse
# 20260519-122929 post ge and ef, ge pulse
# 20260519-123636 post ge and ef, ge pulse; 50 data points

fs_t1_ds = load_dataset('20260519-123636')

xlen = fs_t1_ds.xlen
ylen = fs_t1_ds.ylen

tau = fs_t1_ds.x0.values.reshape((ylen, xlen))[0] * 1e6
cases = fs_t1_ds.x1.values.reshape((ylen, xlen)).T[0]
y0data = fs_t1_ds.y0.values.reshape((ylen, xlen))

In [ ]:
exp_decay_model.set_param_hint("tau", value=30, min=0)
exp_decay_model.set_param_hint("offset", value=y0data[0][-1])
exp_decay_model.set_param_hint("amplitude", value=y0data[0][0] - y0data[0][-1])
exp_decay_model.set_param_hint("n_factor", expr="1", vary=False)

t1_arr = []

for i in range(0, ylen, 2):
    fit_result = exp_decay_model.fit(y0data[i], t=tau)
    t1_arr.append(fit_result.params['tau'].value)

t1_arr = np.array(t1_arr)

exp_decay_model.set_param_hint("tau", value=20, min=0)
exp_decay_model.set_param_hint("offset", value=y0data[1][-1])
exp_decay_model.set_param_hint("amplitude", value=y0data[1][0] - y0data[1][-1])
exp_decay_model.set_param_hint("n_factor", expr="1", vary=False)

fs_t1_arr = []

for i in range(1, ylen, 2):
    fit_result = exp_decay_model.fit(y0data[i], t=tau)
    fs_t1_arr.append(fit_result.params['tau'].value)

fs_t1_arr = np.array(fs_t1_arr)

In [ ]:
plt.plot(tau, 1-y0data[0:ylen:2][0])
plt.plot(tau, 1-y0data[1:ylen:2][0])

In [ ]:
t1_arr.mean()

In [ ]:
fs_t1_arr.mean()

In [ ]:
plt.plot(t1_arr / 2)
plt.plot(fs_t1_arr)

In [ ]:
plt.scatter(t1_arr / 2, fs_t1_arr)

### 4.14 Cavity Charging Test

In [ ]:
comp_sched = device_compiler.compile(cavity_charging_TWPA(qubit=qubit2,
    pulse_amp=[0.5],
    frequency=qubit2.clock_freqs.readout(),
    port=qubit2.ports.readout(),
    clock=qubit2.name + ".ro",
    acq_protocol="ThresholdedAcquisition"))
comp_sched.plot_pulse_diagram(plot_backend="plotly")

In [ ]:
qubit2.reset.duration(400e-6)

In [ ]:
charging_pulse_amp = ManualParameter(name="charging_pulse_amplitude", unit="V", label="Charging Pulse Amplitude")
charging_pulse_amp.batched = True

charging_pulse_amp_setpoints = np.linspace(0, 1, 101)
charging_pulse_amp.batch_size = charging_pulse_amp_setpoints.size

charging_sched_kwargs = dict(
    qubit=qubit2,
    pulse_amp=charging_pulse_amp,
    frequency=qubit2.clock_freqs.readout(),
    port=qubit2.ports.readout(),
    clock=qubit2.name + ".ro",
    acq_protocol="SSBIntegrationComplex",
)
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=cavity_charging_TWPA,
    schedule_kwargs=charging_sched_kwargs,
    real_imag=False,
    batched=True,
)

meas_ctrl.gettables(gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(200)

meas_ctrl.settables(charging_pulse_amp)
meas_ctrl.setpoints(charging_pulse_amp_setpoints)

charging_ds = meas_ctrl.run("cavity charging", soft_avg=1)
charging_ds.tuid

In [ ]:
plt.plot(charging_ds.x0.values, charging_ds.y0.values, 'o-')
plt.xlabel('Charging Pulse Amplitude [V]')
plt.ylabel('Readout Signal [mV]')
# plt.ylabel('Excited State Population')
plt.title('Cavity Charging')

## 5. SNAIL

### 5.1 SNAIL Spectroscopy

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 500

spec_amp = ManualParameter(name="spec_amp", unit="V", label="Spec_Amplitude")
spec_amp.batched = False

In [ ]:
RO_AMP = 3e-3
RO_LEN = 20e-6

In [ ]:
schedule_kwargs = dict(
    spec_pulse_amp = 1,
    spec_pulse_duration = 300e-6, # long enough to stay at the mixed state
    spec_pulse_port = "snail:mw",
    spec_pulse_clock = "snail.01",
    spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
    ro_pulse_amp = RO_AMP, # choose the power from the cavity map that at the bottom of trace f_res
    ro_pulse_duration = RO_LEN, # can be small compared to qubit decay time
    ro_pulse_delay = 200e-9,
    ro_pulse_port = "snail:res",
    ro_pulse_clock = "snail.ro",
    ro_pulse_frequency = snail.clock_freqs.readout(), # just use resonator frequency
    ro_acquisition_delay = ACQ_DELAY,
    ro_integration_time = RO_LEN, # same as the ro_pulse_duration
    init_duration = 300e-6, # dead time long enough to reset the qubit >> T1
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=two_tone_spec_sched_nco_TWPA,
    schedule_kwargs=schedule_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(spec_gettable)

In [ ]:
control_module2.out1_att(0)
control_module2.out1_att()

In [ ]:
GS210_1.off()
# GS210_1.current(0.6585e-3)
# GS210_1.current(-0.885e-3)

In [ ]:
# weird_freq = np.array([5.75, 5.87, 6.08, 6.14, 6.19, 6.23, 6.26, 6.31, 6.40, 6.51, 8.15]) - 0.1

In [ ]:
# LO_FREQ_SNAIL_arr = weird_freq*1e9
LO_FREQ_SNAIL_arr = np.arange(7.2, 7.6, 0.2)*1e9
# LO_FREQ_SNAIL_arr = [9.6e9]

for LO_FREQ_SNAIL in LO_FREQ_SNAIL_arr:
    hardware_cfg['hardware_options']['modulation_frequencies']['snail:mw-snail.01']['lo_freq'] = LO_FREQ_SNAIL
    quantum_device.hardware_config(hardware_cfg)

    quantum_device.cfg_sched_repetitions(50)
    
    spec_freq_setpoints = np.arange(LO_FREQ_SNAIL+50e6, LO_FREQ_SNAIL+250e6, 200e3)
    meas_ctrl.settables(spec_freq)
    meas_ctrl.setpoints(spec_freq_setpoints)

    # meas_ctrl.settables([spec_freq, GS210_1.current])
    # meas_ctrl.setpoints_grid([spec_freq_setpoints, np.linspace(0.62, 0.68, 7)*1e-3])
    
    qs_ds = meas_ctrl.run("Two-tone", soft_avg=4)
    
    qs_analysis = QubitSpectroscopyAnalysis(dataset=qs_ds)
    qs_analysis.run().display_figs_mpl()

In [ ]:
LO_FREQ_SNAIL_arr = [7.2e9]

for LO_FREQ_SNAIL in LO_FREQ_SNAIL_arr:
    hardware_cfg['hardware_options']['modulation_frequencies']['snail:mw-snail.01']['lo_freq'] = LO_FREQ_SNAIL
    quantum_device.hardware_config(hardware_cfg)

    quantum_device.cfg_sched_repetitions(100)
    
    spec_freq_setpoints = np.arange(LO_FREQ_SNAIL+50e6, LO_FREQ_SNAIL+250e6, 200e3)
    meas_ctrl.settables([spec_freq, GS210_1.current])
    meas_ctrl.setpoints_grid([spec_freq_setpoints, np.linspace(-0.1e-3, 0.30e-3, 41)])

    GS210_1.on()
    qs_ds = meas_ctrl.run("Two-tone", soft_avg=5)
    GS210_1.off()
    
    qs_analysis = QubitSpectroscopyAnalysis(dataset=qs_ds)
    qs_analysis.run().display_figs_mpl()

#### 5.1.1 Current Dependent

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 800

In [ ]:
RO_AMP = 3e-3
RO_LEN = 20e-6

In [ ]:
Flux_RS_fit = np.load('Flux_RS_0811_fit.npz')
SNAIL_curr_freq = np.loadtxt('SNAIL_curr_freq.csv')

In [ ]:
ic.timeout(180)

In [ ]:
snail.clock_freqs.readout(np.nan)

In [ ]:
LO_FREQ_SNAIL = 4.8e9
hardware_cfg['hardware_options']['modulation_frequencies']['snail:mw-snail.01']['lo_freq'] = LO_FREQ_SNAIL

In [ ]:
for i, c in tqdm(enumerate(Flux_RS_fit['current'])):
    # if i > 77:
    #     break
    # elif not i > 76:
    #     continue
    if np.isnan(SNAIL_curr_freq[i]):
        continue
    LO_FREQ_SNAIL = SNAIL_curr_freq[i] - 150e6
    hardware_cfg['hardware_options']['modulation_frequencies']['snail:mw-snail.01']['lo_freq'] = LO_FREQ_SNAIL
    
    schedule_kwargs = dict(
        spec_pulse_amp = 0.5,
        spec_pulse_duration = 300e-6, # long enough to stay at the mixed state
        spec_pulse_port = "snail:mw",
        spec_pulse_clock = "snail.01",
        spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
        ro_pulse_amp = RO_AMP, # choose the power from the cavity map that at the bottom of trace f_res
        ro_pulse_duration = RO_LEN, # can be small compared to qubit decay time
        ro_pulse_delay = 200e-9,
        ro_pulse_port = "snail:res",
        ro_pulse_clock = "snail.ro",
        ro_pulse_frequency = Flux_RS_fit['fr_arr'][i]*1e9, # just use resonator frequency
        ro_acquisition_delay = ACQ_DELAY,
        ro_integration_time = RO_LEN, # same as the ro_pulse_duration
        init_duration = 200e-6, # dead time long enough to reset the qubit >> T1
    )
    spec_gettable = ScheduleGettable(
        quantum_device=quantum_device,
        schedule_function=two_tone_spec_sched_nco_TWPA,
        schedule_kwargs=schedule_kwargs,
        real_imag=True,
        batched=True,
    )
    
    meas_ctrl.gettables(spec_gettable)

    quantum_device.cfg_sched_repetitions(200)
    
    spec_freq_setpoints = np.arange(LO_FREQ_SNAIL+100e6, LO_FREQ_SNAIL+200e6, 125e3)
    meas_ctrl.settables(spec_freq)
    meas_ctrl.setpoints(spec_freq_setpoints)
    meas_ctrl.verbose(False)
    
    GS210_1.on()
    GS210_1.current(c/1e3)
    qs_ds = meas_ctrl.run("spectroscopy_current", soft_avg=5)
    GS210_1.off()

#### 5.1.0 SNAIL Rabi

In [ ]:
GS210_1.off()
# GS210_1.current(0.6585e-3)
GS210_1.current(-0.885e-3)

In [ ]:
rabi_pulse_amp = ManualParameter(name="rabi_pulse_amplitude", unit="V", label="Rabi Pulse Amplitude")
rabi_pulse_amp.batched = True
rabi_pulse_amp.batch_size = 200

rabi_pulse_len = ManualParameter(name="rabi_pulse_length", unit="s", label="Rabi Pulse Length")
rabi_pulse_len.batched = True
rabi_pulse_len.batch_size = 10

rabi_sched_kwargs = {
    "pulse_amp": rabi_pulse_amp,
    "pulse_duration": 200e-9,
    "frequency": 3615424279.1499014,
    "qubit": qubit2,
    'port': "snail:mw",
    'clock': "snail.01",
}

rabi_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=rabi_sched_TWPA,
    schedule_kwargs=rabi_sched_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(rabi_gettable)

In [ ]:
control_module2.out1_att(30)
control_module2.out1_att()

In [ ]:
quantum_device.cfg_sched_repetitions(50)

rabi_amplitude_setpoints = np.linspace(-1, 1, 200)

meas_ctrl.settables(rabi_pulse_amp)
meas_ctrl.setpoints(rabi_amplitude_setpoints)

# rabi_length_setpoints = np.arange(4e-9, 4000e-9, 40e-9)

# meas_ctrl.settables(rabi_pulse_len)
# meas_ctrl.setpoints(rabi_length_setpoints)

SNL_rabi_ds = meas_ctrl.run("SNL_Rabi", soft_avg=10)
SNL_rabi_ds

In [ ]:
SNL_rabi_analysis = RabiAnalysis(dataset=SNL_rabi_ds)
SNL_rabi_analysis.run().display_figs_mpl()

In [ ]:
spec_amp = ManualParameter(name="spec_amp", unit="V", label="Spec_Amplitude")
spec_amp.batched = True
spec_amp.batch_size = 200

In [ ]:
RO_AMP = 1.2e-3
RO_LEN = 20e-6

In [ ]:
schedule_kwargs = dict(
    spec_pulse_amp = spec_amp,
    spec_pulse_duration = 200e-9, # Gate time
    spec_pulse_port = "snail:mw",
    spec_pulse_clock = "snail.01",
    spec_pulse_frequencies = np.array([3615424279.1499014]), # SNAIL f01
    ro_pulse_amp = RO_AMP,
    ro_pulse_duration = RO_LEN,
    ro_pulse_delay = 200e-9,
    ro_pulse_port = "qubit2:res",
    ro_pulse_clock = "qubit2.ro",
    ro_pulse_frequency = qubit2.clock_freqs.readout(), # just use resonator frequency
    ro_acquisition_delay = ACQ_DELAY,
    ro_integration_time = RO_LEN, # same as the ro_pulse_duration
    init_duration = 500e-6, # dead time long enough to reset the qubit >> T1
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=two_tone_spec_sched_nco_TWPA,
    schedule_kwargs=schedule_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(spec_gettable)

In [ ]:
control_module2.out1_att(30)
control_module2.out1_att()

In [ ]:
quantum_device.cfg_sched_repetitions(50)
    
spec_amp_setpoints = np.linspace(-1, 1, 200)
meas_ctrl.settables([spec_amp])
meas_ctrl.setpoints_grid([spec_amp_setpoints])

GS210_1.on()
snl_rabi_ds = meas_ctrl.run("SNAIL Rabi", soft_avg=1)
GS210_1.off()

#### 5.1.1 SGS Spectroscopy

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = True
spec_freq.batch_size = 1

# spec_amp = ManualParameter(name="spec_amp", unit="V", label="Spec_Amplitude")
# spec_amp.batched = False

In [ ]:
schedule_kwargs = dict(
    spec_pulse_amp = -40,
    spec_pulse_duration = 300e-6, # long enough to stay at the mixed state
    spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
    snail_drive = SGS100A_1,
    qubit = qubit2, # readout via qubit's cavity
    init_duration = 500e-6, # dead time long enough to reset the qubit >> T1
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=SNAIL_spec_sched_TWPA,
    schedule_kwargs=schedule_kwargs,
    real_imag=True,
    batched=True,
)

meas_ctrl.gettables(spec_gettable)

In [ ]:
# GS210_1.off()
# GS210_1.current(0.6585e-3)
# GS210_1.current(0.0e-3)

In [ ]:
quantum_device.cfg_sched_repetitions(100)

spec_freq_setpoints = np.arange(3.6e9, 3.7e9, 200e3)
meas_ctrl.settables(spec_freq)
meas_ctrl.setpoints(spec_freq_setpoints)

meas_ctrl.settables([spec_freq, GS210_1.current])
meas_ctrl.setpoints_grid([spec_freq_setpoints, np.flip(np.linspace(0.85, 0.92, 6)*1e-3)])

GS210_1.on()
qs_ds = meas_ctrl.run("SGS_spectroscopy", soft_avg=5)
GS210_1.off()

#### 5.1.2 SGS Spectroscopy VNA

In [ ]:
# Manual parameter for batched schedule
spec_freq = ManualParameter(name="spec_freq", unit="Hz", label="Spec_Frequency")
spec_freq.batched = False
spec_freq.batch_size = 1

# spec_amp = ManualParameter(name="spec_amp", unit="V", label="Spec_Amplitude")
# spec_amp.batched = False

In [ ]:
schedule_kwargs = dict(
    spec_pulse_amp = -10,
    spec_pulse_duration = 300e-6, # long enough to stay at the mixed state
    spec_pulse_frequencies = spec_freq, # find f01, f02/2 etc.
    snail_drive = SGS100A_1,
    ro_vna = E5081A_1,
    qubit = qubit1,
    init_duration = 500e-6, # dead time long enough to reset the qubit >> T1
)
spec_gettable = ScheduleGettable(
    quantum_device=quantum_device,
    schedule_function=SNAIL_spec_sched_VNA,
    schedule_kwargs=schedule_kwargs,
    real_imag=False,
    batched=False,
)

meas_ctrl.gettables((spec_gettable, E5081A_1.linear_magnitude, E5081A_1.phase))

In [ ]:
GS210_1.off()
# GS210_1.current(0.6585e-3)
GS210_1.current(0.0e-3)

In [ ]:
plotmon.tuids([])

In [ ]:
E5081A_1.if_bandwidth(30)
E5081A_1.averages_enabled(False)

In [ ]:
quantum_device.cfg_sched_repetitions(1)

spec_freq_setpoints = np.arange(9.67e9, 9.69e9, 100e3)

meas_ctrl.settables(spec_freq)
meas_ctrl.setpoints(spec_freq_setpoints)

# meas_ctrl.settables([spec_freq, GS210_1.current])
# meas_ctrl.setpoints_grid([spec_freq_setpoints, np.flip(np.linspace(0.51, 0.61, 6)*1e-3)])

GS210_1.on()
qs_ds = meas_ctrl.run("SGS_VNA_spectroscopy", soft_avg=10)
GS210_1.off()

### 5.2.0 iSWAP Delay

In [ ]:
# comp_sched = device_compiler.compile(iswap_delay_schedule(qubit_specifier=[qubit1, qubit2], gate_idx=3, delays=[-500e-9]))
# comp_sched.plot_pulse_diagram(plot_backend="plotly")

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import iswap_delay_schedule

In [ ]:
delay = ManualParameter(name="swap_delay", unit="s", label="Swap Delay")
delay.batched = True
delay.batch_size = 500

delay_sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "gate_idx": 3,
    "delays": delay,
    "acq_protocol": "ThresholdedAcquisition",
}

swap_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=iswap_delay_schedule,
    schedule_kwargs=delay_sched_kwargs,
    real_imag=True,
    batched=True,
    num_channels=2,
)

meas_ctrl.gettables(swap_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(500)

delay_setpoints = np.arange(-0.04e-6, 0.04e-6, 1e-9)

meas_ctrl.settables(delay)
meas_ctrl.setpoints(delay_setpoints)

delay_ds = meas_ctrl.run("iSWAP Delay", soft_avg=1)
delay_ds.tuid

### 5.2 iSWAP Continuous Pump

In [ ]:
# pump_freq = qubit2.clock_freqs.f01() - qubit1.clock_freqs.f01()
# pump_freq = 258341806 # 6 dBm
# pump_freq = 258562449 # 8 dBm
pump_freq = 258671197 #258718455 # 9 dBm
# pump_freq = 259462633 # 12 dBm

In [ ]:
hardware_cfg['hardware_options']['modulation_frequencies']['qubit2:mw-qubit2.01']['lo_freq'] = LO_FREQ_QUBIT1 + pump_freq
hardware_cfg['hardware_options']['modulation_frequencies']['qubit2:fs-qubit2.12']['lo_freq'] = LO_FREQ_QUBIT1 + pump_freq

quantum_device.hardware_config(hardware_cfg)

In [ ]:
qubit1.reset.duration(250e-6)
qubit2.reset.duration(250e-6)

In [ ]:
qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

swap_pulse_len = ManualParameter(name="swap_pulse_length", unit="s", label="Swap_Pulse_Length")
swap_pulse_len.batched = True
swap_pulse_len.batch_size = 1001

swap_pulse_power = ManualParameter(name="swap_pulse_power", unit="dBm", label="Swap_Pulse_Power")
swap_pulse_power.batched = False

swap_pulse_freq = ManualParameter(name="spec_freq", unit="Hz", label="Swap_Pulse_Frequency")
swap_pulse_freq.batched = False

swap_sched_kwargs = {
    "pulse_frequency": pump_freq,
    "pulse_amp": 9,
    "pulse_duration": swap_pulse_len,
    "snail_drive": SGS100A_1,
    "qubit_specifier": [qubit1, qubit2],
    "qubit_e": [qubit2],
    "swap_type": "iSWAP",
    "acq_protocol": "ThresholdedAcquisition",
}

swap_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=SNAIL_swap_sched_TWPA,
    schedule_kwargs=swap_sched_kwargs,
    real_imag=True,
    batched=True,
    num_channels=2,
)

meas_ctrl.gettables(swap_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)

# swap_amplitude_setpoints = np.linspace(-10,20,31)

# meas_ctrl.settables(swap_pulse_amp)
# meas_ctrl.setpoints(swap_amplitude_setpoints)

# swap_duration_setpoints = np.arange(0e-9, 40e-6, 80e-9)
# swap_duration_setpoints = np.arange(0e-9, 60e-6, 400e-9)
swap_duration_setpoints = np.arange(0e-9, 4e-6, 40e-9)
# swap_duration_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 120, 2))) * 1e-6

meas_ctrl.settables(swap_pulse_len)
meas_ctrl.setpoints(swap_duration_setpoints)

# swap_frequency_setpoints = np.linspace(pump_freq-2e6, pump_freq+2e6, 21)
# meas_ctrl.settables([swap_pulse_len, swap_pulse_freq])
# meas_ctrl.setpoints_grid([swap_duration_setpoints, swap_frequency_setpoints])

# meas_ctrl.settables([swap_pulse_len, swap_pulse_power])
# meas_ctrl.setpoints_grid([swap_duration_setpoints, np.tile(np.arange(0, 20), 20)])

# swap_frequency_setpoints = np.linspace(pump_freq-2e6, pump_freq+1e6, 21)
# meas_ctrl.settables([swap_pulse_len, swap_pulse_freq, swap_pulse_power])
# meas_ctrl.setpoints_grid([swap_duration_setpoints, swap_frequency_setpoints, W2dBm(np.linspace(0.001, 0.03, 21))[:7]])

swap_ds = meas_ctrl.run("SNAIL "+swap_sched_kwargs['swap_type'], soft_avg=1)
swap_ds.tuid

In [ ]:
iswap_chevron_fit(swap_ds.tuid, plot_fit=True);

In [ ]:
# 20260930-165851
# 20260930-165916

In [ ]:
# quantum_device.cfg_sched_repetitions(100)
# swap_duration_setpoints = np.arange(0e-9, 40e-6, 400e-9)
# test_tuid_list = []

# for dur in range(50,1050,50):
#     qubit2.reset.duration(dur*1e-6)

#     meas_ctrl.settables(swap_pulse_len)
#     meas_ctrl.setpoints(swap_duration_setpoints)

#     swap_ds = meas_ctrl.run("SNAIL "+swap_sched_kwargs['swap_type'], soft_avg=1)
#     test_tuid_list.append(swap_ds.tuid)

In [ ]:
# 20260924-204141

#### 5.2.1 Current Sweep

In [ ]:
tuid_list = []
meas_ctrl.verbose(False)
GS210_1.on()

if_plot = False
for c in tqdm(np.linspace(0.67, 0.73, 21)/1e3):
    GS210_1.current(c)
    for qubit in [qubit1, qubit2]:
        nodes.resonator_calibration(qubit)
        nodes.ramsey_chevron_calibration(qubit, plot_data=if_plot)
        nodes.rabi_calibration(qubit)
        # nodes.rabi_amplification_calibration(qubit, plot=if_plot)
        # nodes.readout_calibration(qubit, plot=if_plot)
        # nodes.T1_calibration(qubit)
        # nodes.T2_echo_calibration(qubit)
    tuid_list.append(plotmon.tuids()[-1])

meas_ctrl.verbose(True)

In [ ]:
tuid_list = ['20260915-194755',
 '20260915-194917',
 '20260915-195041',
 '20260915-195205',
 '20260915-195329',
 '20260915-195453',
 '20260915-195617',
 '20260915-195741',
 '20260915-195905',
 '20260915-200029',
 '20260915-200153',
 '20260915-200316',
 '20260915-200440',
 '20260915-200605',
 '20260915-200728',
 '20260915-200853',
 '20260915-201016',
 '20260915-201140',
 '20260915-201304',
 '20260915-201428',
 '20260915-201552']

In [ ]:
pump_curr_list = np.linspace(0.67, 0.73, 21)/1e3

meas_ctrl.verbose(False)

for idx in tqdm(range(len(pump_curr_list))):
    GS210_1.current(pump_curr_list[idx])
    set_from_snapshot(tuid_list[idx])

    nodes.multiplexed_readout_calibration([qubit1, qubit2], num_points=3000);

    pump_freq = qubit2.clock_freqs.f01() - qubit1.clock_freqs.f01() + 0.6e6

    swap_sched_kwargs["pulse_frequency"] = swap_pulse_freq
    swap_sched_kwargs["pulse_amp"] = 9

    swap_gettable = ScheduleGettable(
        quantum_device,
        schedule_function=SNAIL_swap_sched_TWPA,
        schedule_kwargs=swap_sched_kwargs,
        real_imag=True,
        batched=True,
        num_channels=2,
    )

    meas_ctrl.gettables(swap_gettable)
    quantum_device.cfg_sched_repetitions(100)

    swap_duration_setpoints = np.arange(0e-9, 4e-6, 60e-9)
    swap_frequency_setpoints = np.linspace(pump_freq-1e6, pump_freq+1e6, 21)
    meas_ctrl.settables([swap_pulse_len, swap_pulse_freq])
    meas_ctrl.setpoints_grid([swap_duration_setpoints, swap_frequency_setpoints])
    swap_ds = meas_ctrl.run("SNAIL "+swap_sched_kwargs['swap_type'], soft_avg=1)
    print(swap_ds.tuid)

In [ ]:
# pump_curr_list = np.linspace(0.67, 0.73, 7)/1e3
# pump_freq_list = [259406699, 259304644, 259185287, 259027813, 258831216, 258594181, 258408512]
# pump_snapshot_list = ['20260810-151210-300-9766c5',
#     '20260810-151414-800-bb5c4d',
#     '20260810-151619-195-cf67fa',
#     '20260810-151823-625-6278bb',
#     '20260810-152028-231-063e3a',
#     '20260810-152232-452-9fde2b',
#     '20260810-152437-052-893e80']

# pump_snapshot_list = ['20260915-134127', '20260915-134235','20260915-134344','20260915-134452','20260915-134600',
# '20260915-134708','20260915-134816','20260915-134924','20260915-135032','20260915-135140','20260915-135248',
# '20260915-135356','20260915-135505','20260915-135614','20260915-135722','20260915-135830','20260915-135938',
# '20260915-140051','20260915-140204','20260915-140317', '20260915-140432']

pump_snapshot_list = ['20260915-202038', '20260915-202146','20260915-202254','20260915-202402','20260915-202510',
'20260915-202618','20260915-202726','20260915-202834','20260915-202942','20260915-203050','20260915-203158',
'20260915-203306','20260915-203414','20260915-203522','20260915-203630','20260915-203739','20260915-203846',
'20260915-203954','20260915-204102','20260915-204210', '20260915-204318']

# pump_snapshot_list = ['20260915-175737', '20260915-175846','20260915-175954','20260915-180104','20260915-180213',
# '20260915-180321','20260915-180430','20260915-180538','20260915-180647','20260915-180755','20260915-180904',
# '20260915-181012','20260915-181121','20260915-181229','20260915-181338','20260915-181446','20260915-181554',
# '20260915-181702','20260915-181810','20260915-181918', '20260915-182030']

pump_freq_list = [iswap_chevron_fit(pump_snapshot_list[i]) for i in range(len(pump_snapshot_list))]
pump_curr_list = [dataset_current(pump_snapshot_list[i]) for i in range(len(pump_snapshot_list))]

In [ ]:
swap_list_tuid = []

for _ in tqdm(range(5)):
    for idx, pump_freq in enumerate(pump_freq_list):
        GS210_1.current(pump_curr_list[idx] / 1e3)
        set_from_snapshot(pump_snapshot_list[idx])
        nodes.multiplexed_readout_calibration([qubit1, qubit2], num_points=3000)
        
        swap_sched_kwargs["pulse_frequency"] = pump_freq
        swap_sched_kwargs["pulse_amp"] = 9

        swap_duration_setpoints = np.arange(0, 30e-6, 100e-9)

        swap_gettable = ScheduleGettable(
            quantum_device,
            schedule_function=SNAIL_swap_sched_TWPA,
            schedule_kwargs=swap_sched_kwargs,
            real_imag=True,
            batched=True,
            num_channels=2,
            max_batch_size=swap_duration_setpoints.size
        )

        meas_ctrl.gettables(swap_gettable)
        meas_ctrl.verbose(False)

        quantum_device.cfg_sched_repetitions(100)

        meas_ctrl.settables([swap_pulse_len, SGS100A_1.phase])
        meas_ctrl.setpoints_grid([swap_duration_setpoints, np.zeros(10)])
        swap_ds = meas_ctrl.run("iSWAP Current Sweep", soft_avg=1)
        swap_list_tuid.append(swap_ds.tuid)

In [ ]:
# np.savetxt(get_datadir() + '/iSWAP_pump_curr_9dBm_4.txt', swap_list_tuid, fmt='%s')

#### 5.2.2 Power Sweep

In [ ]:
pump_power_list = W2dBm(np.linspace(0.001, 0.03, 21))
pump_freq_list = np.array([2.57974842e+08, 2.58084210e+08, 2.58246706e+08, 2.58390314e+08,
       2.58486658e+08, 2.58622774e+08, 2.58777517e+08, 2.58939475e+08,
       2.59051136e+08, 2.59155342e+08, 2.59272327e+08, 2.59392023e+08,
       2.59524625e+08, 2.59574192e+08, 2.59752274e+08, 2.59870537e+08,
       2.59959175e+08, 2.60126662e+08, 2.60248960e+08, 2.60344522e+08,
       2.60473845e+08])

t_swap_list = np.array([1430.89534908, 899.04974534, 714.4102194, 604.20519246,
       533.35797234, 481.44829084, 442.88699757, 415.38642205,
       387.11145324, 364.26370528, 344.07038045, 326.32759552,
       313.8624228, 297.52544188, 285.64785854, 275.9350227,
       262.39357591, 255.54257086, 247.16073838, 239.15164503,
       231.68860879])/1e9

In [ ]:
pump_power_list = Vp2dBm(np.linspace(dBm2Vp(-10), dBm2Vp(20), 41))

pump_freq_list = np.array([257.88335217, 257.90480367, 257.94369833, 257.991501  ,
       258.03655533, 258.09602733, 258.1756835 , 258.255225  ,
       258.36727517, 258.47074817, 258.57771217, 258.69827117,
       258.83780217, 258.9888795 , 259.1474075 , 259.30971983,
       259.44610983, 259.64961983, 259.8075365 , 260.012115  ,
       260.18938   , 260.39901583, 260.586329  , 260.85980233,
       261.04176817, 261.30920633, 261.547754  , 261.71493133,
       262.00186167, 262.16416783, 262.41684417, 262.68073917,
       262.92895383, 263.07489133, 263.33942167, 263.5312455 ,
       263.88962067, 264.14802033, 264.40573067, 264.6878145 ,
       264.909075  ]) * 1e6

t_swap_list = np.array([4426.97658279, 2563.58652216, 1708.96371655, 1344.22095463,
       1104.5731608 ,  920.92825288,  797.94072496,  696.70920826,
        620.77711872,  556.913166  ,  505.9761251 ,  465.41804944,
        429.05511363,  399.02049024,  370.45010611,  342.4734359 ,
        322.05460042,  305.38546271,  284.57422493,  267.02051405,
        252.45484819,  237.55218837,  225.0971286 ,  213.52435847,
        200.80296006,  192.06816168,  183.8921971 ,  171.89646769,
        164.31117555,  155.76140157,  146.72249793,  138.68205754,
        130.80579647,  124.3922651 ,  118.630163  ,  111.57194808,
        106.51149304,   99.84176219,   96.1109823 ,   90.1840226 ,
         85.86851419])  / 1e9

In [ ]:
from qcodes.instrument.parameter import ManualParameter

pump_freq = ManualParameter("pump_freq", unit="Hz",  initial_value=pump_freq_list[0])
pump_amp  = ManualParameter("pump_amp",  unit="dBm", initial_value=pump_power_list[0])

swap_sched_kwargs["pulse_frequency"] = pump_freq
swap_sched_kwargs["pulse_amp"]       = pump_amp

N_FREQ, N_TIME = 21, 24


def chevron_axes(t_swap, f_center, n_freq=N_FREQ, n_time=N_TIME,
                 halfwidths=2.0, hw_grid=4e-9):
    """Frequency detunings and swap durations for the chevron at one pump power.

    The chevron half-width is 1/(2 t_swap): scanning +/- `halfwidths` of them keeps
    the central lobe and the first sidelobes in view at every power, rather than
    over-scanning the slow settings and clipping the fast ones.

    The duration axis covers 4 t_swap, two full oscillations on resonance and more
    off it, snapped to the instrument's 4 ns grid.
    """
    span_f = halfwidths / (2 * t_swap)
    freqs = f_center + np.linspace(-span_f, span_f, n_freq)
    dt = max(hw_grid, round(4 * t_swap / n_time / hw_grid) * hw_grid)
    return freqs, dt * np.arange(n_time)


cols_t, cols_f, cols_a = [], [], []
for t_swap, f0, amp in zip(t_swap_list, pump_freq_list, pump_power_list):
    freqs, durations = chevron_axes(t_swap, f0)
    for f in freqs:
        cols_t.append(durations)
        cols_f.append(np.full(durations.size, f))
        cols_a.append(np.full(durations.size, amp))

setpoints = np.column_stack([np.concatenate(cols_t),
                             np.concatenate(cols_f),
                             np.concatenate(cols_a)])
print("%d blocks, %d points, generator span %.1f - %.1f MHz"
      % (len(cols_t), setpoints.shape[0],
         setpoints[:, 1].min() / 1e6, setpoints[:, 1].max() / 1e6))

swap_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=SNAIL_swap_sched_TWPA,
    schedule_kwargs=swap_sched_kwargs,
    real_imag=True,
    batched=True,
    num_channels=2,
)

def arm_chevron():
    """Point meas_ctrl back at the chevron sweep.

    multiplexed_readout_calibration takes meas_ctrl over for its own IQ-blob run
    and does not put it back - it also drops cfg_sched_repetitions to 1 - so this
    has to run after it, not once before the loop.
    """
    meas_ctrl.gettables(swap_gettable)
    meas_ctrl.settables([swap_pulse_len, pump_freq, pump_amp])
    meas_ctrl.setpoints(setpoints)
    meas_ctrl.verbose(True)
    quantum_device.cfg_sched_repetitions(100)


# nodes.multiplexed_readout_calibration([qubit1, qubit2], num_points=3000)
arm_chevron()
# distinct label: "SNAIL iSWAP" is what the duration sweep already writes
chevron_ds = meas_ctrl.run("SNAIL " + swap_sched_kwargs["swap_type"] + " chevron",
                            soft_avg=1)
chevron_ds.tuid


In [ ]:
plt.plot(offset_tau_meas)

In [ ]:
# offset_tau_meas = np.array([42.25139594, 39.57858008, 40.36319469, 38.41236412, 36.00384732,
#     31.96155634, 31.97937235, 29.74568438, 28.8747281 , 30.77325662,
#     27.8081296 , 24.78659911, 25.05219324, 23.05679644, 20.66163988,
#     20.90087177, 20.06859396, 18.0249091 , 16.79605006, 14.27143523,
#     11.41285459,  9.67640478, 10.31922145,  9.80026793,  9.16347381,
#     7.60010272,  7.00280642,  6.03278043,  5.73077545,  5.30414898,
#     4.92462056,  4.34158723,  4.1565757 ,  4.10699145,  3.35002373,
#     4.1126777 ,  4.08898769,  3.95698138,  3.74896049,  2.31536248,
#     3.78678455]) * 1e-6

offset_tau_meas = np.array([45.60815651, 43.355645  , 42.46154371, 41.24001934, 39.18413761,
       32.25125881, 28.57173683, 27.85682823, 26.22034868, 25.10047328,
       24.33450008, 23.27530884, 22.18112514, 20.58321298, 18.96154307,
       18.26429523, 17.3286408 , 15.67362094, 15.08382825, 13.74627368,
       11.6905067 , 10.34747692,  9.79454816,  9.15070634,  8.59938026,
        7.65790496,  7.12020193,  6.13125014,  5.99481412,  5.48051531,
        5.12179521,  4.78700011,  4.41527686,  4.2862574 ,  4.43514487,
        4.43212164,  4.48482822,  4.68215289,  4.32470853,  4.25708562,
        4.05331022]) * 1e-6

def duration_grid(t_swap, offset_tau, n_points=100, envelopes=50,
                  span_min=20e-6, span_max=120e-6, hw_grid=4e-9):
    """Fixed point count, span set by the envelope rather than by the clock.

    2 x offset_tau sits in the middle of the plateau where both decay times fit
    best, clamped to [span_min, span_max]. The guard stops a long span from
    thinning the sampling towards Nyquist, where the fit aliases and returns a
    confidently wrong t_swap.
    """
    span = min(max(envelopes * offset_tau, span_min), span_max)
    dt = span / n_points
    # if 2 * t_swap / dt < 4:
    #     dt = t_swap / 2
    dt = max(hw_grid, round(dt / hw_grid) * hw_grid)
    return dt * np.arange(n_points)


blocks = [duration_grid(ts, ot) for ts, ot in zip(t_swap_list, offset_tau_meas)]


In [ ]:
from qcodes.instrument.parameter import ManualParameter

pump_freq = ManualParameter("pump_freq", unit="Hz",   initial_value=pump_freq_list[0])
pump_amp  = ManualParameter("pump_amp",  unit="dBm",  initial_value=pump_power_list[0])

swap_sched_kwargs["pulse_frequency"] = pump_freq
swap_sched_kwargs["pulse_amp"]       = pump_amp

sizes  = [b.size for b in blocks]
setpoints = np.column_stack([
    np.concatenate(blocks),
    np.repeat(pump_freq_list,  sizes),
    np.repeat(pump_power_list, sizes),
])

swap_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=SNAIL_swap_sched_TWPA,
    schedule_kwargs=swap_sched_kwargs,
    real_imag=True,
    batched=True,
    num_channels=2,
)


def arm_swap():
    """Point meas_ctrl back at the swap sweep.

    multiplexed_readout_calibration takes meas_ctrl over for its own IQ-blob run
    and does not put it back - it also drops cfg_sched_repetitions to 1 - so this
    has to run after it, not once before the loop.
    """
    meas_ctrl.gettables(swap_gettable)
    meas_ctrl.settables([swap_pulse_len, pump_freq, pump_amp])
    meas_ctrl.setpoints(setpoints)
    meas_ctrl.verbose(False)
    quantum_device.cfg_sched_repetitions(100)


swap_list_tuid = []
for rep in tqdm(range(3)):
    # if rep % 5 == 0:
    #     nodes.multiplexed_readout_calibration([qubit1, qubit2], num_points=3000)
    #     nodes.multi_qubit_t1_and_t2([qubit1, qubit2], num_repeats=5)
    arm_swap()
    swap_ds = meas_ctrl.run("SNAIL " + swap_sched_kwargs["swap_type"], soft_avg=1)
    swap_list_tuid.append(swap_ds.tuid)
    time.sleep(5)

In [ ]:
[tuid[:15] for tuid in swap_list_tuid]

In [ ]:
# np.savetxt(get_datadir() + '/iSWAP_power_sweep_0929.txt', [tuid[:15] for tuid in swap_list_tuid], fmt='%s')

### 5.3 iSWAP Pulsed Pump

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of iSWAP gates")
length.batched = True
length_setpoints = np.arange(60)
# length_setpoints = np.arange(0, 80, 2)

sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    "acq_protocol": "ThresholdedAcquisition",
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=iswap_pulsed_pump_schedule,
    schedule_kwargs=sched_kwargs,
    real_imag=False,
    batched=True,
    num_channels=2,
    max_batch_size=length_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

# meas_ctrl.settables(length)
# meas_ctrl.setpoints(length_setpoints)

# meas_ctrl.settables([length, wait_time])
# meas_ctrl.setpoints_grid([length_setpoints, wait_time_setpoints])

meas_ctrl.settables([length, qubit1_qubit2.iswap.pulse_duration])
meas_ctrl.setpoints_grid([length_setpoints, np.arange(486, 502, 1)/1e9])

quantum_device.cfg_sched_repetitions(200)

In [ ]:
iswap_pulsed_ds = meas_ctrl.run("iSWAP Pulsed Pump", soft_avg=10)
display(iswap_pulsed_ds.tuid)
# 20260630-174725

In [ ]:
exec(open(r"C:/Users/1238Tech/AppData/Local/Temp/claude/c--Users-1238Tech-Documents-QBLOX-SNL315/c65c1918-5833-41ed-ba5d-b9cf3c90aaa7/scratchpad/night/fan_manual.py").read())

In [ ]:
theta_p_res = iswap_theta_p_fit(iswap_pulsed_ds, plot=True)
if theta_p_res is not None:
    print(f"pi/2 at {theta_p_res['pulse_duration']*1e9:.2f} ns "
          f"-> use {theta_p_res['pulse_duration_rounded']*1e9:.0f} ns")

In [ ]:
qubit1_qubit2.iswap.pulse_duration(theta_p_res['pulse_duration_rounded'])

### 5.4 Pumped T1 and T2

In [ ]:
pump_freq = 250*1e6

In [ ]:
(qubit2.clock_freqs.f01() - qubit1.clock_freqs.f01()) / 1e6

In [ ]:
SGS100A_1.power(20)
SGS100A_1.frequency(qubit2.clock_freqs.f01() - qubit1.clock_freqs.f01())

In [ ]:
qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True
tau.batch_size = 1000

pump_sched_kwargs = {
    "times": tau,
    "snail_drive": SGS100A_1,
    "qubit_specifier": [qubit1, qubit2],
    "qubit_e": [],
    "acq_protocol": "SSBIntegrationComplex",
    # "artificial_detuning": -22e6,  
}

swap_gettable = ScheduleGettable(
    quantum_device,
    schedule_function=pump_t1_sched_TWPA,
    schedule_kwargs=pump_sched_kwargs,
    real_imag=True,
    batched=True,
    num_channels=2,
)

meas_ctrl.gettables(swap_gettable)

In [ ]:
quantum_device.cfg_sched_repetitions(100)

tau_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 120, 2))) * 1e-6
# tau_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 200, 4))) * 1e-6
# tau_setpoints = np.arange(0, 50, 1) * 1e-6


# meas_ctrl.settables(tau)
# meas_ctrl.setpoints(tau_setpoints)

# meas_ctrl.settables([tau, SGS100A_1.power])
# meas_ctrl.setpoints_grid([tau_setpoints, [20]])
# meas_ctrl.setpoints_grid([tau_setpoints, np.tile(np.linspace(0, 20, 11), 20)])

# meas_ctrl.settables([tau, SGS100A_1.frequency])
# meas_ctrl.setpoints_grid([tau_setpoints, np.linspace(pump_freq-200e6, pump_freq+200e6, 41)])

meas_ctrl.settables([tau, SGS100A_1.frequency, SGS100A_1.power])
meas_ctrl.setpoints_grid([tau_setpoints, np.linspace(pump_freq-200e6, pump_freq+200e6, 41), 
            np.tile(np.linspace(19, 25, 4), 1)])

pump_ds = meas_ctrl.run("Pump T1 2D Sweep", soft_avg=1)
pump_ds.tuid

In [ ]:
power_W_list = np.array([0.0001  , 0.002095, 0.00409 , 0.006085, 0.00808 , 0.010075, 0.01207, 
0.014065, 0.01606 , 0.018055, 0.02005 , 0.022045, 0.02404 , 0.026035])
# power_f01_list = np.array([-0.09016202, -1.65880092, -3.21807606, -4.78091852, -6.39744012, -7.96589391, -9.56862845, 
# -11.20749815, -12.80148431, -14.38985719, -16.08918663, -17.52652522, -19.12515489, -20.82013286]) * 1e6

# power_W_list = np.linspace(0.0001, 0.04, 21)
# power_f01_list = np.array([-0.07072384, -1.44901521, -2.81271103, -4.18306878, -5.59094309, -6.95998653, -8.34900844, 
# -9.80998172, -11.19385115, -12.57336101, -14.03790444, -15.28896048, -16.67040523, -18.11959909])

In [ ]:
power_W_list = np.linspace(0.0001, 0.04, 21)

power_f01_list = np.arange(21) * -1.3883750192307693 + -0.07072384
power_f01_list = power_f01_list * 1e6

In [ ]:
power_W_list = dBm2W(np.linspace(-10, 15, 21))
power_f01_list = -694.493608383841 * power_W_list * 1e6

In [ ]:
power_W_list = dBm2W(np.linspace(-10, 20, 25))

In [ ]:
SGS100A_1.frequency(qubit2.clock_freqs.f01() - qubit1.clock_freqs.f01())

In [ ]:
qubit2.clock_freqs.f01() / 250e6

In [ ]:
tau_setpoints = np.arange(0.1, 20, 0.08) * 1e-6
pump_tuid_list = []

quantum_device.cfg_sched_repetitions(200)
meas_ctrl.verbose(False)

for _ in range(20):
    for i, w in enumerate(tqdm(power_W_list)):
        if i < 10:
            tau_setpoints = np.hstack((np.arange(0, 40, 0.4), np.arange(40, 200, 4))) * 1e-6
            # tau_setpoints = np.arange(0.1, 60, 0.2) * 1e-6
            # pump_sched_kwargs['artificial_detuning'] = power_f01_list[i] + 0.4e6
        elif i < 18 and i >= 10:
            tau_setpoints = np.hstack((np.arange(0, 20, 0.2), np.arange(20, 100, 2))) * 1e-6
            # tau_setpoints = np.arange(0.1, 24, 0.08) * 1e-6
            # pump_sched_kwargs['artificial_detuning'] = power_f01_list[i] + 0.6e6
        elif i >= 18:
            tau_setpoints = np.hstack((np.arange(0, 10, 0.1), np.arange(10, 50, 1))) * 1e-6
            # tau_setpoints = np.arange(0.1, 16, 0.06) * 1e-6
            # pump_sched_kwargs['artificial_detuning'] = power_f01_list[i] + 0.8e6

        SGS100A_1.power(W2dBm(w))
        meas_ctrl.gettables(swap_gettable)

        meas_ctrl.settables(tau)
        meas_ctrl.setpoints(tau_setpoints)

        swap_ds = meas_ctrl.run("Pump T1", soft_avg=1)
        pump_tuid_list.append(swap_ds.tuid)

meas_ctrl.verbose(True)

In [ ]:
len(pump_tuid_list)

In [ ]:
# np.savetxt(get_datadir() + '/pump_T1_Q2_0805.txt', pump_tuid_list, fmt='%s')

# 6. Coupled Qubits

In [ ]:
roadmap = [
    ('Resonator',         'resonator_calibration',           {}),
    ('Ramsey Chevron',    'ramsey_chevron_calibration',      {}),
    ('Rabi',              'rabi_calibration',                {}),
    ('Rabi Amplification','rabi_amplification_calibration',  {}),
    ('DRAG',              'drag_calibration',                {}),
    ('ALLXY',             'allxy_calibration',               {}),
    ('RO Freq Opt',       'ro_freq_optimization',            {}),
    ('Readout',           'readout_calibration',             {}),
    ('T1',                'T1_calibration',                  {}),
    ('T2 Echo',           'T2_echo_calibration',             {}),
]

log = nodes.run_roadmap(roadmap, [qubit1, qubit2], stop_on_fail=False)

In [ ]:
qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

In [ ]:
# if_plot = True
# ro_roadmap = [
#     ('RO Frequency Optimization', 'ro_freq_optimization', {'plot': if_plot}),
#     ('RO Amplitude Optimization', 'ro_amp_optimization',   {'amp_setpoints': np.linspace(0.15, 0.25, 41), 'plot': if_plot}),
#     ('RO Duration Optimization',  'ro_len_optimization',   {'len_setpoints': np.arange(2.0, 6.1, 0.1)/1e6, 'plot': if_plot}),
# ]

# n_loops = 2
# all_logs = []
# for i in range(n_loops):
#     print(f'\n########## Loop {i+1}/{n_loops} ##########')
#     log = nodes.run_roadmap(ro_roadmap, [qubit2])
#     all_logs.append(log)

### 6.0 Multiplexed RO Calibration

In [ ]:
# nodes.readout_calibration(qubit1, plot=1)
# nodes.readout_calibration(qubit2, plot=1)

In [ ]:
# 0.215
# 0.2375
qubit1.measure.pulse_amp(0.215-0.00)
qubit2.measure.pulse_amp(0.2375-0.03)

# 5.7
ro_len_opt = 3.6e-6
qubit1.measure.pulse_duration(ro_len_opt)
qubit1.measure.integration_time(ro_len_opt)

qubit2.measure.pulse_duration(ro_len_opt)
qubit2.measure.integration_time(ro_len_opt)

In [ ]:
nodes.multiplexed_readout_calibration([qubit1, qubit2], plot=True);

In [ ]:
qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

states = ManualParameter(name="states")
states.batched = True
states.batch_size = 600

RO_cali_sched_kwargs = {"qubits": [qubit1, qubit2], "prepared_states": states, "acq_protocol": "SSBIntegrationComplex"} 
# SSBIntegrationComplex or NumericalSeparatedWeightedIntegration

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=multiplexed_readout_calibration_sched_TWPA,
    schedule_kwargs=RO_cali_sched_kwargs,
    real_imag=True,
    batched=True,
    num_channels=2,
)
meas_ctrl.gettables(gettable)

In [ ]:
states_setpoints = np.hstack((np.zeros(3000), np.ones(3000)))

meas_ctrl.settables([states, qubit2.clock_freqs.readout])
meas_ctrl.setpoints_grid([states_setpoints, np.linspace(7266243706.220526-1e6, 7266243706.220526+1e6, 41)])

quantum_device.cfg_sched_repetitions(1)
multi_ro_ds = meas_ctrl.run("Multiplexed Readout Calibration")
multi_ro_ds.tuid

In [ ]:
ro_analysis = ReadoutCalibrationAnalysis(dataset=multi_ro_ds)
ro_analysis.run().display_figs_mpl()

threshold = ro_analysis.quantities_of_interest['acq_threshold'].nominal_value
rotation = (np.rad2deg(ro_analysis.quantities_of_interest['acq_rotation_rad'].nominal_value)) % 360
qubit1.measure.acq_threshold(threshold)
qubit1.measure.acq_rotation(rotation)

In [ ]:
temp_ds = multi_ro_ds.rename({'y0':'y4'})
temp_ds = temp_ds.rename({'y1':'y5'})
temp_ds = temp_ds.rename({'y2':'y0'})
temp_ds = temp_ds.rename({'y3':'y1'})

ro_analysis = ReadoutCalibrationAnalysis(dataset=temp_ds)
ro_analysis.run().display_figs_mpl()

threshold = ro_analysis.quantities_of_interest['acq_threshold'].nominal_value
rotation = (np.rad2deg(ro_analysis.quantities_of_interest['acq_rotation_rad'].nominal_value)) % 360
qubit2.measure.acq_threshold(threshold)
qubit2.measure.acq_rotation(rotation)

### 6.1.1 Qubit1 ZZ

#### Conditional Ramsey

In [ ]:
# SGS100A_1.frequency(370e6)
SGS100A_1.power(6)

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True
tau.batch_size = 600

condition = ManualParameter(name="condition", unit="#", label="Condition")
condition.batched = False

ramsey_sched_kwargs = {
    "qubit": qubit1,
    "qubit_c": qubit2,
    "condition": condition,
    "times": tau,
    "artificial_detuning": 500e3,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=coupled_ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
tau_setpoints = np.arange(qubit1.rxy.duration(), 10e-6, 0.08e-6)
condition_setpoints = [0, 1]

meas_ctrl.settables([tau, condition])
meas_ctrl.setpoints_grid([tau_setpoints, condition_setpoints])

# meas_ctrl.settables([tau, condition, SGS100A_1.frequency])
# meas_ctrl.setpoints_grid([tau_setpoints, condition_setpoints, np.linspace(488, 492, 21)*1e6])

# meas_ctrl.settables([tau, condition, SGS100A_1.power])
# meas_ctrl.setpoints_grid([tau_setpoints, condition_setpoints, Vp2dBm(np.linspace(0.1, 2, 51))])

quantum_device.cfg_sched_repetitions(200)
con_ram_ds = meas_ctrl.run("coupled ramsey chevron", soft_avg=1)
con_ram_ds.tuid

In [ ]:
conditional_ramsey_fit(con_ram_ds, plot=True)

In [ ]:
# 20260728-192707 freq sweep at 3 dBm
# 20260728-234738 freq sweep at 6 dBm
# 20260728-232737 power sweep at midway
# con_ram_ds = load_dataset('20260728-192707')
x2_data = np.unique(con_ram_ds.x2.values)
f01_0_list = []
f01_1_list = []
tau_0_list = []
tau_1_list = []
zeta_ZZ_list = []

for x2 in x2_data:
    temp_ds = con_ram_ds.where(con_ram_ds.x2 == x2, drop=True)
    zeta_ZZ, (fit_0, fit_1) = conditional_ramsey_fit(temp_ds, qubit1)
    zeta_ZZ_list.append(zeta_ZZ)
    f01_0_list.append(fit_0.params['frequency'].value)
    f01_1_list.append(fit_1.params['frequency'].value)
    tau_0_list.append(fit_0.params['tau'].value)
    tau_1_list.append(fit_1.params['tau'].value)

In [ ]:
np.abs(np.array(zeta_ZZ_list)).argmin()
x2_data[np.abs(np.array(zeta_ZZ_list)).argmin()]

In [ ]:
plt.plot(x2_data, np.array(zeta_ZZ_list)/1e3, 'o-')
# plt.plot(x2_data[:80], tau_0_list[:80])

In [ ]:
plt.plot(x2_data / 1e6, np.array(f01_0_list)/1e6, 'o-')
plt.plot(x2_data / 1e6, np.array(f01_1_list)/1e6, 'o-')

In [ ]:
temp_ds = con_ram_ds.where(con_ram_ds.x2 == x2_data[10], drop=True)
zeta_ZZ, (fit_0, fit_1) = conditional_ramsey_fit(temp_ds, qubit1)

In [ ]:
fit_0.plot_fit()

In [ ]:
np.rad2deg(-432549 * 2*np.pi * (497e-9 + 16e-9))

#### Phase Ramsey

In [ ]:
# SGS100A_1.frequency(8.431e9)
# SGS100A_1.power(15.7)
# SGS100A_1.frequency(6e9)
SGS100A_1.power(3)

In [ ]:
SGS100A_1.on()

In [ ]:
phase = ManualParameter(name="phase", unit="deg", label="Phase")
phase.batched = True
phase.batch_size = 200

condition = ManualParameter(name="condition", unit="#", label="Condition")
condition.batched = False

ramsey_sched_kwargs = {
    "qubit": qubit1,
    "qubit_c": qubit2,
    "condition": condition,
    "phases": phase,
    "tau": 0.5e-6
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=coupled_phase_ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
phase_setpoints = np.linspace(-180, 180, 61)
condition_setpoints = [0, 1]

# meas_ctrl.settables([phase, condition])
# meas_ctrl.setpoints_grid([phase_setpoints, condition_setpoints])

meas_ctrl.settables([phase, condition, SGS100A_1.frequency])
meas_ctrl.setpoints_grid([phase_setpoints, condition_setpoints, np.linspace(480, 500, 41)*1e6])

quantum_device.cfg_sched_repetitions(300)
phase_ram_ds = meas_ctrl.run("coupled ramsey chevron", soft_avg=1)
phase_ram_ds.tuid

In [ ]:
# 20260728-164100 freq sweep
x0_data = np.unique(phase_ram_ds.x0)
x1_data = np.unique(phase_ram_ds.x1)
x2_data = np.unique(phase_ram_ds.x2)

y0_data = phase_ram_ds.y0.values.reshape(x2_data.size, x1_data.size, x0_data.size)
y1_data = phase_ram_ds.y1.values.reshape(x2_data.size, x1_data.size, x0_data.size)

phase0_list = []
phase1_list = []
for i in range(x2_data.size):
    c0_data = rotate_real_imag(y0_data[i, 0, :], y1_data[i, 0, :])[0]
    c1_data = rotate_real_imag(y0_data[i, 1, :], y1_data[i, 1, :])[0]

    phase0_list.append(phase_ramsey_fit(x0_data, c0_data))
    phase1_list.append(phase_ramsey_fit(x0_data, c1_data))

phase0_list = np.array(phase0_list)
phase1_list = np.array(phase1_list)

In [ ]:
phase_diff = []

for i in range(x2_data.size):
    if phase0_list[i] > phase1_list[i]:
        phase_diff.append(phase1_list[i] - phase0_list[i])
    else:
        phase_diff.append(phase1_list[i] - (phase0_list[i] + 360))

phase_diff = np.array(phase_diff)

In [ ]:
np.rad2deg(zeta_ZZ * 2*np.pi * qubit1_qubit2.iswap.pulse_duration())

In [ ]:
np.deg2rad(phase_diff) / (2*np.pi * 0.5e-6)

In [ ]:
# plt.plot(x2_data, phase0_list)
# plt.plot(x2_data, phase1_list)
# plt.ylim(80, 120)
# plt.plot(x2_data / 1e6, phase_diff, 'o--')
plt.plot(x2_data / 1e6, np.deg2rad(phase_diff) / (2*np.pi * 0.53e-6) / 1e3, 'o--')
plt.plot(x2_data / 1e6, np.array(zeta_ZZ_list)/1e3, 'o--')

In [ ]:
x0_data = np.unique(phase_ram_ds.x0)

y0_data = phase_ram_ds.y0.values.reshape(phase_ram_ds.ylen, phase_ram_ds.xlen)
y1_data = phase_ram_ds.y1.values.reshape(phase_ram_ds.ylen, phase_ram_ds.xlen)

c0_data = np.abs(rotate_real_imag(y0_data[0], y1_data[0])[0])
c1_data = np.abs(rotate_real_imag(y0_data[1], y1_data[1])[0])

x_peak0 = phase_ramsey_fit(x0_data, c0_data)
x_peak1 = phase_ramsey_fit(x0_data, c1_data)
print(f"x_peak0 = {x_peak0:.4g}, x_peak1 = {x_peak1:.4g}, difference = {x_peak1 - x_peak0:.4g}")

plt.plot(x0_data, c0_data, 'o--', label='Condition 0')
plt.plot(x0_data, c1_data, 's--', label='Condition 1')
plt.xlabel('Phase (deg)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

In [ ]:
def phase_ramsey_fit(x, y):
    def cos_model(x, A, freq, phase, offset):
        return A * np.cos(2 * np.pi * freq * x + phase) + offset

    offset0 = np.mean(y)
    A0 = (np.max(y) - np.min(y)) / 2

    dx = np.mean(np.diff(x))
    yf = np.fft.rfft(y - offset0)
    freqs = np.fft.rfftfreq(len(x), d=dx)
    freq0 = freqs[np.argmax(np.abs(yf[1:])) + 1]

    p0 = [A0, freq0, 0.0, offset0]
    popt, pcov = curve_fit(cos_model, x, y, p0=p0, maxfev=10000)
    A, freq, phase, offset = popt

    # normalize so A is positive (fold sign into phase) so the peak formula
    # below (which assumes A > 0) is valid
    if A < 0:
        A = -A
        phase = phase + np.pi

    period = 1 / freq
    x_peak0 = -phase / (2 * np.pi * freq)

    x_min, x_max = x.min(), x.max()
    n_shift = np.round((np.mean([x_min, x_max]) - x_peak0) / period)
    x_peak = x_peak0 + n_shift * period

    return x_peak


#### Flux tuning

In [ ]:
R1_curr = np.loadtxt('R1_curr075_0623.csv', delimiter=',', skiprows=1)[:, 0]
R1_freq = np.loadtxt('R1_curr075_0623.csv', delimiter=',', skiprows=1)[:, 1]
R2_curr = np.loadtxt('R2_curr075_0623.csv', delimiter=',', skiprows=1)[:, 0]
R2_freq = np.loadtxt('R2_curr075_0623.csv', delimiter=',', skiprows=1)[:, 1]

Q1_curr = np.loadtxt('Q1_curr075_0623.csv', delimiter=',', skiprows=1)[:, 0]
Q1_freq = np.loadtxt('Q1_curr075_0623.csv', delimiter=',', skiprows=1)[:, 1]
Q2_curr = np.loadtxt('Q2_curr075_0623.csv', delimiter=',', skiprows=1)[:, 0]
Q2_freq = np.loadtxt('Q2_curr075_0623.csv', delimiter=',', skiprows=1)[:, 1]

# Flux_R1_fit = np.load('Flux_R1_0622_fit.npz')
# Q1_curr_freq = np.loadtxt('Q1_curr_freq_0622.csv')

# Flux_R2_fit = np.load('Flux_R2_0622_fit.npz')
# Q2_curr_freq = np.loadtxt('Q2_curr_freq_0622.csv')

In [ ]:
meas_ctrl.verbose(False)

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True
tau.batch_size = 501

detuning = ManualParameter(name="detuning", unit="Hz", label="Detuning")
detuning.batched = False

for i, c in enumerate(tqdm(R1_curr)):
    if np.isnan(Q2_curr_freq[i]) or np.isnan(Q1_curr_freq[i]):
        continue

    GS210_1.current(c/1e3)
    GS210_1.on()
    
    qubit1.clock_freqs.f01(Q1_freq[i])
    qubit1.clock_freqs.readout(R1_freq[i])
    qubit2.clock_freqs.f01(Q2_freq[i])
    qubit2.clock_freqs.readout(R2_freq[i])

    ramsey_sched_kwargs = {
        "qubit": qubit1,
        "qubit_c": qubit2,
        "times": tau,
        "artificial_detuning": detuning,
    }
    
    gettable = ScheduleGettable(
        quantum_device,
        schedule_function=coupled_ramsey_sched_TWPA,
        schedule_kwargs=ramsey_sched_kwargs,
        real_imag=False,
        batched=True,
    )
    meas_ctrl.gettables(gettable)
    
    tau_setpoints = np.arange(0, 10e-6, 0.02e-6)
    # detuning_setpoints = np.linspace(-300e3, 300e3, 11)
    detuning_setpoints = [200e3]
    
    meas_ctrl.settables([tau, detuning])
    meas_ctrl.setpoints_grid([tau_setpoints, detuning_setpoints])
    
    quantum_device.cfg_sched_repetitions(200)
    ram_che_ds = meas_ctrl.run("coupled ramsey flux", soft_avg=1)

    ramsey_sched_kwargs = {
        "qubit": qubit1,
        "times": tau,
        "artificial_detuning": detuning,
    }
    
    gettable = ScheduleGettable(
        quantum_device,
        schedule_function=ramsey_sched_TWPA,
        schedule_kwargs=ramsey_sched_kwargs,
        real_imag=False,
        batched=True,
    )
    meas_ctrl.gettables(gettable)

    tau_setpoints = np.arange(0, 10e-6, 0.02e-6)
    # detuning_setpoints = np.linspace(-300e3, 300e3, 11)
    detuning_setpoints = [200e3]
    
    meas_ctrl.settables([tau, detuning])
    meas_ctrl.setpoints_grid([tau_setpoints, detuning_setpoints])
    
    quantum_device.cfg_sched_repetitions(200)    
    ram_che_ds = meas_ctrl.run("ramsey flux", soft_avg=1)
    GS210_1.off()

### 6.1.2 Qubit2 ZZ

#### Chevron

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True
tau.batch_size = 100

detuning = ManualParameter(name="detuning", unit="Hz", label="Detuning")
detuning.batched = False

ramsey_sched_kwargs = {
    "qubit": qubit2,
    "qubit_c": qubit1,
    "times": tau,
    "artificial_detuning": detuning,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=coupled_ramsey_sched_TWPA,
    schedule_kwargs=ramsey_sched_kwargs,
    real_imag=False,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
tau_setpoints = np.arange(qubit2.rxy.duration(), 40e-6, 0.4e-6)
detuning_setpoints = detuning_setpoints = np.linspace(00e3, 500e3, 41)

meas_ctrl.settables([tau, detuning])
meas_ctrl.setpoints_grid([tau_setpoints, detuning_setpoints])

quantum_device.cfg_sched_repetitions(1000)
ram_che_ds = meas_ctrl.run("coupled ramsey chevron", soft_avg=1)
ram_che_ds

In [ ]:
ramsey_tuid = '20250303-224735-777-39ae15'

ramsey_f01 = load_snapshot(ramsey_tuid)['instruments']['qubit2']['submodules']['clock_freqs']['parameters']['f01']['value']
ram_che_ds = load_dataset(ramsey_tuid)

xlen = ram_che_ds.xlen
ylen = ram_che_ds.ylen

tau = ram_che_ds.x0.values.reshape((ylen,xlen))[0]
detuning = ram_che_ds.x1.values.reshape((ylen,xlen)).T[0]
y0data = ram_che_ds.y0.values.reshape(ram_che_ds.ylen,ram_che_ds.xlen)

In [ ]:
plt.pcolor(tau*1e6, detuning/1e3, y0data, rasterized=True)
plt.ylabel('Detuning [kHz]')
plt.xlabel(r'$\tau$ [$\mu$s]')

#### Flux tuning

In [ ]:
Flux_R1_fit = np.load('Flux_R1_0130_fit.npz')
Q1_curr_freq = np.loadtxt('Q1_curr_freq_0201.csv')

In [ ]:
Flux_R2_fit = np.load('Flux_R2_0131_fit.npz')
Q2_curr_freq = np.loadtxt('Q2_curr_freq_0131.csv')

In [ ]:
meas_ctrl.verbose(False)

In [ ]:
tau = ManualParameter(name="tau", unit="s", label="Time")
tau.batched = True
tau.batch_size = 200

detuning = ManualParameter(name="detuning", unit="Hz", label="Detuning")
detuning.batched = False

for i, c in enumerate(tqdm(Flux_R1_fit['current'])):
    if i <= 80:
        continue
    if np.isnan(Q2_curr_freq[i]) or np.isnan(Q1_curr_freq[i]):
        continue

    GS210_1.current(c/1e3)
    GS210_1.on()
    
    qubit1.clock_freqs.f01(Q1_curr_freq[i])
    qubit1.clock_freqs.readout(Flux_R1_fit['fr_arr'][i]*1e9)
    qubit2.clock_freqs.f01(Q2_curr_freq[i])
    qubit2.clock_freqs.readout(Flux_R2_fit['fr_arr'][i]*1e9)

    ramsey_sched_kwargs = {
        "qubit": qubit2,
        "qubit_c": qubit1,
        "times": tau,
        "artificial_detuning": detuning,
    }
    
    gettable = ScheduleGettable(
        quantum_device,
        schedule_function=coupled_ramsey_sched_TWPA,
        schedule_kwargs=ramsey_sched_kwargs,
        real_imag=False,
        batched=True,
    )
    meas_ctrl.gettables(gettable)
    
    tau_setpoints = np.arange(qubit2.rxy.duration(), 20e-6, 0.1e-6)
    detuning_setpoints = detuning_setpoints = np.linspace(-500e3, 100e3, 11)
    
    meas_ctrl.settables([tau, detuning])
    meas_ctrl.setpoints_grid([tau_setpoints, detuning_setpoints])
    
    quantum_device.cfg_sched_repetitions(200)
    ram_che_ds = meas_ctrl.run("coupled ramsey flux", soft_avg=1)

    ramsey_sched_kwargs = {
        "qubit": qubit2,
        "times": tau,
        "artificial_detuning": detuning,
    }
    
    gettable = ScheduleGettable(
        quantum_device,
        schedule_function=ramsey_sched_TWPA,
        schedule_kwargs=ramsey_sched_kwargs,
        real_imag=False,
        batched=True,
    )
    meas_ctrl.gettables(gettable)

    tau_setpoints = np.arange(qubit2.rxy.duration(), 20e-6, 0.1e-6)
    detuning_setpoints = detuning_setpoints = np.linspace(-300e3, 300e3, 11)
    
    meas_ctrl.settables([tau, detuning])
    meas_ctrl.setpoints_grid([tau_setpoints, detuning_setpoints])
    
    quantum_device.cfg_sched_repetitions(200)    
    ram_che_ds = meas_ctrl.run("ramsey flux", soft_avg=1)
    GS210_1.off()

### 6.2 iSWAP Phase Calibration

In [ ]:
# qubit1_qubit2.iswap.pulse_frequency(pump_freq-20e6)
SGS100A_1.frequency(pump_freq-50e6)

In [ ]:
TwoQubitClifford(384).gate_decomposition()

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of iSWAP")
length.batched = True
# length_setpoints = np.arange(1, 17, 1)
length_setpoints = np.arange(2, 15, 2) # pump phase insensitive, individual qubit's AC Stark shift calibrated

vz_angle = ManualParameter(name="angle", unit="Deg", label="Angle")
vz_angle.batched = False
vz_angle_setpoints = np.linspace(-180, 180, 37)

vz_sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    "angle": vz_angle,
    "gate_idx": 384,
    "acq_protocol": "ThresholdedAcquisition",
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=iswap_virtual_z_schedule,
    schedule_kwargs=vz_sched_kwargs,
    batched=True,
    num_channels=2,
    max_batch_size=vz_angle_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

meas_ctrl.settables([length, vz_angle])
meas_ctrl.setpoints_grid([length_setpoints, vz_angle_setpoints])

# meas_ctrl.settables([length, vz_angle, SGS100A_1.frequency])
# meas_ctrl.setpoints_grid([length_setpoints, vz_angle_setpoints, np.linspace(pump_freq-5e6, pump_freq+5e6, 21)])

# meas_ctrl.settables([length, SGS100A_1.phase])
# meas_ctrl.setpoints_grid([length_setpoints, np.linspace(0,360,41)])

quantum_device.cfg_sched_repetitions(200)

In [ ]:
vz_iswap_ds = meas_ctrl.run("iSWAP VZ Calibration", soft_avg=1)
display(vz_iswap_ds.tuid)

In [ ]:
vz_iswap_tuid_list = []
meas_ctrl.verbose(False)
for _ in tqdm(range(30)):
    vz_iswap_ds = meas_ctrl.run("iSWAP VZ Calibration", soft_avg=1)
    vz_iswap_tuid_list.append(vz_iswap_ds.tuid)
meas_ctrl.verbose(True)

In [ ]:
# np.savetxt(get_datadir() + '/phase_test_3.txt', vz_iswap_tuid_list, fmt='%s')

In [ ]:
# vz_iswap_ds = load_dataset('20260325-174015-331-a8601c') # Q1 as target -41.543
# vz_iswap_ds = load_dataset('20260325-175558-250-0ccbb2') # Q2 as target 50.945
# vz_iswap_ds = load_dataset('20260326-123412-867-2c902d') # Q1 as target -37.603 with more operation buffer
# vz_iswap_ds = load_dataset('20260326-123800-618-987adb') # Q2 as target 67.638 with more operation buffer

# vz_iswap_ds = load_dataset('20260506-143839') # Q2 frequency sweep
# vz_iswap_ds = load_dataset('20260506-154110') # Q1 frequency sweep

# vz_iswap_ds = vz_iswap_ds.where(vz_iswap_ds.x2 == np.unique(vz_iswap_ds.x2)[5], drop=True)

N_iswap = np.unique(vz_iswap_ds.x0)
angle = np.unique(vz_iswap_ds.x1)
pop_data = vz_iswap_ds.y1.values.reshape(angle.size, N_iswap.size).T

plt.pcolormesh(angle, N_iswap, pop_data, rasterized=True)
# plt.axvline(90, ls='--', c='white')

plt.xlabel('Angle [Deg]')
plt.ylabel(f'Number of iSWAP')
plt.show()

In [ ]:
from scipy.optimize import curve_fit

def sin_func(x, amp, freq, phase, offset):
    return amp * np.sin(2 * np.pi * freq * x + phase) + offset

opt_angles = []

for idx, n in enumerate(N_iswap):
    # if n % 2 == 0 and n > 0:
    # if n > 0:
        trace = pop_data[idx]
        
        # Initial guess
        amp_guess = (np.max(trace) - np.min(trace)) / 2
        offset_guess = np.mean(trace)
        freq_guess = 1 / 360 # Assuming one full period over 360 degrees
        phase_guess = 0
        
        p0 = [amp_guess, freq_guess, phase_guess, offset_guess]
        
        try:
            popt, _ = curve_fit(sin_func, angle, trace, p0=p0)
            
            # Generate fitted curve over high-resolution angle array
            angle_hr = np.linspace(angle.min(), angle.max(), 1000)
            fit_trace = sin_func(angle_hr, *popt)
            
            # plt.plot(angle, trace, 'o', label=f'N={n} Data')
            # plt.plot(angle_hr, fit_trace, '-', label=f'N={n} Fit')
            
            if n % 4 == 2: # 2, 6, 10 ...
                opt_angle = angle_hr[np.argmin(fit_trace)]
                opt_angles.append(opt_angle)
                print(f"N_iswap = {n}: Minimum at {opt_angle:.3f} Deg")
            elif n % 4 == 0: # 4, 8, 12 ...
                opt_angle = angle_hr[np.argmax(fit_trace)]
                opt_angles.append(opt_angle)
                print(f"N_iswap = {n}: Maximum at {opt_angle:.3f} Deg")
                
        except RuntimeError:
            print(f"Fit failed for N_iswap = {n}")

# plt.xlabel('Angle [Deg]')
# plt.ylabel('Population')
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.show()

# print(f"Average optimal angle: {np.mean(opt_angles):.3f} Deg")

In [ ]:
# Unwrap the angles (assuming they wrap around 360 deg)
# We can convert to radians, unwrap, then back to degrees
opt_angles_rad = np.deg2rad(opt_angles)
unwrapped_rad = np.unwrap(opt_angles_rad)
unwrapped_deg = np.rad2deg(unwrapped_rad) 
# unwrapped_deg = opt_angles

# Generate N_iswap array for the x-axis
# n_iswap_vals = np.array([n for n in N_iswap if n % 2 == 0 and n > 0])
n_iswap_vals = N_iswap

# Linear fit: unwrapped_deg = slope * n_iswap_vals + intercept
p_poly, p_err = np.polyfit(n_iswap_vals, unwrapped_deg, 1, cov=True)
slope, intercept = p_poly
slope_err, intercept_err = np.sqrt(np.diag(p_err))
linear_fit = np.polyval(p_poly, n_iswap_vals)

plt.figure(figsize=(8, 5))
plt.plot(n_iswap_vals, unwrapped_deg, 'o', label='Unwrapped Angles')
plt.plot(n_iswap_vals, linear_fit, '-', label=f'Linear Fit\nSlope: {slope:.3f} deg/gate\nIntercept: {intercept:.3f} deg')
plt.xlabel('N_iswap')
plt.ylabel('Unwrapped Optimal Angle [Deg]')
plt.title('Linear Fit of Optimal Angles vs N_iswap')
plt.legend()
plt.grid(True)
plt.show()

print(f"Fitted Slope     = {np.round(slope, 1)} ± {np.round(slope_err, 1)} deg/step")
print(f"Fitted Intercept = {np.round(intercept, 1)} ± {np.round(intercept_err, 1)} deg")

In [ ]:
p_poly, p_err = np.polyfit(n_iswap_vals[::2], unwrapped_deg[::2], 1, cov=True)
slope, intercept = p_poly
slope_err, intercept_err = np.sqrt(np.diag(p_err))
linear_fit = np.polyval(p_poly, n_iswap_vals[::2])

plt.figure(figsize=(8, 5))
plt.plot(n_iswap_vals[::2], unwrapped_deg[::2], 'o', label='Unwrapped Angles')
plt.plot(n_iswap_vals[::2], linear_fit, '-', label=f'Linear Fit\nSlope: {slope:.3f} deg/gate\nIntercept: {intercept:.3f} deg')

p_poly, p_err = np.polyfit(n_iswap_vals[1::2], unwrapped_deg[1::2], 1, cov=True)
slope, intercept = p_poly
slope_err, intercept_err = np.sqrt(np.diag(p_err))
linear_fit = np.polyval(p_poly, n_iswap_vals[1::2])

plt.plot(n_iswap_vals[1::2], unwrapped_deg[1::2], 'o', label='Unwrapped Angles')
plt.plot(n_iswap_vals[1::2], linear_fit, '-', label=f'Linear Fit\nSlope: {slope:.3f} deg/gate\nIntercept: {intercept:.3f} deg')

plt.xlabel('N_iswap')
plt.ylabel('Unwrapped Optimal Angle [Deg]')
plt.title('Linear Fit of Optimal Angles vs N_iswap')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# [no pre, 1 pre, 2 pre, 3 pre, 5 pre, 10 pre], detuning = -25 MHz, didn't see effect on 0 iSWAP case
N_pre = [0, 1, 2, 3, 5, 10] # number of pre-pulses
test_slope1 = [-118.0, -117.7, -118.2, -118.9, -118.5, -114.7]
test_intercept1 = [16.5, 12.5, 10.7, 10.1, 7.8, 7]

# detuning = -25 MHz, changing iSWAP length
test_length = [4, 10, 16, 20, 30, 40, 60, 70, 100, 120, 140, 180] # ns
test_slope2 = [-0.1, -0.5, -8.6, -19.5, -49.9, -77.8, -131.5, -158.1, 124.8-360, 72.6-360, 22.4-360, -82.5-360]
test_intercept2 = [1.8, 1.7, 4.9, 7, 8.3, 8.0, 6.7, 8.3, 6.5, 10.1, 7.5, 9.9]

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(12, 4))

# First subplot
ax1.set_xlabel('Number of pre-pulses (N_pre)')
ax1.set_ylabel('Slope', color='tab:blue')
ax1.plot(N_pre, test_slope1, 'o-', color='tab:blue', label='Slope')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_title('Slope and Intercept vs N_pre')

ax2 = ax1.twinx()
ax2.set_ylabel('Intercept', color='tab:orange')
ax2.plot(N_pre, test_intercept1, 's-', color='tab:orange', label='Intercept')
ax2.tick_params(axis='y', labelcolor='tab:orange')

# Second subplot
ax3.set_xlabel('iSWAP length (ns)')
ax3.set_ylabel('Slope', color='tab:blue')
ax3.plot(test_length, test_slope2, 'o-', color='tab:blue', label='Slope')
ax3.tick_params(axis='y', labelcolor='tab:blue')
ax3.set_title('Slope and Intercept vs iSWAP length')

ax4 = ax3.twinx()
ax4.set_ylabel('Intercept', color='tab:orange')
ax4.plot(test_length, test_intercept2, 's-', color='tab:orange', label='Intercept')
ax4.tick_params(axis='y', labelcolor='tab:orange')

fig.tight_layout()
plt.show()

In [ ]:
plt.plot(test_x, test_y, 'o-')

In [ ]:
[0, 40, 80, 120] # extra idle time after each iSWAP
[30.8, 47.3, 59.4, 67.1]
[30.5, 16.5, 13.7, 14.2]

[0, 40, 80, 120] # extra idle time before each iSWAP
[36.3, 52.6, 66.5, 75.8]
[30.4, 16.4, 16.0, 9.1]

[0, 1, 2, 3, 4] # operation buffer time
[11.8, -8.4, -2.5, 27.4, 34.9]
[50.9, 69.8, 56.4, 28.2, 24.4]

# iSWAP pump power off
[-2]
[0.8]

[0, 40, 80, 120] # extra idle time after the first X/2
[34.7, 36.0, 36.5, 38.0]
[36.3, 25.0, 26.0, 27.8]

[0, 40, 80, 120] # extra idle time before the last X/2
[38.3, 38.9, 38.4, 36.4]
[36.2, 31.5, 26.9, 32.7]

In [ ]:
(pump_freq - (qubit2.clock_freqs.f01() - qubit1.clock_freqs.f01())) * 40e-9 * 360

In [ ]:
# std_pulses = pop_data.std(axis=0)
# plt.plot(angle, std_pulses)
# plt.xlabel('Angle [Deg]')
# plt.ylabel('Standard Deviation Along Pulses')
# print(f'Min angle: {round(angle[np.argmin(std_pulses)], 3)} Deg')
# print(f'Max angle: {round(angle[np.argmax(std_pulses)], 3)} Deg')

### 6.3 Two-qubit RB

In [ ]:
# """The module provides classes related iSWAP operations."""

# from __future__ import annotations

# from typing import TYPE_CHECKING

# from qcodes.instrument import InstrumentChannel
# from qcodes.instrument.parameter import ManualParameter

# from quantify_scheduler.backends.graph_compilation import OperationCompilationConfig
# from quantify_scheduler.device_under_test.edge import Edge
# from quantify_scheduler.helpers.validators import Numbers
# from quantify_scheduler.resources import BasebandClockResource

# if TYPE_CHECKING:
#     from qcodes.instrument.base import InstrumentBase

# from quantify_scheduler.operations import pulse_library
# from quantify_scheduler.schedules import Schedule
    
# def composite_marker_pulse(
#     pulse_frequency: float,
#     pulse_amp: float,
#     pulse_duration: float,
#     # snail_drive: any,
# ) -> pulse_library.MarkerPulse:

#     # snail_drive.frequency(pulse_frequency)
#     # snail_drive.power(pulse_amp)

#     composite_pulse = pulse_library.MarkerPulse(
#         duration=pulse_duration,
#         port="snail:switch",
#     )

#     return composite_pulse


# class iSWAP(InstrumentChannel):
#     """Submodule containing parameters for performing a iSWAP operation."""

#     def __init__(self, parent: InstrumentBase, name: str, **kwargs: float) -> None:
#         super().__init__(parent=parent, name=name)

#         self.pulse_frequency = ManualParameter(
#             "pulse_frequency",
#             docstring=r"""The pulse frequency in Hz.""",
#             unit="Hz",
#             initial_value=kwargs.get("pulse_frequency", 500e6),
#             vals=Numbers(min_value=0, max_value=2000e6, allow_nan=True),
#             instrument=self,
#         )

#         self.pulse_amp = ManualParameter(
#             "pulse_amp",
#             docstring=r"""Amplitude of the pulse envelope.""",
#             unit="dBm",
#             initial_value=kwargs.get("pulse_amp", 0.0),
#             vals=Numbers(min_value=-100, max_value=30, allow_nan=True),
#             instrument=self,
#         )

#         self.pulse_duration = ManualParameter(
#             "pulse_duration",
#             docstring=r"""The pulse duration in seconds.""",
#             unit="s",
#             initial_value=kwargs.get("pulse_duration", 1e-6),
#             vals=Numbers(min_value=0, max_value=500e-6, allow_nan=True),
#             instrument=self,
#         )


# class CompositeiSWAPEdge(Edge):
#     """
#     An example iSWAP implementation which connects two BasicTransmonElements.

#     This edge implements a square pulse with a R&S SSG for the iSWAP operation
#     between the two BasicTransmonElements.
#     """

#     def __init__(
#         self,
#         parent_element_name: str,
#         child_element_name: str,
#         **kwargs,
#     ) -> None:
#         iswap_data = kwargs.pop("iswap", {})

#         super().__init__(
#             parent_element_name=parent_element_name,
#             child_element_name=child_element_name,
#             **kwargs,
#         )

#         self.add_submodule("iswap", iSWAP(parent=self, name="iswap", **iswap_data))


#     def generate_edge_config(self) -> dict[str, dict[str, OperationCompilationConfig]]:
#         """
#         Generate valid device config.

#         Fills in the edges information to produce a valid device config for the
#         quantify-scheduler making use of the
#         :func:`~.circuit_to_device.compile_circuit_to_device_with_config_validation` function.
#         """
#         edge_op_config = {
#             f"{self.name}": {
#                 "iSWAP": OperationCompilationConfig(
#                     factory_func=composite_marker_pulse,
#                     factory_kwargs={
#                         "pulse_frequency": self.iswap.pulse_frequency(),
#                         "pulse_amp": self.iswap.pulse_amp(),
#                         "pulse_duration": self.iswap.pulse_duration(),
#                         # "snail_drive": SGS100A_1,
#                     },
#                 ),
#             }
#         }

#         return edge_op_config

In [ ]:
"""The module provides classes related iSWAP operations."""

from __future__ import annotations

from typing import TYPE_CHECKING

from qcodes.instrument import InstrumentChannel
from qcodes.instrument.parameter import ManualParameter

from quantify_scheduler.backends.graph_compilation import OperationCompilationConfig
from quantify_scheduler.device_under_test.edge import Edge
from quantify_scheduler.helpers.validators import Numbers
from quantify_scheduler.resources import BasebandClockResource

if TYPE_CHECKING:
    from qcodes.instrument.base import InstrumentBase

from quantify_scheduler.operations import pulse_library
from quantify_scheduler.schedules import Schedule
    
def composite_marker_pulse(
    pulse_frequency: float,
    pulse_amp: float,
    pulse_duration: float,
    # snail_drive: any,
) -> pulse_library.MarkerPulse:

    # snail_drive.frequency(pulse_frequency)
    # snail_drive.power(pulse_amp)

    composite_pulse = pulse_library.MarkerPulse(
        duration=pulse_duration,
        port="snail:switch",
    )

    return composite_pulse


class iSWAPChannel(InstrumentChannel):
    """Submodule containing parameters for performing a iSWAP operation.

    Named iSWAPChannel (not iSWAP) so that ``from pycqed_randomized_benchmarking.utils
    import *`` -- which exports the iSWAP gate Operation(qC, qT) -- cannot shadow it.
    """

    def __init__(self, parent: InstrumentBase, name: str, **kwargs: float) -> None:
        super().__init__(parent=parent, name=name)

        self.pulse_frequency = ManualParameter(
            "pulse_frequency",
            docstring=r"""The pulse frequency in Hz.""",
            unit="Hz",
            initial_value=kwargs.get("pulse_frequency", 500e6),
            vals=Numbers(min_value=0, max_value=2000e6, allow_nan=True),
            instrument=self,
        )

        self.pulse_amp = ManualParameter(
            "pulse_amp",
            docstring=r"""Amplitude of the pulse envelope.""",
            unit="dBm",
            initial_value=kwargs.get("pulse_amp", 0.0),
            vals=Numbers(min_value=-100, max_value=30, allow_nan=True),
            instrument=self,
        )

        self.pulse_duration = ManualParameter(
            "pulse_duration",
            docstring=r"""The pulse duration in seconds.""",
            unit="s",
            initial_value=kwargs.get("pulse_duration", 1e-6),
            vals=Numbers(min_value=0, max_value=500e-6, allow_nan=True),
            instrument=self,
        )

        # Per-iSWAP AC-Stark phases (dissertation 4.3.2 part 3). Read by
        # pycqed_randomized_benchmarking.utils._ac_stark_phases and carried in
        # the virtual-Z register of _iswap_rz_corrections. The difference
        # q1 - q2 also absorbs the pump phase phi_p, so it is only valid for
        # the pump frequency/phase it was calibrated at.
        self.ac_stark_phase_q1 = ManualParameter(
            "ac_stark_phase_q1",
            docstring=r"""AC-Stark phase correction of qubit1 (the parent element) per iSWAP, in degrees.""",
            unit="deg",
            initial_value=kwargs.get("ac_stark_phase_q1", 0.0),
            vals=Numbers(min_value=-720, max_value=720, allow_nan=True),
            instrument=self,
        )

        self.ac_stark_phase_q2 = ManualParameter(
            "ac_stark_phase_q2",
            docstring=r"""AC-Stark phase correction of qubit2 (the child element) per iSWAP, in degrees.""",
            unit="deg",
            initial_value=kwargs.get("ac_stark_phase_q2", 0.0),
            vals=Numbers(min_value=-720, max_value=720, allow_nan=True),
            instrument=self,
        )


class CompositeiSWAPEdge(Edge):
    """
    An example iSWAP implementation which connects two BasicTransmonElements.

    This edge implements a square pulse with a R&S SSG for the iSWAP operation
    between the two BasicTransmonElements.
    """

    def __init__(
        self,
        parent_element_name: str,
        child_element_name: str,
        **kwargs,
    ) -> None:
        iswap_data = kwargs.pop("iswap", {})

        super().__init__(
            parent_element_name=parent_element_name,
            child_element_name=child_element_name,
            **kwargs,
        )

        self.add_submodule("iswap", iSWAPChannel(parent=self, name="iswap", **iswap_data))


    def generate_edge_config(self) -> dict[str, dict[str, OperationCompilationConfig]]:
        """
        Generate valid device config.

        Fills in the edges information to produce a valid device config for the
        quantify-scheduler making use of the
        :func:`~.circuit_to_device.compile_circuit_to_device_with_config_validation` function.
        """
        edge_op_config = {
            f"{self.name}": {
                "iSWAP": OperationCompilationConfig(
                    factory_func=composite_marker_pulse,
                    factory_kwargs={
                        "pulse_frequency": self.iswap.pulse_frequency(),
                        "pulse_amp": self.iswap.pulse_amp(),
                        "pulse_duration": self.iswap.pulse_duration(),
                        # "snail_drive": SGS100A_1,
                    },
                ),
            }
        }

        return edge_op_config


In [ ]:
# import contextlib
# from qcodes.instrument.base import Instrument

# _ISWAP_PARS = ("pulse_frequency", "pulse_amp", "pulse_duration")

# _old = Instrument._all_instruments.get("qubit1_qubit2")
# _keep = {p: _old.iswap.parameters[p]() for p in _ISWAP_PARS} if _old is not None else {}
# if _old is not None:
#     _old.close()

# qubit1_qubit2 = CompositeiSWAPEdge(
#     parent_element_name=qubit1.name, child_element_name=qubit2.name
# )
# quantum_device.add_edge(qubit1_qubit2)

# for p, v in _keep.items():
#     qubit1_qubit2.iswap.parameters[p](v)
# print("restored iswap:", _keep or "(fresh edge, defaults in use)")


In [ ]:
qubit1_qubit2 = CompositeiSWAPEdge(parent_element_name=qubit1.name, child_element_name=qubit2.name)
quantum_device.add_edge(qubit1_qubit2)

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import RBAnalysis, randomized_benchmarking_schedule, iswap_virtual_z_schedule, iswap_pulsed_pump_schedule

In [ ]:
qubit2.measure.acq_channel(1)

In [ ]:
# qubit1_qubit2.iswap.pulse_amp(-100)
# SGS100A_1.power(qubit1_qubit2.iswap.pulse_amp())
# qubit1_qubit2.iswap.pulse_frequency(400e6)
# SGS100A_1.frequency(qubit1_qubit2.iswap.pulse_frequency())
# qubit1_qubit2.iswap.pulse_duration(606e-9)
# qubit1_qubit2.iswap.pulse_duration(0e-9)

In [ ]:
qubit1_qubit2.iswap.pulse_duration()

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of Clifford gates")
length.batched = True
# length_setpoints = np.arange(0,40,4)
length_setpoints = np.arange(20)

seed = ManualParameter(name="seed", unit="", label="Seeds")
seed.batched = False
seed_setpoints = np.random.randint(0, 2**31 - 1, size=3, dtype=np.int32)
# seed_setpoints = np.array([3130])

rb_sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    "seeds": seed_setpoints,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=randomized_benchmarking_schedule,
    schedule_kwargs=rb_sched_kwargs,
    real_imag=False,
    batched=True,
    num_channels=2,
    max_batch_size=length_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

# meas_ctrl.settables([length, seed])
# meas_ctrl.setpoints_grid([length_setpoints, seed_setpoints])

meas_ctrl.settables([length, qubit1_qubit2.iswap.pulse_duration])
meas_ctrl.setpoints_grid([length_setpoints, np.arange(580, 610, 2)/1e9])

quantum_device.cfg_sched_repetitions(200)

In [ ]:
rb_2q_ds = meas_ctrl.run("Two-qubit Randomized benchmarking", soft_avg=1)
display(rb_2q_ds.tuid)

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import tomography_pre_build, tomography_schedule

def construct_confusion_matrix(
    qubit_specifier,
    instrument_coordinator=None,
    quantum_device=None,
    acq_protocol='ThresholdedAcquisition',
    bin_mode='append',
    repetitions=2000,
    pad_identity: bool = True,
):
    """
    Measure the readout confusion matrix by preparing each computational
    basis state and reconstructing its density matrix.

    Parameters
    ----------
    instrument_coordinator, quantum_device
        Both required -- forwarded to ``tomography_schedule``, which raises
        if either is missing.
    pad_identity
        Whether to pad the all-identity tomography rotation with an idle pulse.

    Returns
    -------
    np.ndarray
        4x4 confusion matrix, columns indexed by prepared state.
    """
    readout_confusion_matrix = np.empty((4, 4))
    for i, operation_idx in enumerate([0, 72, 3, 75]): # correspond to preparing |00>, |01>, |10>, |11> respectively
        rho, _ = calculate_density_matrix(
            tomography_schedule(
                qubit_specifier,
                operation_state_idx=operation_idx,
                repetitions=repetitions,
                instrument_coordinator=instrument_coordinator,
                quantum_device=quantum_device,
                acq_protocol=acq_protocol,
                bin_mode=bin_mode,
                pad_identity=pad_identity,
            )
        )
        readout_confusion_matrix[i] = np.diag(rho).real
    return readout_confusion_matrix.T

### 6.4 Tomography

In [ ]:
qubit1_qubit2.iswap.pulse_amp(9)
SGS100A_1.power(qubit1_qubit2.iswap.pulse_amp())
qubit1_qubit2.iswap.pulse_frequency(pump_freq)
SGS100A_1.frequency(qubit1_qubit2.iswap.pulse_frequency())
qubit1_qubit2.iswap.pulse_duration(491e-9) #253 #494

In [ ]:
# qubit1_qubit2.iswap.pulse_amp(12)
# SGS100A_1.power(qubit1_qubit2.iswap.pulse_amp())
# qubit1_qubit2.iswap.pulse_frequency(pump_freq)
# SGS100A_1.frequency(qubit1_qubit2.iswap.pulse_frequency())
# qubit1_qubit2.iswap.pulse_duration(343e-9)

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import TwoQubitClifford

In [ ]:
TwoQubitClifford(3).gate_decomposition()

In [ ]:
SGS100A_1.phase(0)

In [ ]:
qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

readout_confusion_matrix = construct_confusion_matrix(qubit_specifier=[qubit1, qubit2], repetitions=2000, quantum_device=quantum_device, instrument_coordinator=ic)
print(f"The confusion matrix is:\n{np.round(readout_confusion_matrix, 3)}")

In [ ]:
op_idx = 5760 + 16

tomo_phase_list = tomography_pre_build(
    [qubit1, qubit2], op_idx, quantum_device)

state_tg_ds = tomography_schedule(
    qubit_specifier=[qubit1, qubit2],
    operation_state_idx=op_idx,
    repetitions=4000,
    instrument_coordinator=ic,
    quantum_device=quantum_device,
    correction_phase=tomo_phase_list,
)

tomo_tuid = save_retrieve_acquisition_dataset(state_tg_ds, 'State Tomography')
print(tomo_tuid)

In [ ]:
rho, stokes = calculate_density_matrix(load_dataset(tomo_tuid), confusion_matrix=readout_confusion_matrix)
print(f"The density matrix is:\n{np.round(rho, 3)}")

rho_Qobj = qutip.Qobj(rho, dims=[[2,2],[2,2]])
rho_q1 = rho_Qobj.ptrace(0)
rho_q2 = rho_Qobj.ptrace(1)

print('========================')
print(f"Q1 density matrix is:\n{np.round(rho_q1.full(),3)}")
print('========================')
print(f"Q2 density matrix is:\n{np.round(rho_q2.full(),3)}")

In [ ]:
extract_population_phase(rho_q1, "Q1")
extract_population_phase(rho_q2, "Q2")

In [ ]:
t0 = T2t(rho2T(rho))  # Initial guess for the optimization, derived from the current density matrix
stokes = compute_stokes(rho)  # Measured Stokes parameters from the current density matrix

# Perform least squares optimization
mle_result = least_squares(mle_residuals, t0, args=(stokes,), max_nfev=3000)

# Extract the final physical density matrix
optimal_t = mle_result.x
T_optimal = t2T(optimal_t)
rho_mle = T2rho(T_optimal)

print("MLE optimization finished. Success:", mle_result.success)
assert(mle_result.success)

In [ ]:
print(f"The density matrix is:\n{np.round(rho_mle, 3)}")

# op_idx = 5760+3
# ideal_state = np.array([
#     [0. +0.j , 0. +0.j , 0. +0.j , 0. +0.j ],
#     [0. +0.j , 0.5+0.j , 0.5+0.j , 0. +0.j ],
#     [0. +0.j , 0.5+0.j , 0.5+0.j , 0. +0.j ],
#     [0. +0.j , 0. +0.j , 0. +0.j , 0. +0.j ]
# ])

op_idx = 5760+16
ideal_state = np.array([
    [0.5+0.j , 0.5+0.j , 0. +0.j , 0. +0.j ],
    [0.5+0.j , 0.5+0.j , 0. +0.j , 0. +0.j ],
    [0. +0.j , 0. +0.j , 0. +0.j , 0. +0.j ],
    [0. +0.j , 0. +0.j , 0. +0.j , 0. +0.j ]
])

# op_idx = 5760+384
# ideal_state = np.array([
#     [0.5+0.j , 0. +0.j , 0.5+0.j , 0. +0.j ],
#     [0. +0.j , 0. +0.j , 0. +0.j , 0. +0.j ],
#     [0.5+0.j , 0. +0.j , 0.5+0.j , 0. +0.j ],
#     [0. +0.j , 0. +0.j , 0. +0.j , 0. +0.j ]
# ])

sqrt_ideal_state = sqrtm(ideal_state)

state_fidelity = (np.abs(sqrtm(sqrt_ideal_state @ rho_mle @ sqrt_ideal_state).trace())) ** 2

print(f"The state fidelity is: {np.round(state_fidelity, decimals=3)}")

In [ ]:
bell_state_rho_list = []
bell_state_ds_tuid_list = []
bell_cm_list = []          # NEW: confusion matrix used for each run
bell_lin_rho_list = []     # NEW: CM-corrected linear-inversion rho (before MLE), for pooling
bell_f_list = []

tomo_phase_list = tomography_pre_build(
    [qubit1, qubit2], op_idx, quantum_device)

for i in tqdm(range(10)):
    readout_confusion_matrix = construct_confusion_matrix(qubit_specifier=[qubit1, qubit2], repetitions=2000,
                                                          quantum_device=quantum_device, instrument_coordinator=ic)
    state_tg_ds = tomography_schedule(
        qubit_specifier=[qubit1, qubit2],
        operation_state_idx=op_idx,
        repetitions=4000,
        instrument_coordinator=ic,
        quantum_device=quantum_device,
        correction_phase=tomo_phase_list,
    )
    tomo_tuid = save_retrieve_acquisition_dataset(state_tg_ds, 'State Tomography')

    rho, stokes = calculate_density_matrix(load_dataset(tomo_tuid), confusion_matrix=readout_confusion_matrix)

    t0 = T2t(rho2T(rho))
    stokes = compute_stokes(rho)
    mle_result = least_squares(mle_residuals, t0, args=(stokes,), max_nfev=3000)
    rho_mle = T2rho(t2T(mle_result.x))

    if(mle_result.success):
        bell_state_rho_list.append(rho_mle)
        bell_state_ds_tuid_list.append(tomo_tuid)
        bell_cm_list.append(readout_confusion_matrix.copy())   # NEW
        bell_lin_rho_list.append(rho)                          # NEW
        bell_f = (np.abs(sqrtm(sqrt_ideal_state @ rho_mle @ sqrt_ideal_state).trace())) ** 2
        bell_f_list.append(bell_f)
        print(f"Run {i+1}: MLE optimization finished. Success: {mle_result.success}, Fidelity: {np.round(bell_f, decimals=3)}")

print(np.round(np.array(bell_state_rho_list).mean(axis=0), decimals=3))
print('per-run mean F (biased low):', np.round(np.array(bell_f_list).mean(), decimals=4),
      np.round(np.array(bell_f_list).std(), decimals=4))

# NEW: pooled estimate -- average the linear-inversion rho, one MLE, bootstrap over runs for the error bar
def pooled_fidelity(lin_rhos):
    rho_avg = np.mean(lin_rhos, axis=0)
    res = least_squares(mle_residuals, T2t(rho2T(rho_avg)), args=(compute_stokes(rho_avg),), max_nfev=3000)
    return np.real(np.trace(ideal_state @ T2rho(t2T(res.x))))

lin_rhos = np.array(bell_lin_rho_list)
pooled_f = pooled_fidelity(lin_rhos)
rng = np.random.default_rng(0)
boot_f = [pooled_fidelity(lin_rhos[rng.integers(0, len(lin_rhos), len(lin_rhos))]) for _ in range(200)]
print(f'pooled F: {pooled_f:.4f} ± {np.std(boot_f):.4f} (bootstrap over runs)')


In [ ]:
rho_avg = np.mean(lin_rhos, axis=0)
mle_result = least_squares(mle_residuals, T2t(rho2T(rho_avg)), args=(compute_stokes(rho_avg),), max_nfev=3000)

# Extract the final physical density matrix
optimal_t = mle_result.x
T_optimal = t2T(optimal_t)
rho_mle = T2rho(T_optimal)

print("MLE optimization finished. Success:", mle_result.success)
assert(mle_result.success)

rho_Qobj = qutip.Qobj(rho_mle, dims=[[2,2],[2,2]])
rho_q1 = rho_Qobj.ptrace(0)
rho_q2 = rho_Qobj.ptrace(1)

extract_population_phase(rho_q1, "Q1")
extract_population_phase(rho_q2, "Q2")

In [ ]:
# import csv

# name = 'bell_state/Bell_state_0924_3'   # change per set

# with open(f'{name}.csv', 'w', newline='') as f:
#     writer = csv.writer(f)
#     header = ['tuid'] + [f'rho_{i}{j}_{part}' for i in range(4) for j in range(4) for part in ('re', 'im')]
#     writer.writerow(header)
#     for tuid, rho in zip(bell_state_ds_tuid_list, bell_state_rho_list):
#         row = [tuid]
#         for val in rho.flatten():
#             row += [val.real, val.imag]
#         writer.writerow(row)

# # NEW: confusion matrix per run, CM[measured, prepared], basis |Q1 Q2> = 00, 01, 10, 11
# with open(f'{name}_confusion.csv', 'w', newline='') as f:
#     writer = csv.writer(f)
#     writer.writerow(['tuid'] + [f'cm_{i}{j}' for i in range(4) for j in range(4)])
#     for tuid, cm in zip(bell_state_ds_tuid_list, bell_cm_list):
#         writer.writerow([tuid] + list(np.asarray(cm).ravel()))


In [ ]:
# cms = {r[0]: np.array(r[1:17], float).reshape(4, 4)
#        for r in list(csv.reader(open('bell_state/Bell_state_0924_1_confusion.csv')))[1:]}
# lin_rhos = [calculate_density_matrix(t, confusion_matrix=cms[t])[0] for t in cms]


In [ ]:
0.9764739720885693, 0.010109309961825609

#### 6.4.1 Calibration Test

In [ ]:
import snl_qblox.pycqed_randomized_benchmarking.utils as rbu
# Fresh readout confusion matrix, measured once so every run below shares it
readout_confusion_matrix = construct_confusion_matrix(
    qubit_specifier=[qubit1, qubit2], repetitions=2000,
    quantum_device=quantum_device, instrument_coordinator=ic)
print("confusion matrix:\n", np.round(readout_confusion_matrix, 3))

op_idx = 5760 + 384                     # X90 on Q1 -> iSWAP: Q0 phase depends only on AC_STARK_PHASE_Q0
base = rbu.AC_STARK_PHASE_Q0
deltas = np.array([-20, -10, 0, 10, 20], dtype=float)   # symmetric: null expected near base, sign unknown
q0_phase = []
try:
    for delta in deltas:
        rbu.AC_STARK_PHASE_Q0 = base + delta
        # must run AFTER the patch: the AC-Stark phase is baked into the correction pair here
        corr = tomography_pre_build([qubit1, qubit2], op_idx, quantum_device)
        ds = tomography_schedule([qubit1, qubit2], op_idx, repetitions=4000,
                                 instrument_coordinator=ic, quantum_device=quantum_device,
                                 correction_phase=corr)
        tuid = save_retrieve_acquisition_dataset(ds, 'State Tomography')
        rho, _ = calculate_density_matrix(load_dataset(tuid), confusion_matrix=readout_confusion_matrix)
        r = qutip.Qobj(rho, dims=[[2, 2], [2, 2]]).ptrace(0).full()     # Q0 = qubit1
        q0_phase.append(np.angle(r[0, 1]))
        print(f"delta={delta:+5.1f}  q0 correction={corr[0][0]:+8.3f}  "
              f"Q0 phase={np.degrees(q0_phase[-1]):+8.3f}  |rho01|={abs(r[0, 1]):.3f}  ({tuid})")
finally:
    rbu.AC_STARK_PHASE_Q0 = base          # restored even if a run fails or is interrupted

y = np.degrees(np.unwrap(q0_phase))
m, b = np.polyfit(deltas, y, 1)
s = np.sign(m)                                   # physics fixes |slope| = 1; take only the sign from data
b1 = np.mean(y - s * deltas)
print(f"free fit: slope = {m:+.3f}   (must be close to +1 or -1, otherwise don't trust the null)")
print(f"residuals with slope {s:+.0f}: {np.round(y - (s * deltas + b1), 2)}")
print(f"AC_STARK_PHASE_Q0 that nulls Q0: {base - b1 / s:.2f}   (free-slope fit: {base - b / m:.2f})")


In [ ]:
import snl_qblox.pycqed_randomized_benchmarking.utils as rbu
# Fresh readout confusion matrix, measured once so every run below shares it
readout_confusion_matrix = construct_confusion_matrix(
    qubit_specifier=[qubit1, qubit2], repetitions=2000,
    quantum_device=quantum_device, instrument_coordinator=ic)
print("confusion matrix:\n", np.round(readout_confusion_matrix, 3))

op_idx = 5760 + 16                      # X90 on Q0 -> iSWAP: Q1 phase depends only on AC_STARK_PHASE_Q1
base = rbu.AC_STARK_PHASE_Q1
deltas = np.array([-20, -10, 0, 10, 20], dtype=float)   # symmetric: null expected near base
q1_phase = []
try:
    for delta in deltas:
        rbu.AC_STARK_PHASE_Q1 = base + delta
        # must run AFTER the patch: the AC-Stark phase is baked into the correction pair here
        corr = tomography_pre_build([qubit1, qubit2], op_idx, quantum_device)
        ds = tomography_schedule([qubit1, qubit2], op_idx, repetitions=4000,
                                 instrument_coordinator=ic, quantum_device=quantum_device,
                                 correction_phase=corr)
        tuid = save_retrieve_acquisition_dataset(ds, 'State Tomography')
        rho, _ = calculate_density_matrix(load_dataset(tuid), confusion_matrix=readout_confusion_matrix)
        r = qutip.Qobj(rho, dims=[[2, 2], [2, 2]]).ptrace(1).full()     # Q1 = qubit2
        q1_phase.append(np.angle(r[0, 1]))
        print(f"delta={delta:+5.1f}  q1 correction={corr[0][1]:+8.3f}  "
              f"Q1 phase={np.degrees(q1_phase[-1]):+8.3f}  |rho01|={abs(r[0, 1]):.3f}  ({tuid})")
finally:
    rbu.AC_STARK_PHASE_Q1 = base          # restored even if a run fails or is interrupted

y = np.degrees(np.unwrap(q1_phase))
m, b = np.polyfit(deltas, y, 1)
s = np.sign(m)                                   # physics fixes |slope| = 1; take only the sign from data
b1 = np.mean(y - s * deltas)
print(f"free fit: slope = {m:+.3f}   (must be close to +1 or -1, otherwise don't trust the null)")
print(f"residuals with slope {s:+.0f}: {np.round(y - (s * deltas + b1), 2)}")
print(f"AC_STARK_PHASE_Q1 that nulls Q1: {base - b1 / s:.2f}   (free-slope fit: {base - b / m:.2f})")


### 6.5 Process Tomography

In [ ]:
from scipy.linalg import solve

from snl_qblox.tomography_tools import PMATRIX_PATH as file_location

with open(file_location, "rb") as file:
    PMatrix = pickle.load(file)

In [ ]:
U_iSWAP = qutip.Qobj([[1, 0, 0, 0], [0, 0, 1j, 0], [0, 1j, 0, 0], [0, 0, 0, 1]], dims=[[2, 2], [2, 2]])
U_I = qutip.tensor(qutip.qeye(2), qutip.qeye(2))

dims = [[[2, 2], [2, 2]], [[2, 2], [2, 2]]]

Ideal_iSWAP_chi_Qobj = qutip.to_chi(U_iSWAP)

Ideal_I_chi_Qobj = qutip.to_chi(U_I)

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import process_tomography_pre_build, process_tomography_schedule

In [ ]:
n_experiments_qpt = 36 * 36
qpt_experiment_idx_list = range(n_experiments_qpt)

correction_phase = process_tomography_pre_build(
    [qubit1, qubit2], 5760, qpt_experiment_idx_list, quantum_device)

In [ ]:
import xarray as xr

operation_state_idx = 5760
repetitions = 400
batch_size = 162
identity_reference = True
meas_ctrl.verbose(False)

QPT_tuid_list = []

for _ in tqdm(range(40)):  # Assuming you want to run this more than once
    if _ % 5 == 0:
        nodes.multiplexed_readout_calibration([qubit1, qubit2])
        nodes.multi_qubit_t1_and_t2([qubit1, qubit2], num_repeats=5)
    
    experiment_batches_qpt = [
        qpt_experiment_idx_list[i : i + batch_size]
        for i in range(0, n_experiments_qpt, batch_size)
    ]

    gst_ds_qpt_batches = []
    batch_start = 0
    for batch_idx, experiment_batch in enumerate(experiment_batches_qpt, start=1):
        print(f"Running batch {batch_idx}/{len(experiment_batches_qpt)} with {len(experiment_batch)} experiments")
        # correction_phase = process_tomography_pre_build(
        #     [qubit1, qubit2], operation_state_idx, experiment_batch, quantum_device)

        batch_ds = process_tomography_schedule(
            qubit_specifier=[qubit1, qubit2],
            experiment_list=experiment_batch,
            operation_state_idx=operation_state_idx,
            repetitions=repetitions,
            identity_reference=identity_reference,
            instrument_coordinator=ic,
            quantum_device=quantum_device,
            # correction_phase=correction_phase[experiment_batch],
            correction_phase=None,
        )

        if identity_reference:
            batch_len = len(experiment_batch) * 2
        else:
            batch_len = len(experiment_batch)

        batch_range = np.arange(batch_start, batch_start + batch_len)
        batch_ds = batch_ds.assign_coords(
            acq_index_0=batch_range,
            acq_index_1=batch_range,
        )
        gst_ds_qpt_batches.append(batch_ds)
        batch_start += batch_len

    if identity_reference:
            full_index_qpt = np.arange(2 * n_experiments_qpt)
    else:
        full_index_qpt = np.arange(n_experiments_qpt)
    gst_ds_qpt = None
    for batch_ds in gst_ds_qpt_batches:
        expanded_batch_ds = batch_ds.reindex(
            acq_index_0=full_index_qpt,
            acq_index_1=full_index_qpt,
        )
        if gst_ds_qpt is None:
            gst_ds_qpt = expanded_batch_ds
        else:
            gst_ds_qpt = gst_ds_qpt.combine_first(expanded_batch_ds)

    print(gst_ds_qpt.sizes)
    qpt_tuid = save_retrieve_acquisition_dataset(gst_ds_qpt, 'QPT idx ' + str(operation_state_idx))
    QPT_tuid_list.append(qpt_tuid)
    print('QPT tuid:', qpt_tuid[:15])

In [ ]:
import xarray as xr

operation_state_idx = 5760
repetitions = 400
batch_size = 162
identity_reference = True
meas_ctrl.verbose(False)

QPT_tuid_list = []

for _ in tqdm(range(40)):  # Assuming you want to run this more than once
    if _ % 5 == 0:
        nodes.multiplexed_readout_calibration([qubit1, qubit2])
        nodes.multi_qubit_t1_and_t2([qubit1, qubit2], num_repeats=5)
    
    experiment_batches_qpt = [
        qpt_experiment_idx_list[i : i + batch_size]
        for i in range(0, n_experiments_qpt, batch_size)
    ]

    gst_ds_qpt_batches = []
    batch_start = 0
    for batch_idx, experiment_batch in enumerate(experiment_batches_qpt, start=1):
        print(f"Running batch {batch_idx}/{len(experiment_batches_qpt)} with {len(experiment_batch)} experiments")
        # correction_phase = process_tomography_pre_build(
        #     [qubit1, qubit2], operation_state_idx, experiment_batch, quantum_device)

        batch_ds = process_tomography_schedule(
            qubit_specifier=[qubit1, qubit2],
            experiment_list=experiment_batch,
            operation_state_idx=operation_state_idx,
            repetitions=repetitions,
            identity_reference=identity_reference,
            instrument_coordinator=ic,
            quantum_device=quantum_device,
            # correction_phase=correction_phase[experiment_batch],
            correction_phase=None,
        )

        if identity_reference:
            batch_len = len(experiment_batch) * 2
        else:
            batch_len = len(experiment_batch)

        batch_range = np.arange(batch_start, batch_start + batch_len)
        batch_ds = batch_ds.assign_coords(
            acq_index_0=batch_range,
            acq_index_1=batch_range,
        )
        gst_ds_qpt_batches.append(batch_ds)
        batch_start += batch_len

    if identity_reference:
            full_index_qpt = np.arange(2 * n_experiments_qpt)
    else:
        full_index_qpt = np.arange(n_experiments_qpt)
    gst_ds_qpt = None
    for batch_ds in gst_ds_qpt_batches:
        expanded_batch_ds = batch_ds.reindex(
            acq_index_0=full_index_qpt,
            acq_index_1=full_index_qpt,
        )
        if gst_ds_qpt is None:
            gst_ds_qpt = expanded_batch_ds
        else:
            gst_ds_qpt = gst_ds_qpt.combine_first(expanded_batch_ds)

    print(gst_ds_qpt.sizes)
    qpt_tuid = save_retrieve_acquisition_dataset(gst_ds_qpt, 'QPT idx ' + str(operation_state_idx))
    QPT_tuid_list.append(qpt_tuid)
    print('QPT tuid:', qpt_tuid[:15])

In [ ]:
print([i[:15] for i in QPT_tuid_list])

In [ ]:
# QPT_tuid_list = ['20260715-1457', '20260715-1459', '20260715-1502', '20260715-1504', '20260715-1507',
#  '20260715-1510', '20260715-1512', '20260715-1515', '20260715-1518', '20260715-1520', '20260715-1523',
#  '20260715-1525', '20260715-1528', '20260715-1531', '20260715-1533', '20260715-1536', '20260715-1539', '20260715-1541',
#  '20260715-1544', '20260715-1546'] # iSWAP QPT data

# QPT_tuid_list = ['20260812-1112', '20260812-1115', '20260812-1118', '20260812-1120', '20260812-1123', 
# '20260812-1126', '20260812-1128', '20260812-1131', '20260812-1133', '20260812-1136', '20260812-1139', 
# '20260812-1141', '20260812-1144', '20260812-1146', '20260812-1149', '20260812-1152', '20260812-1154', 
# '20260812-1157', '20260812-1159', '20260812-1202'] # iSWAP QPT data

# QPT_tuid_list = ['20260812-1211', '20260812-1214', '20260812-1217', '20260812-1219', '20260812-1222', 
# '20260812-1224', '20260812-1227', '20260812-1230', '20260812-1232', '20260812-1235', '20260812-1237', 
# '20260812-1240', '20260812-1243', '20260812-1245', '20260812-1248', '20260812-1250', '20260812-1253', 
# '20260812-1256', '20260812-1258', '20260812-1301'] # idle QPT data

QPT_tuid_list = ['20260819-1709', '20260819-1711', '20260819-1714', '20260819-1717', '20260819-1719',
 '20260819-1722', '20260819-1725', '20260819-1727', '20260819-1730', '20260819-1733'] # iSWAP QPT data


In [ ]:
QPT_tuid_list = ['20260924-2103', '20260924-2108', '20260924-2113', '20260924-2118', '20260924-2123', '20260924-2128', '20260924-2133', '20260924-2138', '20260924-2143', '20260924-2148', '20260924-2153', '20260924-2158', '20260924-2203', '20260924-2208', '20260924-2213', '20260924-2218', '20260924-2223', '20260924-2228', '20260924-2233', '20260924-2238']

In [ ]:
import xarray as xr

operation_state_idx = 5760
repetitions = 400
batch_size = 162
identity_reference = True
meas_ctrl.verbose(False)

QPT_tuid_list = []

for _ in tqdm(range(100)):  # Assuming you want to run this more than once
    if _ % 4 == 0:
        nodes.multiplexed_readout_calibration([qubit1, qubit2])
    n_experiments_qpt = 36 * 36
    qpt_experiment_idx_list = range(n_experiments_qpt)

    experiment_batches_qpt = [
        qpt_experiment_idx_list[i : i + batch_size]
        for i in range(0, n_experiments_qpt, batch_size)
    ]

    gst_ds_qpt_batches = []
    batch_start = 0
    for batch_idx, experiment_batch in enumerate(experiment_batches_qpt, start=1):
        print(f"Running batch {batch_idx}/{len(experiment_batches_qpt)} with {len(experiment_batch)} experiments")
        # correction_phase = process_tomography_pre_build(
        #     [qubit1, qubit2], operation_state_idx, experiment_batch, quantum_device)

        batch_ds = process_tomography_schedule(
            qubit_specifier=[qubit1, qubit2],
            experiment_list=experiment_batch,
            operation_state_idx=operation_state_idx,
            repetitions=repetitions,
            identity_reference=identity_reference,
            instrument_coordinator=ic,
            quantum_device=quantum_device,
            correction_phase=correction_phase[experiment_batch],
        )

        if identity_reference:
            batch_len = len(experiment_batch) * 2
        else:
            batch_len = len(experiment_batch)

        batch_range = np.arange(batch_start, batch_start + batch_len)
        batch_ds = batch_ds.assign_coords(
            acq_index_0=batch_range,
            acq_index_1=batch_range,
        )
        gst_ds_qpt_batches.append(batch_ds)
        batch_start += batch_len

    if identity_reference:
            full_index_qpt = np.arange(2 * n_experiments_qpt)
    else:
        full_index_qpt = np.arange(n_experiments_qpt)
    gst_ds_qpt = None
    for batch_ds in gst_ds_qpt_batches:
        expanded_batch_ds = batch_ds.reindex(
            acq_index_0=full_index_qpt,
            acq_index_1=full_index_qpt,
        )
        if gst_ds_qpt is None:
            gst_ds_qpt = expanded_batch_ds
        else:
            gst_ds_qpt = gst_ds_qpt.combine_first(expanded_batch_ds)

    print(gst_ds_qpt.sizes)
    qpt_tuid = save_retrieve_acquisition_dataset(gst_ds_qpt, 'QPT idx ' + str(operation_state_idx))
    QPT_tuid_list.append(qpt_tuid)
    print('QPT tuid:', qpt_tuid[:13])

In [ ]:
p00_list = []
ref_p00_list = []

for qpt_tuid in QPT_tuid_list:
    gst_ds_qpt = load_dataset(qpt_tuid)

    if gst_ds_qpt.acq_index_0.size == 2592:
        q1_state_append = gst_ds_qpt['y0'][:,::2].values
        q2_state_append = gst_ds_qpt['y1'][:,::2].values
        reference_q1_state_append = gst_ds_qpt['y0'][:,1::2].values
        reference_q2_state_append = gst_ds_qpt['y1'][:,1::2].values

        ref_p00 = (1-reference_q1_state_append) * (1-reference_q2_state_append)
        ref_p00 = ref_p00.mean(axis=0)

    elif gst_ds_qpt.acq_index_0.size == 1296:
        q1_state_append = gst_ds_qpt['y0'].values
        q2_state_append = gst_ds_qpt['y1'].values

    p00 = (1-q1_state_append) * (1-q2_state_append)
    p00 = p00.mean(axis=0)

    p00_list.append(p00)
    ref_p00_list.append(ref_p00)

p00 = np.mean(p00_list, axis=0)
ref_p00 = np.mean(ref_p00_list, axis=0)

In [ ]:
probs = p00.reshape(6,6,6,6)

pauli_basis_measurements = lambda q, n: map_from_bloch_state_to_pauli_basis2(q, n, probs)

measurement_vector = np.array(
        [pauli_basis_measurements(np.floor(v / 16).astype(int), v % 16) for v in range(16**2)]
    )

chi_vector = solve(PMatrix, measurement_vector)
chi_matrix = chi_vector.reshape(16, 16)

if gst_ds_qpt.acq_index_0.size == 2592:
    probs = ref_p00.reshape(6,6,6,6)

    pauli_basis_measurements = lambda q, n: map_from_bloch_state_to_pauli_basis2(q, n, probs)

    measurement_vector = np.array(
            [pauli_basis_measurements(np.floor(v / 16).astype(int), v % 16) for v in range(16**2)]
        )

    ref_chi_vector = solve(PMatrix, measurement_vector)
    ref_chi_matrix = ref_chi_vector.reshape(16, 16)

In [ ]:
dims = [[[2, 2], [2, 2]], [[2, 2], [2, 2]]]

measured_chi_Qobj = qutip.Qobj(16 * chi_matrix, superrep='chi', dims=dims)
reference_chi_Qobj = qutip.Qobj(16 * ref_chi_matrix, superrep='chi', dims=dims)

corrected_PTM = get_spam_corrected_gate(qutip.to_super(measured_chi_Qobj).full(), qutip.to_super(reference_chi_Qobj).full(), p=0)
corrected_chi_Qobj = qutip.to_chi(qutip.Qobj(corrected_PTM, superrep='super', dims=dims))

In [ ]:
import itertools
from qutip import matrix_histogram

op_label = [["$I$", "$X$", "$Y$", "$Z$"] for i in range(2)]

def qpt_plot_cmap(chi, lbls_list, title='', cmap=plt.cm.bwr, figsize=(8, 6), threshold=None):
    """Like qt.qpt_plot_combined but with a user-chosen (cyclic) colormap."""
    xlabels = ["".join(p) for p in itertools.product(*lbls_list)]
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    matrix_histogram(chi, xlabels, xlabels, bar_style='abs', color_style='phase',
                     cmap=cmap, options={'threshold': threshold}, ax=ax)
    ax.set_title(title)
    return fig, ax

# your call becomes:
fig, ax = qpt_plot_cmap(corrected_chi_Qobj / 16, op_label, 'iSWAP RPE', cmap=plt.cm.bwr, figsize=(10, 8), threshold=0.0)
ax.set_zticks([0, 0.125, 0.25])
ax.view_init(elev=30, azim=-35)

ax.tick_params(axis='x', labelsize=10)   # the II, IX, IY... labels
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='z', labelsize=12)   # the height (0.0, 0.2...) numbers
plt.show()

In [ ]:
U_fit, w = closest_unitary(corrected_chi_Qobj)
ang_geom = deg(extract_angles(U_fit))
ang_fit, F_fit, res = fit_iswap_rpe_to_chi(corrected_chi_Qobj)
ang_fit_d = deg(ang_fit)

print("Closest-unitary Choi eigs:  min %.3f  max %.3f" % (w.min(), w.max()))
print("\nObservable angles (deg):")
for k in ["theta_p", "theta_s", "theta_d", "phi_zz", "theta_sum"]:
    print(f"   {k:10s} = {ang_geom[k]:8.2f}   (fit {ang_fit_d[k]:8.2f})")
print("\nIndividuals (degenerate for full iSWAP -> ignore):")
for k in ["theta_1", "theta_2", "phi_p"]:
    print(f"   {k:10s} = {ang_geom[k]:8.2f}")
print(f"\nProcess fidelity to best-fit iSWAP_RPE : {F_fit:.4f}   (infidelity {1-F_fit:.4f})")
print("NOTE: this compares to the *fitted* gate (with its phi_zz etc.), not the textbook ideal iSWAP.")

In [ ]:
U_fit

In [ ]:
target_gate = "U_fit"  # Options: "iSWAP", "I", "U_fit"
input_chi_Qobj = corrected_chi_Qobj

if target_gate == "iSWAP":
    # print("custom process and gate fidelity:", np.round(QPT_fidelity(input_chi_Qobj.full() / 16, Ideal_iSWAP_chi_Qobj.full() / 16), 4))
    print("qutip process and gate fidelity:", np.round([qutip.process_fidelity(input_chi_Qobj, U_iSWAP), qutip.average_gate_fidelity(input_chi_Qobj, U_iSWAP)], 4))
elif target_gate == "I":
    # print("custom process and gate fidelity:", np.round(QPT_fidelity(input_chi_Qobj.full() / 16, Ideal_I_chi_Qobj.full() / 16), 4))
    print("qutip process and gate fidelity:", np.round([qutip.process_fidelity(input_chi_Qobj, U_I), qutip.average_gate_fidelity(input_chi_Qobj, U_I)], 4))
elif target_gate == "U_fit":
    print("qutip process and gate fidelity:", np.round([qutip.process_fidelity(input_chi_Qobj, U_fit), qutip.average_gate_fidelity(input_chi_Qobj, U_fit)], 4))    

In [ ]:
# --- Demo 1: reconstruct from the notebook's chi_obj (a FULL iSWAP) ---
U_rec = chi_to_unitary(corrected_chi_Qobj)
print('Reconstructed U from chi_obj:')
print(np.round(U_rec.full(), 3))
print('\nAngles (deg):')
for k, v in extract_angles(U_rec).items():
    print(f'  {k:8s} = {v:8.2f}' + ('   <- degenerate at full iSWAP' if np.isnan(v) else ''))

## 6.6 Simultaneous RB

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import simultaneous_randomized_benchmarking_schedule

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of Clifford gates")
length.batched = True
length_setpoints = np.arange(0, 110, 1)

seed = ManualParameter(name="seed", unit="", label="Seeds")
seed.batched = True
seed_setpoints = np.random.randint(0, 2**31 - 1, size=30, dtype=np.int32)

rb_sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    "seeds": seed,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=simultaneous_randomized_benchmarking_schedule,
    schedule_kwargs=rb_sched_kwargs,
    real_imag=False,
    batched=True,
    num_channels=2,
    max_batch_size=length_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

meas_ctrl.settables([length, seed])
meas_ctrl.setpoints_grid([length_setpoints, seed_setpoints])

quantum_device.cfg_sched_repetitions(200)

In [ ]:
simul_rb_ds = meas_ctrl.run("Simultaneous Randomized benchmarking")
display(simul_rb_ds.tuid)

In [ ]:
# 200 ns gate time, no DRAG calibration as it doesn't seem to improve the gate fidelity 
# '20260413-141521-271-b66905' at 0 mA
# '20260413-135801-681-3cb805', at Kerr-free point, -0.72 mA
rb_simple_fit(load_dataset(simul_rb_ds.tuid))

## 6.7 Multi Qubit T1 and T2

In [ ]:
nodes.multi_qubit_t1_and_t2([qubit1, qubit2], num_repeats=1)

In [ ]:
_, ([q1_t1,q1_t2,q1_t2_ram,q1_t2_ram_freq], [q2_t1,q2_t2,q2_t2_ram,q2_t2_ram_freq]) = \
    nodes.multi_qubit_t1_and_t2(
    [qubit1, qubit2],
    tau_setpoints=np.hstack((np.arange(0, 40, 2), np.arange(40, 120, 4))) * 1e-6,
    cases=(1, 2, 3), num_repeats=20,
    artificial_detuning=100e3, repetitions=200,
)

In [ ]:
qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

In [ ]:
tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
tau.batched = True
tau.batch_size = 600

case = ManualParameter(name="case", label="Case")

t1_and_t2_sched_kwargs = {"qubit_specifier": [qubit1, qubit2], "artificial_detuning": 100e3, 
    "times": tau, "case": case, "acq_protocol": "ThresholdedAcquisition"}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=multi_qubit_t1_and_t2_TWPA,
    schedule_kwargs=t1_and_t2_sched_kwargs,
    real_imag=True,
    batched=True,
)
meas_ctrl.gettables(gettable)

In [ ]:
tau_setpoints = np.hstack((np.arange(0, 40, 2), np.arange(40, 120, 4))) * 1e-6
case_setpoints = np.tile([1,2,3], 20)

meas_ctrl.settables([tau, case])
meas_ctrl.setpoints_grid([tau_setpoints, case_setpoints])

quantum_device.cfg_sched_repetitions(200)
multi_t1_and_t2_ds = meas_ctrl.run("Multi Qubit T1 and T2", soft_avg=1)
multi_t1_and_t2_ds.tuid

In [ ]:
[q1_t1, q1_t2, q1_t2_ram, q1_t2_ram_freq], [q2_t1, q2_t2, q2_t2_ram, q2_t2_ram_freq] = t1_and_t2_fit(multi_t1_and_t2_ds.tuid, r_squared=0.8)

In [ ]:
multi_t1_and_t2_tuid_list = []
meas_ctrl.verbose(False)

qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

for _ in tqdm(range(50)):  # Assuming you want to run this more than once
    nodes.multiplexed_readout_calibration([qubit1, qubit2]);
    # T1 and T2 measurement
    tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
    tau.batched = True
    tau.batch_size = 600

    case = ManualParameter(name="case", label="Case")

    t1_and_t2_sched_kwargs = {"qubit_specifier": [qubit1, qubit2], "artificial_detuning": 400e3, 
        "times": tau, "case": case, "acq_protocol": "ThresholdedAcquisition"}

    gettable = ScheduleGettable(
        quantum_device,
        schedule_function=multi_qubit_t1_and_t2_TWPA,
        schedule_kwargs=t1_and_t2_sched_kwargs,
        real_imag=True,
        batched=True,
    )
    meas_ctrl.gettables(gettable)

    tau_setpoints = np.hstack((np.arange(0, 40, 2), np.arange(40, 200, 4))) * 1e-6
    case_setpoints = np.tile([1,2,3], 60)

    meas_ctrl.settables([tau, case])
    meas_ctrl.setpoints_grid([tau_setpoints, case_setpoints])

    quantum_device.cfg_sched_repetitions(200)
    multi_t1_and_t2_ds = meas_ctrl.run("Multi Qubit T1 and T2", soft_avg=1)
    multi_t1_and_t2_tuid_list.append(multi_t1_and_t2_ds.tuid)

In [ ]:
simultaneous_rb_tuid = ['20260720-115929', '20260720-121141', '20260720-123338', '20260720-124550', '20260720-130424', 
'20260720-133733', '20260720-163732', '20260720-220233', '20260720-222321', '20260720-223623',
'20260720-225217', '20260720-231902', '20260720-234245']

current_list = [dataset_current(ds) for ds in simultaneous_rb_tuid]

In [ ]:
# np.savetxt(get_datadir() + '/multi_t1t2_curr_0721_tuid_arr.txt', multi_t1_and_t2_tuid_list, fmt='%s')

In [ ]:
tuid_list = ['20260918-163826-762-c3909f',
 '20260918-163956-975-1ddcd5',
 '20260918-164127-112-90b643',
 '20260918-164257-047-edde37',
 '20260918-164427-107-866037',
 '20260918-164556-941-bb1a77',
 '20260918-164727-107-7ed641',
 '20260918-164856-992-ff24a7',
 '20260918-165027-142-01625f',
 '20260918-165157-103-93b431',
 '20260918-165327-247-e0ea75',
 '20260918-165457-117-ca4b2a',
 '20260918-165626-729-322510',
 '20260918-165756-477-ee2abe',
 '20260918-165926-135-561ab2',
 '20260918-170055-842-b02193',
 '20260918-170225-638-bc428c',
 '20260918-170355-517-e970a0',
 '20260918-170525-451-40774b',
 '20260918-170655-432-0c4858',
 '20260918-170825-143-a6664e']

current_list = [dataset_current(ds) for ds in tuid_list]

In [ ]:
multi_t1_and_t2_tuid_list = []
GS210_1.on()
meas_ctrl.verbose(False)

qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

for _ in tqdm(range(20)):  # Assuming you want to run this more than once
    for tuid in tuid_list:
        GS210_1.current(dataset_current(tuid) * 1e-3)
        set_from_snapshot(tuid)

        # RO Calibrate
        nodes.multiplexed_readout_calibration([qubit1, qubit2], num_points=3000)
        for qubit in [qubit1, qubit2]:
            nodes.rabi_calibration(qubit)

        # T1 and T2 measurement
        tau = ManualParameter(name="tau_delay", unit="s", label="Delay")
        tau.batched = True
        tau.batch_size = 600
    
        case = ManualParameter(name="case", label="Case")
    
        t1_and_t2_sched_kwargs = {"qubit_specifier": [qubit1, qubit2], "artificial_detuning": 200e3, 
            "times": tau, "case": case, "acq_protocol": "ThresholdedAcquisition"}
    
        gettable = ScheduleGettable(
            quantum_device,
            schedule_function=multi_qubit_t1_and_t2_TWPA,
            schedule_kwargs=t1_and_t2_sched_kwargs,
            real_imag=True,
            batched=True,
        )
        meas_ctrl.gettables(gettable)
    
        tau_setpoints = np.hstack((np.arange(0, 40, 2), np.arange(40, 120, 4))) * 1e-6
        case_setpoints = np.tile([1,2], 20)
    
        meas_ctrl.settables([tau, case])
        meas_ctrl.setpoints_grid([tau_setpoints, case_setpoints])
    
        quantum_device.cfg_sched_repetitions(200)
        multi_t1_and_t2_ds = meas_ctrl.run("Multi Qubit T1 and T2", soft_avg=1)
        multi_t1_and_t2_tuid_list.append(multi_t1_and_t2_ds.tuid)

In [ ]:
# np.savetxt(get_datadir() + '/multi_t1t2_curr075_0919_tuid_arr.txt', multi_t1_and_t2_tuid_list, fmt='%s')

# 7. Robust Phase Estimation

## 7.d Phase Sum

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import iswap_RPE_d, iswap_RPE_f, compute_iswap_t1

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of iSWAP")
length.batched = True
length_setpoints = np.arange(1, 31, 1)

wait_time = ManualParameter(name="wait_time", unit="s", label="Wait time")
wait_time_setpoints = np.arange(0, 111, 10) / 1e9

RPE_sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    # "wait_time": 0,
    "sqrt_iswap_duration" : np.round(qubit1_qubit2.iswap.pulse_duration() / 2, 9),
    "acq_protocol": "ThresholdedAcquisition",
    "quantum_device": quantum_device,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=iswap_RPE_d,
    schedule_kwargs=RPE_sched_kwargs,
    batched=True,
    num_channels=2,
    max_batch_size=length_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

meas_ctrl.settables(length)
meas_ctrl.setpoints(length_setpoints)

# meas_ctrl.settables([length, wait_time])
# meas_ctrl.setpoints_grid([length_setpoints, wait_time_setpoints])

# meas_ctrl.settables([length, SGS100A_1.phase])
# meas_ctrl.setpoints_grid([length_setpoints, np.linspace(0, 180, 31)])

quantum_device.cfg_sched_repetitions(200)

In [ ]:
RPE_d_ds = meas_ctrl.run("RPE_d", soft_avg=1)
display(RPE_d_ds.tuid)
# 20260709-154204 sweeping pump phase

In [ ]:
sched = iswap_RPE_d([qubit1, qubit2], sqrt_iswap_duration=np.round(qubit1_qubit2.iswap.pulse_duration() / 2, 9), lengths=[4], quantum_device=quantum_device)
np.array(compute_iswap_t1(sched, quantum_device))*1e9

In [ ]:
x0_data = np.unique(RPE_d_ds.x0)
y0_data = RPE_d_ds.y0.values
y1_data = RPE_d_ds.y1.values

guess = decay_osci_model.guess(P2Z(y0_data), t=x0_data)
guess['tau'].value = 15
guess['phase'].value = 0
guess['amplitude'].value = 1
guess['offset'].value = 0
fit_result = decay_osci_model.fit(P2Z(y0_data), params=guess, t=x0_data)

fit_result.plot_fit()
print(f'θ_s = {round(np.rad2deg(fit_result.params["frequency"].value * 2 * np.pi), 2)} deg')
fit_result.params

In [ ]:
np.rad2deg(-432549 * 2*np.pi * (1e-9))

## 7.f Phase Diff

In [ ]:
length = ManualParameter(name="length", unit="#", label="Number of Repetitions")
length.batched = True
length_setpoints = np.arange(1, 16, 1)

RPE_sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    "acq_protocol": "ThresholdedAcquisition",
    "quantum_device": quantum_device,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=iswap_RPE_f,
    schedule_kwargs=RPE_sched_kwargs,
    batched=True,
    num_channels=2,
    max_batch_size=length_setpoints.size,
)

meas_ctrl.gettables(gettable)
meas_ctrl.verbose(True)

meas_ctrl.settables(length)
meas_ctrl.setpoints(length_setpoints)

# meas_ctrl.settables([length, SGS100A_1.phase])
# meas_ctrl.setpoints_grid([length_setpoints, np.linspace(0, 180, 31)])

quantum_device.cfg_sched_repetitions(200)

In [ ]:
RPE_f_ds = meas_ctrl.run("RPE_f", soft_avg=1)
display(RPE_f_ds.tuid)
SGS100A_1.phase(0)
# 20260709-153914 sweeping pump phase

In [ ]:
# meas_ctrl.verbose(False)

# for t in tqdm(np.arange(0, 161, 20) / 1e9):
#     RPE_sched_kwargs["wait_time"] = t

#     gettable = ScheduleGettable(
#         quantum_device,
#         schedule_function=iswap_RPE_f,
#         schedule_kwargs=RPE_sched_kwargs,
#         batched=True,
#         num_channels=2,
#         max_batch_size=length_setpoints.size,
#     )

#     meas_ctrl.gettables(gettable)
#     meas_ctrl.settables([length, SGS100A_1.phase])
#     meas_ctrl.setpoints_grid([length_setpoints, np.linspace(0, 180, 31)])

#     quantum_device.cfg_sched_repetitions(200)

#     RPE_f_ds = meas_ctrl.run("RPE_f", soft_avg=1)
#     display(RPE_f_ds.tuid)

# meas_ctrl.verbose(True)

In [ ]:
# plt.plot(length_setpoints, P2Z(RPE_f_ds.y0.values))
# plt.plot(length_setpoints, np.cos(length_setpoints * np.deg2rad(35)) * np.exp(-length_setpoints / 12), '--')

In [ ]:
wait_time_list = np.array([0, 20, 40, 60, 80, 100, 120, 140, 160])
phase_step_list = np.array([-3.95, -15.45, -27.00, -38.30, -49.80, -61.20, -72.75, -84.05, -5.55-90])
phase_step_list = phase_step_list - phase_step_list[0]

time_dep_phase = (258688310 - (4765653542 - 4507760421)) * 1e-9 * 360

In [ ]:
round(time_dep_phase*150, 2)

In [ ]:
plt.plot(wait_time_list, -phase_step_list/2, 'o-')
plt.plot(wait_time_list, wait_time_list * time_dep_phase)

## 7.sum new schedule

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import iswap_RPE_theta_sum

length = ManualParameter(name="length", unit="#", label="Number of iSWAP pairs")
length.batched = True
length_setpoints = np.arange(1, 26, 1)

RPE_sum_sched_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    "acq_protocol": "ThresholdedAcquisition",
    "quantum_device": quantum_device,
}

gettable = ScheduleGettable(
    quantum_device,
    schedule_function=iswap_RPE_theta_sum,
    schedule_kwargs=RPE_sum_sched_kwargs,
    batched=True,
    num_channels=2,
    max_batch_size=length_setpoints.size,
)
meas_ctrl.gettables(gettable)
meas_ctrl.settables(length)
meas_ctrl.setpoints(length_setpoints)
quantum_device.cfg_sched_repetitions(200)

RPE_sum_ds = meas_ctrl.run("RPE_theta_sum", soft_avg=1)
display(RPE_sum_ds.tuid)


In [ ]:
x = np.asarray(RPE_sum_ds.x0.values, float)
z = P2Z(RPE_sum_ds.y0.values)

best = None
for f0 in np.linspace(0.02, 0.49, 30):
    g = decay_osci_model.guess(z, t=x)
    g['tau'].value, g['phase'].value, g['amplitude'].value, g['offset'].value = 15, 0, 1, 0
    g['frequency'].value = f0
    r = decay_osci_model.fit(z, params=g, t=x)
    if best is None or r.chisqr < best.chisqr:
        best = r

rate = best.params['frequency'].value * 360
best.plot_fit()
print(f"rate = {rate:.2f} deg/pair  ->  |theta_sum| = {180 - rate:.2f} deg   (R^2 = {best.rsquared:.3f})")


## 7.X Phase Test

In [ ]:
n_warm = 2
for op_idx in (5760 + 16, 5760 + 384):
    tomo_phase_list = tomography_pre_build(
        [qubit1, qubit2], op_idx, quantum_device, warmup_iswaps=n_warm)
    state_tg_ds = tomography_schedule(
        qubit_specifier=[qubit1, qubit2],
        operation_state_idx=op_idx,
        repetitions=4000,
        instrument_coordinator=ic,
        quantum_device=quantum_device,
        correction_phase=tomo_phase_list,
        warmup_iswaps=n_warm,
    )
    print(op_idx, save_retrieve_acquisition_dataset(state_tg_ds, f'State Tomography warmup{n_warm}'))


In [ ]:
RPE_sum_sched_kwargs["warmup_iswaps"] = 2
gettable = ScheduleGettable(
    quantum_device,
    schedule_function=iswap_RPE_theta_sum,
    schedule_kwargs=RPE_sum_sched_kwargs,
    batched=True,
    num_channels=2,
    max_batch_size=length_setpoints.size,
)
meas_ctrl.gettables(gettable)
meas_ctrl.settables(length)
meas_ctrl.setpoints(length_setpoints)
RPE_sum_ds = meas_ctrl.run("RPE_theta_sum warmup2", soft_avg=1)
display(RPE_sum_ds.tuid)


In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import (
    iswap_RPE_theta_sum, tomography_pre_build, tomography_schedule)

warm_list = [0, 2, 5, 10]

qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

# One confusion matrix for the whole sweep, so phases compare like-for-like
readout_confusion_matrix = construct_confusion_matrix(
    qubit_specifier=[qubit1, qubit2], repetitions=2000,
    quantum_device=quantum_device, instrument_coordinator=ic)
print(np.round(readout_confusion_matrix, 3))

length = ManualParameter(name="length", unit="#", label="Number of iSWAP pairs")
length.batched = True
length_setpoints = np.arange(1, 26, 1)

def fit_rpe_rate(ds, ch='y0'):
    x = np.asarray(ds.x0.values, float)
    z = P2Z(ds[ch].values)
    best = None
    for f0 in np.linspace(0.02, 0.49, 30):
        g = decay_osci_model.guess(z, t=x)
        g['tau'].value, g['phase'].value, g['amplitude'].value, g['offset'].value = 15, 0, 1, 0
        g['frequency'].value = f0
        r = decay_osci_model.fit(z, params=g, t=x)
        if best is None or r.chisqr < best.chisqr:
            best = r
    return best

warm_results = []
for n_warm in tqdm(warm_list):
    row = {'n_warm': n_warm}

    # ---- state tomography: 5760+384 (excitation ends on Q0), 5760+16 (ends on Q1) ----
    for op_idx, key, el in ((5760 + 384, 'q0', (0, 2)), (5760 + 16, 'q1', (0, 1))):
        phases = tomography_pre_build([qubit1, qubit2], op_idx, quantum_device,
                                      warmup_iswaps=n_warm)
        raw = tomography_schedule(
            qubit_specifier=[qubit1, qubit2],
            operation_state_idx=op_idx,
            repetitions=4000,
            instrument_coordinator=ic,
            quantum_device=quantum_device,
            correction_phase=phases,
            warmup_iswaps=n_warm,
        )
        tuid = save_retrieve_acquisition_dataset(raw, f'State Tomography {op_idx} warmup{n_warm}')
        rho, _ = calculate_density_matrix(load_dataset(tuid), confusion_matrix=readout_confusion_matrix)
        c = np.asarray(rho)[el]
        row[f'tomo_{key}_tuid'] = tuid
        row[f'tomo_{key}_phase'] = np.degrees(np.angle(c))
        row[f'tomo_{key}_coh'] = abs(c)

    # ---- RPE_theta_sum ----
    RPE_sum_sched_kwargs = {
        "qubit_specifier": [qubit1, qubit2],
        "lengths": length,
        "acq_protocol": "ThresholdedAcquisition",
        "quantum_device": quantum_device,
        "warmup_iswaps": n_warm,
    }
    gettable = ScheduleGettable(
        quantum_device,
        schedule_function=iswap_RPE_theta_sum,
        schedule_kwargs=RPE_sum_sched_kwargs,
        batched=True,
        num_channels=2,
        max_batch_size=length_setpoints.size,
    )
    meas_ctrl.gettables(gettable)
    meas_ctrl.settables(length)
    meas_ctrl.setpoints(length_setpoints)
    meas_ctrl.verbose(False)
    quantum_device.cfg_sched_repetitions(200)
    rpe_ds = meas_ctrl.run(f"RPE_theta_sum warmup{n_warm}", soft_avg=1)
    fit = fit_rpe_rate(rpe_ds)
    p = fit.params
    row['rpe_tuid'] = rpe_ds.tuid
    row['rpe_rate'] = p['frequency'].value * 360
    row['rpe_rate_err'] = (p['frequency'].stderr or np.nan) * 360
    row['rpe_phase0'] = np.degrees(p['phase'].value)
    row['rpe_tau'] = p['tau'].value
    row['rpe_r2'] = fit.rsquared

    warm_results.append(row)

meas_ctrl.verbose(True)

print(f"{'warm':>4} | {'tomo Q0':>8} {'tomo Q1':>8} {'-(sum)':>8} {'diff':>8} | "
      f"{'RPE rate':>9} {'|th_sum|':>8} {'phase0':>7} {'tau':>5} {'R2':>5}")
for r in warm_results:
    s = r['tomo_q0_phase'] + r['tomo_q1_phase']
    d = r['tomo_q0_phase'] - r['tomo_q1_phase']
    print(f"{r['n_warm']:>4} | {r['tomo_q0_phase']:8.2f} {r['tomo_q1_phase']:8.2f} {-s:8.2f} {d:8.2f} | "
          f"{r['rpe_rate']:6.2f}+-{r['rpe_rate_err']:.2f} {180 - r['rpe_rate']:8.2f} "
          f"{r['rpe_phase0']:7.1f} {r['rpe_tau']:5.1f} {r['rpe_r2']:5.3f}")


In [ ]:
import datetime as dt
import matplotlib.dates as mdates
from scipy.optimize import curve_fit
from snl_qblox.pycqed_randomized_benchmarking.utils import iswap_RPE_d

monitor_minutes = 30
ram_tau = np.arange(0, 10e-6, 0.2e-6)
ram_det = np.linspace(-300e3, 300e3, 7)

def _osc(m, a, tau, w, ph, c):
    return a * np.exp(-m / tau) * np.cos(np.deg2rad(w * m + ph)) + c

def fit_rate(x, z):
    """Multi-start decaying-cosine fit; returns (rate_deg_per_step, err)."""
    best = None
    for w0 in np.linspace(1, 179, 60):
        for ph0 in (-90, 0, 90):
            try:
                p, cov = curve_fit(_osc, x, z, p0=[0.9, 15, w0, ph0, 0.1],
                                   bounds=([0, 1, 0, -360, -1], [2, 300, 180, 360, 1]), maxfev=20000)
            except Exception:
                continue
            cost = np.sum((_osc(x, *p) - z) ** 2)
            if best is None or cost < best[0]:
                best = (cost, p, cov)
    _, p, cov = best
    return p[2], np.sqrt(cov[2, 2])

length = ManualParameter(name="length", unit="#", label="Number of iSWAP")
length.batched = True
rpe_d_setpoints = np.arange(1, 31, 1)
rpe_d_kwargs = {
    "qubit_specifier": [qubit1, qubit2],
    "lengths": length,
    "sqrt_iswap_duration": np.round(qubit1_qubit2.iswap.pulse_duration() / 2, 9),
    "acq_protocol": "ThresholdedAcquisition",
    "quantum_device": quantum_device,
}

f01_ref = {q.name: q.clock_freqs.f01() for q in (qubit1, qubit2)}
drift_log = []
t_end = dt.datetime.now() + dt.timedelta(minutes=monitor_minutes)
meas_ctrl.verbose(False)

while dt.datetime.now() < t_end:
    row = {}

    # ---- RPE_d: |theta_sum + phi_zz| per iSWAP ----
    qubit1.measure.acq_channel(0)
    qubit2.measure.acq_channel(1)
    gettable = ScheduleGettable(quantum_device, schedule_function=iswap_RPE_d,
                                schedule_kwargs=rpe_d_kwargs, batched=True,
                                num_channels=2, max_batch_size=rpe_d_setpoints.size)
    meas_ctrl.gettables(gettable)
    meas_ctrl.settables(length)
    meas_ctrl.setpoints(rpe_d_setpoints)
    quantum_device.cfg_sched_repetitions(200)
    row['t_rpe'] = dt.datetime.now()
    ds = meas_ctrl.run("RPE_d drift", soft_avg=1)
    x = np.asarray(ds.x0.values, float)
    r0, e0 = fit_rate(x, P2Z(ds.y0.values))
    r1, e1 = fit_rate(x, P2Z(ds.y1.values))
    row['rpe_tuid'] = ds.tuid
    row['rpe_rate'] = (r0 + r1) / 2
    row['rpe_err'] = 0.5 * np.hypot(e0, e1)

    # ---- Ramsey chevron on each qubit, frames NOT updated ----
    for q in (qubit1, qubit2):
        motzoi = q.rxy.motzoi()
        try:
            row[f't_{q.name}'] = dt.datetime.now()
            ok, f01 = nodes.ramsey_chevron_calibration(
                q, tau_setpoints=ram_tau, detuning_setpoints=ram_det, update=False)
        finally:
            q.rxy.motzoi(motzoi)
        row[f'df_{q.name}'] = (f01 - f01_ref[q.name]) if ok else np.nan

    drift_log.append(row)
    print(f"{row['t_rpe']:%H:%M:%S}  RPE_d {row['rpe_rate']:6.2f} +- {row['rpe_err']:.2f} deg   "
          f"df q1 {row['df_qubit1']/1e3:+7.1f} kHz   df q2 {row['df_qubit2']/1e3:+7.1f} kHz")

meas_ctrl.verbose(True)

# ---- plots over clock time ----
t_rpe = [r['t_rpe'] for r in drift_log]
rate = np.array([r['rpe_rate'] for r in drift_log])
rate_err = np.array([r['rpe_err'] for r in drift_log])
df1 = np.array([r['df_qubit1'] for r in drift_log]) / 1e3
df2 = np.array([r['df_qubit2'] for r in drift_log]) / 1e3
dsum = df1 + df2

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
a1.errorbar(t_rpe, rate, yerr=rate_err, fmt='o', label='RPE_d rate')
a1.set_ylabel(r'$|\theta_{sum}+\phi_{zz}|$ per iSWAP [deg]')
a1.grid(); a1.legend()
a2.plot([r['t_qubit1'] for r in drift_log], df1, 'o', label='qubit1')
a2.plot([r['t_qubit2'] for r in drift_log], df2, 's', label='qubit2')
a2.plot(t_rpe, dsum, '^', label='sum')
a2.set_ylabel(r'$f_{01} - f_{01}^{ref}$ [kHz]')
a2.grid(); a2.legend()
a2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
a2.set_xlabel('Clock time')
plt.tight_layout(); plt.show()

# ---- does the RPE_d drift follow the frequency drift? ----
ok = np.isfinite(dsum)
if ok.sum() >= 3:
    slope, icpt = np.polyfit(dsum[ok], rate[ok], 1)
    r = np.corrcoef(dsum[ok], rate[ok])[0, 1]
    print(f"RPE_d rate vs (df1+df2): slope {slope:+.3f} deg/kHz, r = {r:+.2f}   "
          f"(frequency-jitter prediction: |slope| ~ 0.23 deg/kHz)")


In [ ]:
from qcodes.instrument.base import Instrument
import importlib
import snl_qblox.helpers as helpers
import snl_qblox.calibration_nodes as calibration_nodes
import snl_qblox.pycqed_randomized_benchmarking.utils as rb_utils
importlib.reload(rb_utils); importlib.reload(helpers); importlib.reload(calibration_nodes)
from snl_qblox.pycqed_randomized_benchmarking.utils import *
from snl_qblox.calibration_nodes import CalibrationNodes

_ISWAP_PARS = ("pulse_frequency", "pulse_amp", "pulse_duration")
_old = Instrument._all_instruments.get("qubit1_qubit2")
_keep = {p: _old.iswap.parameters[p]() for p in _ISWAP_PARS} if _old is not None else {}
if _old is not None:
    try:
        quantum_device.remove_edge("qubit1_qubit2")
    except KeyError:
        pass
    _old.close()

qubit1_qubit2 = CompositeiSWAPEdge(parent_element_name=qubit1.name, child_element_name=qubit2.name)
quantum_device.add_edge(qubit1_qubit2)
for p, v in _keep.items():
    qubit1_qubit2.iswap.parameters[p](v)
qubit1_qubit2.iswap.ac_stark_phase_q1(98.5)
qubit1_qubit2.iswap.ac_stark_phase_q2(-106.5)
print("restored:", _keep, "| stark:", qubit1_qubit2.iswap.ac_stark_phase_q1(), qubit1_qubit2.iswap.ac_stark_phase_q2())

nodes = CalibrationNodes(quantum_device, meas_ctrl, ACQ_DELAY, instrument_coordinator=ic)


In [ ]:
qubit1_qubit2.iswap.pulse_frequency(258671197)
qubit1_qubit2.iswap.pulse_amp(9)
qubit1_qubit2.iswap.pulse_duration(491e-9)

print({p: qubit1_qubit2.iswap.parameters[p]() for p in
       ("pulse_frequency", "pulse_amp", "pulse_duration", "ac_stark_phase_q1", "ac_stark_phase_q2")})
print("generator:", SGS100A_1.frequency(), SGS100A_1.power(), SGS100A_1.phase())


In [ ]:
importlib.reload(calibration_nodes)
from snl_qblox.calibration_nodes import CalibrationNodes
nodes = CalibrationNodes(quantum_device, meas_ctrl, ACQ_DELAY, instrument_coordinator=ic)

ok, stark_log = nodes.run_stark_phase_calibration([qubit1, qubit2], rounds=3)

In [ ]:
ok, chk = nodes.stark_phase_sum_calibration([qubit1, qubit2], update=False)
print(chk['residual'], chk['residual_err'])


# 8. Gate Set Tomography

In [ ]:
from snl_qblox.pycqed_randomized_benchmarking.utils import GST_schedule, GST_pre_build

### 8.1 Single Qubit

In [ ]:
from pygsti.modelpacks import smq1Q_XYI

# 1) get the target Model
mdl_ideal = smq1Q_XYI.target_model()

# 2) get the building blocks needed to specify which circuits are needed
prep_fiducials, meas_fiducials = smq1Q_XYI.prep_fiducials(), smq1Q_XYI.meas_fiducials()
germs = smq1Q_XYI.germs()
maxLengths = [1,2,4] # roughly gives the length of the sequences used by GST

# 3) generate "fake" data from a depolarized version of mdl_ideal
mdl_true = mdl_ideal.depolarize(op_noise=0.01, spam_noise=0.001)
listOfExperiments = pygsti.circuits.create_lsgst_circuits(
    mdl_ideal, prep_fiducials, meas_fiducials, germs, maxLengths)

In [ ]:
qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

gst_ds = GST_schedule(qubit_specifier=[qubit1], experiment_list=listOfExperiments, repetitions=400)
gst_tuid = save_retrieve_acquisition_dataset(gst_ds, 'Q1 GST')
print(gst_tuid)

In [ ]:
gst_ds = load_dataset(gst_tuid)
dataset_txt = gst_ds_to_pygsti_dataset_txt(gst_ds, listOfExperiments, mode="auto")
ds = ds_from_dataset_txt(dataset_txt)

In [ ]:
# #Run GST
results = pygsti.run_stdpractice_gst(ds, mdl_ideal, prep_fiducials, meas_fiducials, 
                                    germs, maxLengths, modes="full TP,Target", verbosity=1)

mdl_estimate = results.estimates['full TP'].models['stdgaugeopt']
print("2DeltaLogL(estimate, data): ", pygsti.tools.two_delta_logl(mdl_estimate, ds))
print("2DeltaLogL(true, data): ", pygsti.tools.two_delta_logl(mdl_true, ds))
print("2DeltaLogL(ideal, data): ", pygsti.tools.two_delta_logl(mdl_ideal, ds))

In [ ]:
# 4) Build and save HTML report
report_dir_1q_xyi = f"GST_reports/{gst_tuid}_Q1_XYI"

pygsti.report.construct_standard_report(
    results,
    title='Q1 GST Report',
    verbosity=1,
).write_html(report_dir_1q_xyi, auto_open=False, verbosity=1)

print('Report written to:', report_dir_1q_xyi)

In [ ]:
requested_raw = list(listOfExperiments)
dataset_ckts = list(ds.keys())
dataset_set = set(dataset_ckts)

missing_raw = [c for c in requested_raw if c not in dataset_set]

requested_completed = mdl_ideal.complete_circuits(requested_raw)

In [ ]:
# comp_sched = device_compiler.compile(pygsti_circuit_to_operation(qubit_names=["qubit1"], rxy_duration=200e-9, operation_buffer_time=0, pygsti_circuit=test_circuit))
# comp_sched.plot_pulse_diagram(plot_backend="plotly")

In [ ]:
qubit1.rxy.motzoi(0)
qubit2.rxy.motzoi(0)

roadmap = [
    ('ALLXY Before',      'allxy_calibration',               {}),
    ('RB Before',         'randomized_benchmarking',         {}),
    ('DRAG',              'drag_calibration',                {}),
    ('ALLXY Before',      'allxy_calibration',               {}),
    ('RB After',          'randomized_benchmarking',         {}),
]

log = nodes.run_roadmap(roadmap, [qubit1, qubit2], stop_on_fail=False)

### 8.2 Two-Qubit iSWAP

In [ ]:
import pygsti

from pygsti.algorithms.fiducialselection import find_fiducials
from pygsti.algorithms.germselection import find_germs
from pygsti.algorithms import germselection as gsel

from pygsti.modelpacks import smq2Q_XYICNOT

In [ ]:
custom_iswap_unitary = iSWAP_RPE((np.pi/2, 0, 0, 0, np.deg2rad(-80.75))).full()
# custom_iswap_unitary = iSWAP_RPE((0, 0, 0, 0, np.deg2rad(-80.75))).full()

pspec_iswap_rpe = pygsti.processors.QubitProcessorSpec(
    num_qubits=2,
    gate_names=["Gi", "Gxpi2", "Gypi2", "Giswap"],
    nonstd_gate_unitaries={"Giswap": custom_iswap_unitary},
    availability={
        "Gi": [(0,), (1,)],
        "Gxpi2": [(0,), (1,)],
        "Gypi2": [(0,), (1,)],
        "Giswap": [(0, 1)],
    },
)

print("Qubit labels are", pspec_iswap_rpe.qubit_labels)
print("Gi gates on qubits:", pspec_iswap_rpe.resolved_availability("Gi"))
print("X(pi/2) gates on qubits:", pspec_iswap_rpe.resolved_availability("Gxpi2"))
print("custom iSWAP_RPE gates on qubits:", pspec_iswap_rpe.resolved_availability("Giswap"))

iswap_mdl_rpe = pygsti.models.create_explicit_model(
    pspec_iswap_rpe,
    simulator="matrix",
    ideal_gate_type="full",
)
print("Simulator:", type(iswap_mdl_rpe.sim).__name__)
print("Model parameters:", iswap_mdl_rpe.num_params)

In [ ]:
# Workaround for a pyGSTi 0.9.14.3 bug: set_all_parameterizations('CPTP'/'CPTPLND') always
# crashes for a model whose POVM is a ComputationalBasisPOVM (pyGSTi's own default, i.e. what
# create_explicit_model always produces) with:
#   ValueError: Could not convert POVM to to type(s): ('CPTPLND',)
#   {'CPTPLND': "local variable 'base_items' referenced before assignment"}
# Root cause: in pygsti/modelmembers/povms/__init__.py, povms.convert()'s Lindblad-conversion
# branch has a "special easy case" for ComputationalBasisPOVM that sets base_povm but never
# sets base_items, while code right after unconditionally does: for item in base_items: ...
# This is unrelated to the custom gate or its parameters -- reproduces identically with
# pyGSTi's own built-in Giswap model too.
#
# Fix: swap the POVM to an equivalent UnconstrainedPOVM (verified bit-for-bit identical
# measurement effects) so povms.convert() takes its other, working branch. Verified this has
# NO effect on 'full'/'full TP'/'TP'/'Target' modes (those paths never special-case
# ComputationalBasisPOVM) -- it only fixes the previously-broken CPTP/CPTPLND path. Safe to run
# regardless of which modes you pass to run_stdpractice_gst.
from pygsti.modelmembers.povms import UnconstrainedPOVM

_old_povm = iswap_mdl_rpe.povms["Mdefault"]
iswap_mdl_rpe.povms["Mdefault"] = UnconstrainedPOVM(
    dict(_old_povm.items()), _old_povm.evotype, _old_povm.state_space
)
print("POVM type is now:", type(iswap_mdl_rpe.povms["Mdefault"]).__name__)


In [ ]:
# iSWAP rpe = (np.pi/2, 0, 0, 0, np.deg2rad(-80.75))
generated_germs_rpe = [
    pygsti.circuits.Circuit('Gi:0@(0,1)'),
    pygsti.circuits.Circuit('Gi:1@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:1@(0,1)'),
    pygsti.circuits.Circuit('Gypi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gypi2:1@(0,1)'),
    pygsti.circuits.Circuit('Giswap:0:1@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Gxpi2:1Gypi2:0Gypi2:0Gypi2:1Gxpi2:1@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Gxpi2:0Gxpi2:1Gypi2:1Gxpi2:1@(0,1)'),
    pygsti.circuits.Circuit('Gi:0Gxpi2:0Giswap:0:1Giswap:0:1Giswap:0:1Gxpi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Gypi2:0Giswap:0:1Giswap:0:1Giswap:0:1Gypi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:1Gxpi2:1Giswap:0:1Giswap:0:1Giswap:0:1Gypi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Gxpi2:0Giswap:0:1Gypi2:1Giswap:0:1Gypi2:1@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:1Gypi2:0Giswap:0:1Giswap:0:1Gypi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Giswap:0:1Giswap:0:1Gypi2:0Gypi2:1Gypi2:1@(0,1)'),
    pygsti.circuits.Circuit('Gi:0Gypi2:0Gypi2:0Giswap:0:1Gxpi2:1@(0,1)'),
    pygsti.circuits.Circuit('Gi:0Gypi2:1Giswap:0:1Gxpi2:0Gxpi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Gxpi2:1Gxpi2:1Gypi2:0Giswap:0:1Giswap:0:1@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:1Giswap:0:1Giswap:0:1Giswap:0:1Gypi2:0Gypi2:0@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Giswap:0:1Giswap:0:1Gypi2:1Giswap:0:1Gxpi2:1@(0,1)'),
    pygsti.circuits.Circuit('Gxpi2:0Gxpi2:1Gypi2:0Gypi2:0@(0,1)'),
]

# iSWAP rpe = (0, 0, 0, 0, np.deg2rad(-80.75))
# generated_germs_rpe = [
#     pygsti.circuits.Circuit('Gi:0@(0,1)'),
#     pygsti.circuits.Circuit('Gi:1@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:0@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:1@(0,1)'),
#     pygsti.circuits.Circuit('Gypi2:0@(0,1)'),
#     pygsti.circuits.Circuit('Gypi2:1@(0,1)'),
#     pygsti.circuits.Circuit('Giswap:0:1@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:0Gxpi2:1Gypi2:0Gypi2:0Gypi2:1Gxpi2:1@(0,1)'),
#     pygsti.circuits.Circuit('Gi:0Gypi2:0Gypi2:0Gxpi2:0Gypi2:1Gypi2:1@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:0Gxpi2:0Gxpi2:1Gypi2:1Gypi2:1@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:1Gypi2:0Gypi2:1Giswap:0:1Gypi2:1@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:0Gypi2:1Giswap:0:1Giswap:0:1Gypi2:1@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:0Gypi2:0Giswap:0:1Giswap:0:1Gxpi2:1Gypi2:0@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:1Gxpi2:1Gypi2:0Giswap:0:1Giswap:0:1Giswap:0:1@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:1Gypi2:0Giswap:0:1Giswap:0:1Gypi2:0@(0,1)'),
#     pygsti.circuits.Circuit('Gxpi2:0Gypi2:1Gxpi2:1Giswap:0:1Giswap:0:1Gxpi2:1@(0,1)'),
# ]

print(f"Generated germs (custom iSWAP_RPE): {len(generated_germs_rpe)}")
is_complete_infl_rpe = gsel.test_germ_set_infl(iswap_mdl_rpe, generated_germs_rpe)
print(f"Generated set infl completeness: {is_complete_infl_rpe}")

# Reproducibility note: the hardcoded list above is the exact output of the real search below
# (seed=1234, randomize=False -> deterministic), captured so the ~16 min greedy search doesn't
# need to re-run every time this notebook executes. mode must be the hyphenated "single-Jac"
# (this installed pygsti's find_germs docstring claims "singleJac" but that value actually
# raises ValueError in find_germs_breadthfirst_greedy -- confirmed empirically).
# generated_germs_rpe = gsel.find_germs(
#     iswap_mdl_rpe,
#     seed=1234,
#     randomize=False,
#     num_gs_copies=1,
#     algorithm="greedy",
#     mode="single-Jac",
#     mem_limit=2 * 1024**3,
#     toss_random_frac=0.98,
#     force="singletons",
#     verbosity=2,
# )

In [ ]:
# 1) Build listOfExperiments for 2Q iSWAP model
# Fiducials + LGST/LSGST experiment design for the custom iSWAP_RPE model.
# Reusing the same smq2Q_XYICNOT fiducial set as the standard-iSWAP pipeline above, since the
# custom gate is not part of a pyGSTi standard modelpack with its own natural fiducials.
prep_fiducials_rpe = smq2Q_XYICNOT.prep_fiducials()
meas_fiducials_rpe = smq2Q_XYICNOT.meas_fiducials()
print(f"prep fiducials: {len(prep_fiducials_rpe)}")
print(f"meas fiducials: {len(meas_fiducials_rpe)}")

maxLengths_2q_iswap_rpe = [1,2,4]

listOfExperiments_2q_iswap_rpe = pygsti.circuits.create_lsgst_circuits(
    iswap_mdl_rpe, prep_fiducials_rpe, meas_fiducials_rpe, generated_germs_rpe, maxLengths_2q_iswap_rpe
)

print(f"#experiments (2Q iSWAP_RPE): {len(listOfExperiments_2q_iswap_rpe)}")


In [ ]:
# 2) Run 2Q GST experiment in batches and combine to one Dataset
import xarray as xr

qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)
meas_ctrl.verbose(False)

repetitions = 2000
batch_size = 60
n_experiments = len(listOfExperiments_2q_iswap_rpe)
experiment_batches_2q_iswap = [
    listOfExperiments_2q_iswap_rpe[i : i + batch_size]
    for i in range(0, n_experiments, batch_size)
]

GST_tuid_list = []
for _ in tqdm(range(1)):  # Assuming you want to run this more than once
    gst_ds_2q_iswap_batches = []
    batch_start = 0

    # nodes.multiplexed_readout_calibration([qubit1, qubit2])
    # nodes.multi_qubit_t1_and_t2([qubit1, qubit2], num_repeats=5)

    for batch_idx, experiment_batch in enumerate(tqdm(experiment_batches_2q_iswap), start=1):
        print(f"Running batch {batch_idx}/{len(experiment_batches_2q_iswap)} with {len(experiment_batch)} experiments")
        qst_phase_list = GST_pre_build(
            [qubit1, qubit2], experiment_batch, quantum_device)

        batch_ds = GST_schedule(
            qubit_specifier=[qubit1, qubit2],
            experiment_list=experiment_batch,
            repetitions=repetitions,
            instrument_coordinator=ic,
            quantum_device=quantum_device,
            correction_phase=qst_phase_list,
            # correction_phase=None
        )

        batch_len = len(experiment_batch)
        batch_range = np.arange(batch_start, batch_start + batch_len)
        batch_ds = batch_ds.assign_coords(
            acq_index_0=batch_range,
            acq_index_1=batch_range,
        )
        gst_ds_2q_iswap_batches.append(batch_ds)
        batch_start += batch_len

    full_index = np.arange(n_experiments)
    gst_ds_2q_iswap = None
    for batch_ds in gst_ds_2q_iswap_batches:
        expanded_batch_ds = batch_ds.reindex(
            acq_index_0=full_index,
            acq_index_1=full_index,
        )
        if gst_ds_2q_iswap is None:
            gst_ds_2q_iswap = expanded_batch_ds
        else:
            gst_ds_2q_iswap = gst_ds_2q_iswap.combine_first(expanded_batch_ds)

    print(gst_ds_2q_iswap.sizes)
    gst_tuid_2q_iswap = save_retrieve_acquisition_dataset(gst_ds_2q_iswap, 'Q1Q2 iSWAP RPE GST')
    print('2Q iSWAP GST tuid:', gst_tuid_2q_iswap)
    GST_tuid_list.append(gst_tuid_2q_iswap)
    

In [ ]:
GST_tuid_list

In [ ]:
# ['20261001-130551-045-c8859a',
#  '20261001-133756-380-029adc',
#  '20261001-141001-887-529c30',
#  '20261001-144209-231-14db15',

In [ ]:
# 20260904-151135

In [ ]:
# iSWAP GST
# ['20260909-194049-210-9f84d4',
#  '20260909-204657-876-3ea0b6',
#  '20260909-215303-003-4a7ce8',
#  '20260909-225911-601-b1ef91',
#  '20260910-000519-772-ef1f6a']

# idle reference GST
# ['20260910-030938-307-7fd102',
#  '20260910-041211-208-d543c0',
#  '20260910-051441-634-00fceb',
#  '20260910-061713-096-e30af6',
#  '20260910-071946-789-642577']

In [ ]:
# 20260819-122741
# 20260819-130049
# 20260819-140409 repetition=2000
# 20260819-154652 repetition=2000 idle reference
# gst_ds_2q_iswap = load_dataset("20260819-122741")

### 8.3 Two-Qubit XYI

In [ ]:
from pygsti.modelpacks import smq2Q_XYI

mdl_ideal_2q_xyi = smq2Q_XYI.target_model()
prep_fiducials_2q_xyi = smq2Q_XYI.prep_fiducials()
meas_fiducials_2q_xyi = smq2Q_XYI.meas_fiducials()
germs_2q_xyi = smq2Q_XYI.germs()
maxLengths_2q_xyi = [1, 2, 4]

listOfExperiments_2q_xyi = pygsti.circuits.create_lsgst_circuits(
    mdl_ideal_2q_xyi,
    prep_fiducials_2q_xyi,
    meas_fiducials_2q_xyi,
    germs_2q_xyi,
    maxLengths_2q_xyi,
 )

print(f"#experiments (2Q XYI): {len(listOfExperiments_2q_xyi)}")
print(f"prep fiducials: {len(prep_fiducials_2q_xyi)}")
print(f"meas fiducials: {len(meas_fiducials_2q_xyi)}")
print(f"germs: {len(germs_2q_xyi)}")

In [ ]:
# 2) Run 2Q XYI GST experiment in batches and combine to one Dataset
import xarray as xr

qubit1.measure.acq_channel(0)
qubit2.measure.acq_channel(1)

batch_size = 200
n_experiments_2q_xyi = len(listOfExperiments_2q_xyi)
experiment_batches_2q_xyi = [
    listOfExperiments_2q_xyi[i : i + batch_size]
    for i in range(0, n_experiments_2q_xyi, batch_size)
]

gst_ds_2q_xyi_batches = []
batch_start = 0
for batch_idx, experiment_batch in enumerate(experiment_batches_2q_xyi, start=1):
    print(f"Running batch {batch_idx}/{len(experiment_batches_2q_xyi)} with {len(experiment_batch)} experiments")
    batch_ds = GST_schedule(
        qubit_specifier=[qubit1, qubit2],
        experiment_list=experiment_batch,
        repetitions=500,
    )

    batch_len = len(experiment_batch)
    batch_range = np.arange(batch_start, batch_start + batch_len)
    batch_ds = batch_ds.assign_coords(
        acq_index_0=batch_range,
        acq_index_1=batch_range,
    )
    gst_ds_2q_xyi_batches.append(batch_ds)
    batch_start += batch_len

full_index_2q_xyi = np.arange(n_experiments_2q_xyi)
gst_ds_2q_xyi = None
for batch_ds in gst_ds_2q_xyi_batches:
    expanded_batch_ds = batch_ds.reindex(
        acq_index_0=full_index_2q_xyi,
        acq_index_1=full_index_2q_xyi,
    )
    if gst_ds_2q_xyi is None:
        gst_ds_2q_xyi = expanded_batch_ds
    else:
        gst_ds_2q_xyi = gst_ds_2q_xyi.combine_first(expanded_batch_ds)

print(gst_ds_2q_xyi.sizes)
gst_tuid_2q_xyi = save_retrieve_acquisition_dataset(gst_ds_2q_xyi, 'Q1Q2 XYI GST')
print('2Q XYI GST tuid:', gst_tuid_2q_xyi)
try:
    hp_ssg.pulsemod_state('ON')
except Exception as exc:
    hp_ssg = find_or_create_instrument(HP83732B, recreate=True, name='HP83732B_1', address='TCPIP0::172.16.0.12::1234::SOCKET')
    hp_ssg.pulsemod_state('ON')

In [ ]:
print('2DeltaLogL(estimate, data):', pygsti.tools.two_delta_logl(mdl_estimate_2q_xyi, ds_2q_xyi))
print('2DeltaLogL(ideal target, data):', pygsti.tools.two_delta_logl(mdl_ideal_2q_xyi, ds_2q_xyi))

In [ ]:
# 3) Convert dataset + run GST analysis
# gst_tuid_2q_xyi = '20260305-141542-628-6c89e9'

gst_ds_2q_xyi = load_dataset(gst_tuid_2q_xyi)

# Normalize data var names if dataset stores channels as integer keys
data_vars_now = list(gst_ds_2q_xyi.data_vars)
if 'y0' not in data_vars_now and 'y1' not in data_vars_now and 0 in data_vars_now and 1 in data_vars_now:
    gst_ds_2q_xyi = gst_ds_2q_xyi.rename({0: 'y0', 1: 'y1'})

dataset_txt_2q_xyi = gst_ds_to_pygsti_dataset_txt(
    gst_ds_2q_xyi, listOfExperiments_2q_xyi, mode='2q'
 )
ds_2q_xyi = ds_from_dataset_txt(dataset_txt_2q_xyi)

results_2q_xyi = pygsti.run_stdpractice_gst(
    ds_2q_xyi,
    mdl_ideal_2q_xyi,
    prep_fiducials_2q_xyi,
    meas_fiducials_2q_xyi,
    germs_2q_xyi,
    maxLengths_2q_xyi,
    modes=['full TP', 'Target'],
    verbosity=2,
 )

mdl_estimate_2q_xyi = results_2q_xyi.estimates['full TP'].models['stdgaugeopt']

print('2DeltaLogL(estimate, data):', pygsti.tools.two_delta_logl(mdl_estimate_2q_xyi, ds_2q_xyi))
print('2DeltaLogL(ideal target, data):', pygsti.tools.two_delta_logl(mdl_ideal_2q_xyi, ds_2q_xyi))

In [ ]:
# 4) Build and save HTML report
report_dir_2q_xyi = f"GST_reports/{gst_tuid_2q_xyi}_2Q_XYI"

pygsti.report.construct_standard_report(
    results_2q_xyi,
    title='Q1Q2 XYI GST Report',
    verbosity=1,
).write_html(report_dir_2q_xyi, auto_open=False, verbosity=1)

print('Report written to:', report_dir_2q_xyi)